# Raports

In [3]:
import pandas as pd
import numpy as np

FEATURES = [
    'missing_joints_count','missing_joints_ratio','collapsed_joints_count',
    'center_of_mass_x','center_of_mass_y','center_of_mass_z',
    'distance_from_origin',
    'bbox_width','bbox_height','bbox_depth','bbox_volume',
    'max_joint_distance_from_com',
    'distance_from_floor','below_floor',
    'left_forearm_length','right_forearm_length',
    'left_shin_length','right_shin_length',
    'arm_length_symmetry','leg_length_symmetry',
    'body_forward_x','body_forward_y','body_forward_z'
]

def safe_pearson_corr(x, y, min_n=10):
    """Returns pearson r, or np.nan if undefined."""
    x = pd.to_numeric(pd.Series(x), errors="coerce")
    y = pd.to_numeric(pd.Series(y), errors="coerce")

    m = x.notna() & y.notna()
    x = x[m].astype(float).values
    y = y[m].astype(float).values

    if len(x) < min_n:
        return np.nan
    if np.std(x) == 0 or np.std(y) == 0:
        return np.nan

    # stable pearson
    r = np.corrcoef(x, y)[0, 1]
    return float(r)

def per_app_corr(df, label, min_n=10):
    rows = []
    apps = sorted(df["App"].dropna().unique().tolist())
    for app in apps:
        dfa = df[df["App"] == app]
        # ensure label numeric
        y = pd.to_numeric(dfa[label], errors="coerce")
        for f in FEATURES:
            x = pd.to_numeric(dfa[f], errors="coerce")
            r = safe_pearson_corr(x, y, min_n=min_n)
            rows.append({"App": app, "Feature": f, "Corr": r, "N_used": int((x.notna() & y.notna()).sum())})
    return pd.DataFrame(rows)

# Run
corr_spatial  = per_app_corr(df, "Spatial",  min_n=10)
corr_temporal = per_app_corr(df, "Temporal", min_n=10)

# Pivot tables for inspection
pivot_spatial  = corr_spatial.pivot(index="Feature", columns="App", values="Corr")
pivot_temporal = corr_temporal.pivot(index="Feature", columns="App", values="Corr")

print("Spatial corr table shape:", pivot_spatial.shape)
print("Temporal corr table shape:", pivot_temporal.shape)

# Optional: show strongest |corr| per app (top 8)
def top_corrs(corr_df, app, k=8):
    d = corr_df[corr_df["App"] == app].copy()
    d["abs"] = d["Corr"].abs()
    return d.sort_values("abs", ascending=False).head(k)[["Feature","Corr","N_used"]]

for app in sorted(df["App"].unique()):
    print("\n===", app, "Spatial top corrs ===")
    print(top_corrs(corr_spatial, app, k=8).to_string(index=False))


def per_app_constant_features(df, eps=1e-12):
    out = []
    for app in sorted(df["App"].unique()):
        dfa = df[df["App"] == app]
        for f in FEATURES:
            x = pd.to_numeric(dfa[f], errors="coerce").dropna().astype(float).values
            if len(x) == 0:
                out.append((app, f, "all_nan"))
            elif np.std(x) <= eps:
                out.append((app, f, "constant"))
    return pd.DataFrame(out, columns=["App","Feature","Issue"])

const_df = per_app_constant_features(df)
print(const_df.head(50))
print("\nCounts:\n", const_df.groupby(["App","Issue"]).size())

Spatial corr table shape: (23, 6)
Temporal corr table shape: (23, 6)

=== Archery Spatial top corrs ===
             Feature      Corr  N_used
    left_shin_length -0.583391     100
      body_forward_x  0.505786     100
    center_of_mass_x -0.491096     100
distance_from_origin -0.472625     100
    center_of_mass_z -0.435250     100
      body_forward_z  0.358173     100
 leg_length_symmetry -0.323245     100
right_forearm_length  0.216497     100

=== PhantomLimb Spatial top corrs ===
             Feature      Corr  N_used
          bbox_width  0.404916     325
    left_shin_length  0.289470     325
         bbox_height -0.250010     325
   right_shin_length  0.223479     325
distance_from_origin  0.215984     325
    center_of_mass_z -0.207993     325
right_forearm_length -0.205070     325
      body_forward_z  0.186565     325

=== PianoTiles Spatial top corrs ===
                    Feature      Corr  N_used
           center_of_mass_y  0.627201     440
                bbox_heig

In [6]:
print("\n=== Per-app label balance ===")
for app in sorted(df["App"].unique()):
    sub = df[df["App"] == app]
    print(f"\nAPP={app}  N={len(sub)}")
    print("  Spatial:", sub["Spatial"].value_counts().to_dict())
    print("  Temporal:", sub["Temporal"].value_counts().to_dict())

print("\n=== Majority baseline (per app) ===")
for app in sorted(df["App"].unique()):
    sub = df[df["App"] == app]
    for lab in ["Spatial", "Temporal"]:
        vc = sub[lab].value_counts()
        maj_acc = (vc.max() / vc.sum()) if len(vc) else np.nan
        print(f"APP={app:12s} label={lab:8s} majority_acc={maj_acc:.3f} dist={vc.to_dict()}")


=== Per-app label balance ===

APP=Archery  N=100
  Spatial: {1: 86, 0: 14}
  Temporal: {1: 86, 0: 14}

APP=PhantomLimb  N=325
  Spatial: {0: 293, 1: 32}
  Temporal: {0: 190, 1: 135}

APP=PianoTiles  N=440
  Spatial: {0: 228, 1: 212}
  Temporal: {0: 386, 1: 54}

APP=Puzzle  N=299
  Spatial: {1: 213, 0: 86}
  Temporal: {1: 213, 0: 86}

APP=Sea  N=300
  Spatial: {0: 168, 1: 132}
  Temporal: {0: 164, 1: 136}

APP=War  N=280
  Spatial: {0: 176, 1: 104}
  Temporal: {0: 164, 1: 116}

=== Majority baseline (per app) ===
APP=Archery      label=Spatial  majority_acc=0.860 dist={1: 86, 0: 14}
APP=Archery      label=Temporal majority_acc=0.860 dist={1: 86, 0: 14}
APP=PhantomLimb  label=Spatial  majority_acc=0.902 dist={0: 293, 1: 32}
APP=PhantomLimb  label=Temporal majority_acc=0.585 dist={0: 190, 1: 135}
APP=PianoTiles   label=Spatial  majority_acc=0.518 dist={0: 228, 1: 212}
APP=PianoTiles   label=Temporal majority_acc=0.877 dist={0: 386, 1: 54}
APP=Puzzle       label=Spatial  majority_acc=0.7


# LR/HGB Spatial

In [1]:
# =========================
# 0) INSTALL / IMPORTS
# =========================
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

SEED = 42
np.random.seed(SEED)

# =========================
# 1) LOAD DATA
# =========================
CANDIDATE_PATHS = [
    "/content/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/VR_Metrics/extracted_metrics_all_apps.csv",
]

DATA_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        DATA_PATH = p
        break

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find extracted_metrics_all_apps.csv in common locations.\n"
        "Put it in /content OR adjust CANDIDATE_PATHS."
    )

df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH, "| shape:", df.shape)

# =========================
# 2) CLEAN / NORMALIZE TYPES
# =========================
df = df.replace("", np.nan)

required = {"App", "Spatial", "Temporal"}
missing_req = required - set(df.columns)
if missing_req:
    raise ValueError(f"Missing required columns in CSV: {missing_req}")

# Convert bool-like strings to 0/1 BEFORE pd.to_numeric()
BOOL_MAP = {"True": 1, "False": 0, "TRUE": 1, "FALSE": 0, True: 1, False: 0}
df = df.replace(BOOL_MAP)

# Force both labels present (your extraction pipeline should guarantee this)
df = df.dropna(subset=["App", "Spatial", "Temporal"]).copy()

# Normalize labels to int
for col in ["Spatial", "Temporal"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Drop any rows that became NaN after coercion (safety)
df = df.dropna(subset=["Spatial", "Temporal"]).copy()
df["Spatial"] = df["Spatial"].astype(int)
df["Temporal"] = df["Temporal"].astype(int)

apps = sorted(df["App"].unique().tolist())
print("Apps:", apps)
print("Label balance (Spatial):", df["Spatial"].value_counts(dropna=False).to_dict())
print("Label balance (Temporal):", df["Temporal"].value_counts(dropna=False).to_dict())

# =========================
# 3) FEATURE COLUMN SELECTION
# =========================
ID_COLS = [c for c in ["GlobalID", "EntryID"] if c in df.columns]
DROP_COLS = ["App", "Spatial", "Temporal"] + ID_COLS
feature_cols = [c for c in df.columns if c not in DROP_COLS]

# Coerce features to numeric (booleans already mapped)
for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("Num features:", len(feature_cols))
print("Example feature cols:", feature_cols[:15])

# =========================
# 4) MODELS (two baselines)
# =========================
# A) Tree-ish model (strong baseline)
hgb_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", HistGradientBoostingClassifier(
        random_state=SEED,
        max_depth=6,
        learning_rate=0.05,
        max_iter=400,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=20
    ))
])

# B) Linear baseline (good sanity check)
lr_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        random_state=SEED,
        max_iter=2000,
        class_weight="balanced",
        n_jobs=-1
    ))
])

MODELS = {
    "HGB": hgb_model,
    "LR_balanced": lr_model
}

# =========================
# 5) EVAL HELPERS
# =========================
def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred).tolist(),
        "pred_dist": dict(zip(*np.unique(y_pred, return_counts=True))),
    }

def run_loao(label_col: str, model_name: str, model):
    results = []
    for test_app in apps:
        train_df = df[df["App"] != test_app].copy()
        test_df  = df[df["App"] == test_app].copy()

        X_train = train_df[feature_cols].values
        y_train = train_df[label_col].values

        X_test = test_df[feature_cols].values
        y_test = test_df[label_col].values

        # If training collapses to a single class, skip (can't learn a classifier)
        uniq = np.unique(y_train)
        if len(uniq) < 2:
            print(f"[{label_col} | {model_name}] HELD-OUT={test_app:12s}  "
                  f"SKIP (train has single class: {uniq})")
            continue

        # Fit on ALL training data (no wasted split)
        model.fit(X_train, y_train)

        # Evaluate on held-out app
        yhat = model.predict(X_test)
        m = eval_binary(y_test, yhat)

        row = {
            "label": label_col,
            "model": model_name,
            "held_out_app": test_app,
            "n_train": int(len(y_train)),
            "n_test": int(len(y_test)),
            **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
            "cm": m["cm"],
            "pred_dist": m["pred_dist"],
        }

        print(f"[{label_col} | {model_name}] HELD-OUT={test_app:12s}  "
              f"acc={row['acc']:.3f} f1={row['f1']:.3f}  "
              f"prec={row['precision']:.3f} rec={row['recall']:.3f}  "
              f"n_test={row['n_test']}")
        results.append(row)

    return pd.DataFrame(results)

# =========================
# 6) RUN LOAO FOR BOTH LABELS
# =========================
all_runs = []
for label_col in ["Spatial", "Temporal"]:
    for model_name, model in MODELS.items():
        res = run_loao(label_col, model_name, model)
        all_runs.append(res)

results_df = pd.concat(all_runs, ignore_index=True)

print("\n================ SUMMARY (mean over held-out apps) ================")
summary = results_df.groupby(["label", "model"])[["acc", "f1", "precision", "recall"]].mean().reset_index()
print(summary)

OUT_PATH = "/content/loao_metrics_results.csv"
results_df.to_csv(OUT_PATH, index=False)
print("\nSaved per-app LOAO results to:", OUT_PATH)

Loaded: /content/extracted_metrics_all_apps.csv | shape: (1744, 28)
Apps: ['Archery', 'PhantomLimb', 'PianoTiles', 'Puzzle', 'Sea', 'War']
Label balance (Spatial): {0: 965, 1: 779}
Label balance (Temporal): {0: 1004, 1: 740}
Num features: 23
Example feature cols: ['missing_joints_count', 'missing_joints_ratio', 'collapsed_joints_count', 'center_of_mass_x', 'center_of_mass_y', 'center_of_mass_z', 'distance_from_origin', 'bbox_width', 'bbox_height', 'bbox_depth', 'bbox_volume', 'max_joint_distance_from_com', 'distance_from_floor', 'below_floor', 'left_forearm_length']


/tmp/ipython-input-3144438320.py:55: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace(BOOL_MAP)


[Spatial | HGB] HELD-OUT=Archery       acc=0.860 f1=0.925  prec=0.860 rec=1.000  n_test=100
[Spatial | HGB] HELD-OUT=PhantomLimb   acc=0.480 f1=0.214  prec=0.126 rec=0.719  n_test=325
[Spatial | HGB] HELD-OUT=PianoTiles    acc=0.534 f1=0.081  prec=0.818 rec=0.042  n_test=440
[Spatial | HGB] HELD-OUT=Puzzle        acc=0.201 f1=0.000  prec=0.000 rec=0.000  n_test=299
[Spatial | HGB] HELD-OUT=Sea           acc=0.907 f1=0.886  prec=0.956 rec=0.826  n_test=300
[Spatial | HGB] HELD-OUT=War           acc=0.671 f1=0.685  prec=0.532 rec=0.962  n_test=280
[Spatial | LR_balanced] HELD-OUT=Archery       acc=0.860 f1=0.925  prec=0.860 rec=1.000  n_test=100
[Spatial | LR_balanced] HELD-OUT=PhantomLimb   acc=0.332 f1=0.069  prec=0.040 rec=0.250  n_test=325
[Spatial | LR_balanced] HELD-OUT=PianoTiles    acc=0.500 f1=0.000  prec=0.000 rec=0.000  n_test=440
[Spatial | LR_balanced] HELD-OUT=Puzzle        acc=0.288 f1=0.000  prec=0.000 rec=0.000  n_test=299
[Spatial | LR_balanced] HELD-OUT=Sea           a

In [2]:
# ===============================================================
# NOTE (IMPORTANT): PhantomLimb dominates the dataset (~4105 rows)
# compared to other apps (~280–440 rows). To keep LOAO evaluation
# app-fair and reduce training bias, we CAP PhantomLimb to N rows
# (default: 400) via deterministic sampling (SEED).
#
# This script prints BEFORE/AFTER per-app row counts so we can
# confirm the cap is applied.
# ===============================================================

# =========================
# 0) IMPORTS / SEED
# =========================
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

SEED = 42
np.random.seed(SEED)

# =========================
# 1) LOAD DATA
# =========================
CANDIDATE_PATHS = [
    "/content/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/VR_Metrics/extracted_metrics_all_apps.csv",
]

DATA_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        DATA_PATH = p
        break

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find extracted_metrics_all_apps.csv in common locations.\n"
        "Put it in /content OR adjust CANDIDATE_PATHS."
    )

df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH, "| shape:", df.shape)

# =========================
# 2) CLEAN / NORMALIZE TYPES
# =========================
df = df.replace("", np.nan)

required = {"App", "Spatial", "Temporal"}
missing_req = required - set(df.columns)
if missing_req:
    raise ValueError(f"Missing required columns in CSV: {missing_req}")

# Convert bool-like strings to 0/1 BEFORE numeric coercion
BOOL_MAP = {"True": 1, "False": 0, "TRUE": 1, "FALSE": 0, True: 1, False: 0}
df = df.replace(BOOL_MAP)

# Require both labels present
df = df.dropna(subset=["App", "Spatial", "Temporal"]).copy()

# Normalize labels to int
for col in ["Spatial", "Temporal"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df = df.dropna(subset=["Spatial", "Temporal"]).copy()

df["Spatial"] = df["Spatial"].astype(int)
df["Temporal"] = df["Temporal"].astype(int)

# Print counts BEFORE capping
print("\n================ PER-APP COUNTS (BEFORE CAP) ================")
before_counts = df["App"].value_counts().sort_index()
print(before_counts.to_string())
print("TOTAL (before):", int(len(df)))

# =========================
# 2.5) CAP PhantomLimb (FAIRNESS CONTROL)
# =========================
PHANTOMLIMB_CAP = 400  # <-- change to 300 or 400 as desired

print("\n===============================================================")
print(f"Applying PhantomLimb cap: PHANTOMLIMB_CAP = {PHANTOMLIMB_CAP} (SEED={SEED})")
print("===============================================================\n")

df_parts = []
for app, g in df.groupby("App"):
    if app == "PhantomLimb" and len(g) > PHANTOMLIMB_CAP:
        g = g.sample(n=PHANTOMLIMB_CAP, random_state=SEED)
    df_parts.append(g)

df = pd.concat(df_parts, ignore_index=True)

# Print counts AFTER capping
print("================ PER-APP COUNTS (AFTER CAP) ================")
after_counts = df["App"].value_counts().sort_index()
print(after_counts.to_string())
print("TOTAL (after):", int(len(df)))

apps = sorted(df["App"].unique().tolist())
print("\nApps:", apps)
print("Label balance (Spatial):", df["Spatial"].value_counts(dropna=False).to_dict())
print("Label balance (Temporal):", df["Temporal"].value_counts(dropna=False).to_dict())

# =========================
# 3) FEATURE COLUMN SELECTION
# =========================
ID_COLS = [c for c in ["GlobalID", "EntryID"] if c in df.columns]
DROP_COLS = ["App", "Spatial", "Temporal"] + ID_COLS
feature_cols = [c for c in df.columns if c not in DROP_COLS]

# Coerce features to numeric
for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("\nNum features:", len(feature_cols))
print("Example feature cols:", feature_cols[:15])

# =========================
# 4) MODELS (two baselines)
# =========================
hgb_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", HistGradientBoostingClassifier(
        random_state=SEED,
        max_depth=6,
        learning_rate=0.05,
        max_iter=400,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=20
    ))
])

lr_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        random_state=SEED,
        max_iter=2000,
        class_weight="balanced",
        n_jobs=-1
    ))
])

MODELS = {
    "HGB": hgb_model,
    "LR_balanced": lr_model
}

# =========================
# 5) EVAL HELPERS
# =========================
def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred).tolist(),
        "pred_dist": dict(zip(*np.unique(y_pred, return_counts=True))),
    }

def run_loao(label_col: str, model_name: str, model):
    results = []
    for test_app in apps:
        train_df = df[df["App"] != test_app].copy()
        test_df  = df[df["App"] == test_app].copy()

        X_train = train_df[feature_cols].values
        y_train = train_df[label_col].values

        X_test = test_df[feature_cols].values
        y_test = test_df[label_col].values

        # If training collapses to a single class, skip (can't learn a classifier)
        uniq = np.unique(y_train)
        if len(uniq) < 2:
            print(f"[{label_col} | {model_name}] HELD-OUT={test_app:12s}  "
                  f"SKIP (train has single class: {uniq})")
            continue

        # Fit on ALL training data
        model.fit(X_train, y_train)

        # Evaluate on held-out app
        yhat = model.predict(X_test)
        m = eval_binary(y_test, yhat)

        row = {
            "label": label_col,
            "model": model_name,
            "held_out_app": test_app,
            "n_train": int(len(y_train)),
            "n_test": int(len(y_test)),
            **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
            "cm": m["cm"],
            "pred_dist": m["pred_dist"],
        }

        print(f"[{label_col} | {model_name}] HELD-OUT={test_app:12s}  "
              f"acc={row['acc']:.3f} f1={row['f1']:.3f}  "
              f"prec={row['precision']:.3f} rec={row['recall']:.3f}  "
              f"n_test={row['n_test']}")
        results.append(row)

    return pd.DataFrame(results)

# =========================
# 6) RUN LOAO FOR BOTH LABELS
# =========================
all_runs = []
for label_col in ["Spatial", "Temporal"]:
    for model_name, model in MODELS.items():
        res = run_loao(label_col, model_name, model)
        all_runs.append(res)

results_df = pd.concat(all_runs, ignore_index=True)

print("\n================ SUMMARY (mean over held-out apps) ================")
summary = results_df.groupby(["label", "model"])[["acc", "f1", "precision", "recall"]].mean().reset_index()
print(summary)

OUT_PATH = "/content/loao_metrics_results.csv"
results_df.to_csv(OUT_PATH, index=False)
print("\nSaved per-app LOAO results to:", OUT_PATH)

Loaded: /content/extracted_metrics_all_apps.csv | shape: (1744, 28)

================ PER-APP COUNTS (BEFORE CAP) ================
App
Archery        100
PhantomLimb    325
PianoTiles     440
Puzzle         299
Sea            300
War            280
TOTAL (before): 1744

Applying PhantomLimb cap: PHANTOMLIMB_CAP = 400 (SEED=42)

================ PER-APP COUNTS (AFTER CAP) ================
App
Archery        100
PhantomLimb    325
PianoTiles     440
Puzzle         299
Sea            300
War            280
TOTAL (after): 1744

Apps: ['Archery', 'PhantomLimb', 'PianoTiles', 'Puzzle', 'Sea', 'War']
Label balance (Spatial): {0: 965, 1: 779}
Label balance (Temporal): {0: 1004, 1: 740}

Num features: 23
Example feature cols: ['missing_joints_count', 'missing_joints_ratio', 'collapsed_joints_count', 'center_of_mass_x', 'center_of_mass_y', 'center_of_mass_z', 'distance_from_origin', 'bbox_width', 'bbox_height', 'bbox_depth', 'bbox_volume', 'max_joint_distance_from_com', 'distance_from_floor', 'b

/tmp/ipython-input-3292830653.py:65: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace(BOOL_MAP)


[Spatial | HGB] HELD-OUT=Archery       acc=0.860 f1=0.925  prec=0.860 rec=1.000  n_test=100
[Spatial | HGB] HELD-OUT=PhantomLimb   acc=0.551 f1=0.291  prec=0.172 rec=0.938  n_test=325
[Spatial | HGB] HELD-OUT=PianoTiles    acc=0.534 f1=0.113  prec=0.684 rec=0.061  n_test=440
[Spatial | HGB] HELD-OUT=Puzzle        acc=0.288 f1=0.000  prec=0.000 rec=0.000  n_test=299
[Spatial | HGB] HELD-OUT=Sea           acc=0.867 f1=0.829  prec=0.951 rec=0.735  n_test=300
[Spatial | HGB] HELD-OUT=War           acc=0.686 f1=0.692  prec=0.544 rec=0.952  n_test=280
[Spatial | LR_balanced] HELD-OUT=Archery       acc=0.860 f1=0.925  prec=0.860 rec=1.000  n_test=100
[Spatial | LR_balanced] HELD-OUT=PhantomLimb   acc=0.332 f1=0.069  prec=0.040 rec=0.250  n_test=325
[Spatial | LR_balanced] HELD-OUT=PianoTiles    acc=0.500 f1=0.000  prec=0.000 rec=0.000  n_test=440
[Spatial | LR_balanced] HELD-OUT=Puzzle        acc=0.288 f1=0.000  prec=0.000 rec=0.000  n_test=299
[Spatial | LR_balanced] HELD-OUT=Sea           a

# LR/HGB Temoporal

In [4]:
# ===============================================================
# (A) PREP BLOCK — build Temporal 5-frame sequences with 138 features
#     Using: mean, std, delta, range, mean_abs_vel, vel_std
#     => 6 * 23 = 138 features (if you have 23 metrics/features)
# ===============================================================

import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

# ---------
# Choose the temporal aggregations (6 blocks => 138 if F=23)
# ---------
TEMPORAL_AGG = ("mean", "std", "delta", "range", "mean_abs_vel", "vel_std")
SEQ_LEN = 5

def aggregate_window(window: np.ndarray, use=TEMPORAL_AGG) -> np.ndarray:
    """
    window: (T, F) array
    return: (K*F,) aggregated vector
    """
    feats = []

    if "mean" in use:
        feats.append(np.nanmean(window, axis=0))
    if "std" in use:
        feats.append(np.nanstd(window, axis=0))
    if "delta" in use:
        feats.append(window[-1] - window[0])
    if "range" in use:
        feats.append(np.nanmax(window, axis=0) - np.nanmin(window, axis=0))

    if "mean_abs_vel" in use or "vel_std" in use:
        diff = np.diff(window, axis=0)  # (T-1, F)
        if "mean_abs_vel" in use:
            feats.append(np.nanmean(np.abs(diff), axis=0))
        if "vel_std" in use:
            feats.append(np.nanstd(diff, axis=0))

    return np.concatenate(feats, axis=0)

def build_sequences_for_app(app_df: pd.DataFrame,
                           feature_cols,
                           label_col="Temporal",
                           seq_len=5,
                           agg=TEMPORAL_AGG):
    """
    Sliding window over rows within ONE app.
    Label rule: center frame label (i + seq_len//2)
    Aggregation: aggregate_window() per metric.
    """
    # Sort by EntryID so the window is temporal
    # (works for numeric IDs and timestamp-like IDs)
    app_df = app_df.sort_values("EntryID").reset_index(drop=True)

    X = app_df[feature_cols].values
    y = app_df[label_col].values.astype(int)

    if len(app_df) < seq_len:
        return np.empty((0, len(agg)*len(feature_cols))), np.empty((0,), dtype=int)

    X_seq, y_seq = [], []
    mid = seq_len // 2

    for i in range(len(app_df) - seq_len + 1):
        window = X[i:i+seq_len]  # (T,F)
        X_seq.append(aggregate_window(window, use=agg))
        y_seq.append(int(y[i + mid]))

    return np.asarray(X_seq), np.asarray(y_seq)

# ---------
# Diagnostics: expected temporal feature dimension
# ---------
print("TEMPORAL_AGG blocks:", TEMPORAL_AGG, "| blocks =", len(TEMPORAL_AGG))
print("SEQ_LEN:", SEQ_LEN)

# After you load df and compute feature_cols in your pipeline:
# print("Frame-level feature count F:", len(feature_cols))
# print("Expected seq-level feature count:", len(TEMPORAL_AGG)*len(feature_cols))

TEMPORAL_AGG blocks: ('mean', 'std', 'delta', 'range', 'mean_abs_vel', 'vel_std') | blocks = 6
SEQ_LEN: 5


In [5]:
# ===============================================================
# (B) MODEL BLOCK — LOAO:
#     - Spatial: frame-level (same as before)
#     - Temporal: sequence-level (SEQ_LEN=5, 138 features if F=23)
# ===============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

SEED = 42

# --- Models (same idea as before) ---
hgb_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", HistGradientBoostingClassifier(
        random_state=SEED,
        max_depth=6,
        learning_rate=0.05,
        max_iter=400,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=20
    ))
])

lr_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        random_state=SEED,
        max_iter=2000,
        class_weight="balanced",
        n_jobs=-1
    ))
])

MODELS = {
    "HGB": hgb_model,
    "LR_balanced": lr_model
}

def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred).tolist(),
        "pred_dist": dict(zip(*np.unique(y_pred, return_counts=True))),
    }

def build_sequences_from_df(df_in: pd.DataFrame, feature_cols, label_col="Temporal"):
    """
    Build sequences for multiple apps and concatenate.
    Prevents windows crossing app boundaries by doing groupby("App").
    """
    X_list, y_list = [], []
    for app, g in df_in.groupby("App"):
        X_a, y_a = build_sequences_for_app(
            g, feature_cols,
            label_col=label_col,
            seq_len=SEQ_LEN,
            agg=TEMPORAL_AGG
        )
        if len(y_a) > 0:
            X_list.append(X_a)
            y_list.append(y_a)

    if not X_list:
        return np.empty((0, len(TEMPORAL_AGG)*len(feature_cols))), np.empty((0,), dtype=int)

    return np.vstack(X_list), np.concatenate(y_list)

def run_loao(label_col: str, model_name: str, model, df: pd.DataFrame, feature_cols):
    apps = sorted(df["App"].unique().tolist())
    results = []

    for test_app in apps:
        train_df = df[df["App"] != test_app].copy()
        test_df  = df[df["App"] == test_app].copy()

        # --------------------------
        # Spatial: frame-level
        # Temporal: seq-level
        # --------------------------
        if label_col == "Temporal":
            X_train, y_train = build_sequences_from_df(train_df, feature_cols, label_col="Temporal")
            X_test,  y_test  = build_sequences_for_app(test_df, feature_cols, label_col="Temporal",
                                                       seq_len=SEQ_LEN, agg=TEMPORAL_AGG)

        else:
            X_train = train_df[feature_cols].values
            y_train = train_df[label_col].values.astype(int)
            X_test  = test_df[feature_cols].values
            y_test  = test_df[label_col].values.astype(int)

        if len(y_train) == 0 or len(y_test) == 0:
            print(f"[{label_col} | {model_name}] HELD-OUT={test_app:12s} SKIP (empty split)")
            continue

        uniq = np.unique(y_train)
        if len(uniq) < 2:
            print(f"[{label_col} | {model_name}] HELD-OUT={test_app:12s} SKIP (single-class train: {uniq})")
            continue

        model.fit(X_train, y_train)
        yhat = model.predict(X_test)
        m = eval_binary(y_test, yhat)

        row = {
            "label": label_col,
            "model": model_name,
            "held_out_app": test_app,
            "n_train": int(len(y_train)),
            "n_test": int(len(y_test)),
            "acc": m["acc"],
            "f1": m["f1"],
            "precision": m["precision"],
            "recall": m["recall"],
            "cm": m["cm"],
            "pred_dist": m["pred_dist"],
        }

        # Print with extra info about seq dimensionality for Temporal
        if label_col == "Temporal":
            print(f"[{label_col} | {model_name}] HELD-OUT={test_app:12s} "
                  f"(SEQ_LEN={SEQ_LEN}, seqF={X_train.shape[1]})  "
                  f"acc={row['acc']:.3f} f1={row['f1']:.3f} "
                  f"prec={row['precision']:.3f} rec={row['recall']:.3f} "
                  f"n_test={row['n_test']}")
        else:
            print(f"[{label_col} | {model_name}] HELD-OUT={test_app:12s}  "
                  f"acc={row['acc']:.3f} f1={row['f1']:.3f} "
                  f"prec={row['precision']:.3f} rec={row['recall']:.3f} "
                  f"n_test={row['n_test']}")

        results.append(row)

    return pd.DataFrame(results)

# =========================
# RUN BOTH LABELS
# =========================
all_runs = []
for label_col in ["Spatial", "Temporal"]:
    for model_name, model in MODELS.items():
        res = run_loao(label_col, model_name, model, df, feature_cols)
        all_runs.append(res)

results_df = pd.concat(all_runs, ignore_index=True)

print("\n================ SUMMARY (mean over held-out apps) ================")
summary = results_df.groupby(["label", "model"])[["acc", "f1", "precision", "recall"]].mean().reset_index()
print(summary)

OUT_PATH = "/content/loao_metrics_results_seq_temporal.csv"
results_df.to_csv(OUT_PATH, index=False)
print("\nSaved per-app LOAO results to:", OUT_PATH)

[Spatial | HGB] HELD-OUT=Archery       acc=0.860 f1=0.925 prec=0.860 rec=1.000 n_test=100
[Spatial | HGB] HELD-OUT=PhantomLimb   acc=0.551 f1=0.291 prec=0.172 rec=0.938 n_test=325
[Spatial | HGB] HELD-OUT=PianoTiles    acc=0.534 f1=0.113 prec=0.684 rec=0.061 n_test=440
[Spatial | HGB] HELD-OUT=Puzzle        acc=0.288 f1=0.000 prec=0.000 rec=0.000 n_test=299
[Spatial | HGB] HELD-OUT=Sea           acc=0.867 f1=0.829 prec=0.951 rec=0.735 n_test=300
[Spatial | HGB] HELD-OUT=War           acc=0.686 f1=0.692 prec=0.544 rec=0.952 n_test=280
[Spatial | LR_balanced] HELD-OUT=Archery       acc=0.860 f1=0.925 prec=0.860 rec=1.000 n_test=100
[Spatial | LR_balanced] HELD-OUT=PhantomLimb   acc=0.332 f1=0.069 prec=0.040 rec=0.250 n_test=325
[Spatial | LR_balanced] HELD-OUT=PianoTiles    acc=0.500 f1=0.000 prec=0.000 rec=0.000 n_test=440
[Spatial | LR_balanced] HELD-OUT=Puzzle        acc=0.288 f1=0.000 prec=0.000 rec=0.000 n_test=299
[Spatial | LR_balanced] HELD-OUT=Sea           acc=0.617 f1=0.228 pr

/tmp/ipython-input-68296431.py:27: RuntimeWarning: Mean of empty slice
  feats.append(np.nanmean(window, axis=0))
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-68296431.py:33: RuntimeWarning: All-NaN slice encountered
  feats.append(np.nanmax(window, axis=0) - np.nanmin(window, axis=0))
/tmp/ipython-input-68296431.py:38: RuntimeWarning: Mean of empty slice
  feats.append(np.nanmean(np.abs(diff), axis=0))
/tmp/ipython-input-68296431.py:27: RuntimeWarning: Mean of empty slice
  feats.append(np.nanmean(window, axis=0))
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-68296431.py:33: RuntimeWarning: All-NaN slice encountered
  feats.append(np.nanmax(window, axis=0) - np.

[Temporal | HGB] HELD-OUT=Archery      (SEQ_LEN=5, seqF=138)  acc=0.896 f1=0.943 prec=0.902 rec=0.988 n_test=96


/tmp/ipython-input-68296431.py:27: RuntimeWarning: Mean of empty slice
  feats.append(np.nanmean(window, axis=0))
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-68296431.py:33: RuntimeWarning: All-NaN slice encountered
  feats.append(np.nanmax(window, axis=0) - np.nanmin(window, axis=0))
/tmp/ipython-input-68296431.py:38: RuntimeWarning: Mean of empty slice
  feats.append(np.nanmean(np.abs(diff), axis=0))
/tmp/ipython-input-68296431.py:27: RuntimeWarning: Mean of empty slice
  feats.append(np.nanmean(window, axis=0))
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-68296431.py:33: RuntimeWarning: All-NaN slice encountered
  feats.append(np.nanmax(window, axis=0) - np.

[Temporal | HGB] HELD-OUT=PhantomLimb  (SEQ_LEN=5, seqF=138)  acc=0.654 f1=0.284 prec=1.000 rec=0.165 n_test=321


...

[Temporal | LR_balanced] HELD-OUT=Puzzle       (SEQ_LEN=5, seqF=138)  acc=0.586 f1=0.679 prec=0.772 rec=0.606 n_test=295


/tmp/ipython-input-68296431.py:27: RuntimeWarning: Mean of empty slice
  feats.append(np.nanmean(window, axis=0))
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-68296431.py:33: RuntimeWarning: All-NaN slice encountered
  feats.append(np.nanmax(window, axis=0) - np.nanmin(window, axis=0))
/tmp/ipython-input-68296431.py:38: RuntimeWarning: Mean of empty slice
  feats.append(np.nanmean(np.abs(diff), axis=0))
/tmp/ipython-input-68296431.py:27: RuntimeWarning: Mean of empty slice
  feats.append(np.nanmean(window, axis=0))
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-68296431.py:33: RuntimeWarning: All-NaN slice encountered
  feats.append(np.nanmax(window, axis=0) - np.

[Temporal | LR_balanced] HELD-OUT=Sea          (SEQ_LEN=5, seqF=138)  acc=0.453 f1=0.623 prec=0.453 rec=1.000 n_test=296


/tmp/ipython-input-68296431.py:27: RuntimeWarning: Mean of empty slice
  feats.append(np.nanmean(window, axis=0))
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-68296431.py:33: RuntimeWarning: All-NaN slice encountered
  feats.append(np.nanmax(window, axis=0) - np.nanmin(window, axis=0))
/tmp/ipython-input-68296431.py:38: RuntimeWarning: Mean of empty slice
  feats.append(np.nanmean(np.abs(diff), axis=0))
/tmp/ipython-input-68296431.py:27: RuntimeWarning: Mean of empty slice
  feats.append(np.nanmean(window, axis=0))
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-68296431.py:33: RuntimeWarning: All-NaN slice encountered
  feats.append(np.nanmax(window, axis=0) - np.

[Temporal | LR_balanced] HELD-OUT=War          (SEQ_LEN=5, seqF=138)  acc=0.558 f1=0.655 prec=0.487 rec=1.000 n_test=276

================ SUMMARY (mean over held-out apps) ================
      label        model       acc        f1  precision    recall
0   Spatial          HGB  0.630811  0.474986   0.535260  0.614265
1   Spatial  LR_balanced  0.542290  0.317633   0.403300  0.396465
2  Temporal          HGB  0.700241  0.528135   0.621713  0.557131
3  Temporal  LR_balanced  0.709248  0.699356   0.651604  0.816130

Saved per-app LOAO results to: /content/loao_metrics_results_seq_temporal.csv


In [7]:
# ===============================================================
# VR APP-INDEPENDENT METRICS - LOAO BASELINES
# Patch A: Per-app label sanity + majority baseline
# Patch B: NaN-safe temporal window aggregation (removes warnings)
#
# NOTE:
# - Spatial is evaluated frame-level (single row -> label)
# - Temporal is evaluated sequence-level with sliding window (SEQ_LEN)
# - Optional: Downsample PhantomLimb to match other apps (toggle below)
# ===============================================================

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

SEED = 42
np.random.seed(SEED)

# =========================
# 1) LOAD DATA
# =========================
CANDIDATE_PATHS = [
    "/content/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/VR_Metrics/extracted_metrics_all_apps.csv",
]

DATA_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        DATA_PATH = p
        break

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find extracted_metrics_all_apps.csv in common locations.\n"
        "Put it in /content OR adjust CANDIDATE_PATHS."
    )

df = pd.read_csv(DATA_PATH)
print("\n================= LOADED DATA =================")
print("Loaded:", DATA_PATH, "| shape:", df.shape)

# =========================
# 2) CLEAN / NORMALIZE TYPES
# =========================
df = df.replace("", np.nan)

required = {"App", "Spatial", "Temporal"}
missing_req = required - set(df.columns)
if missing_req:
    raise ValueError(f"Missing required columns in CSV: {missing_req}")

for col in ["Spatial", "Temporal"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["App"]).copy()
df = df.dropna(subset=["Spatial", "Temporal"], how="all").copy()

df["Spatial"] = df["Spatial"].astype(int)
df["Temporal"] = df["Temporal"].astype(int)

apps = sorted(df["App"].unique().tolist())
print("Apps:", apps)

# =========================
# OPTIONAL: Downsample PhantomLimb
# =========================
DOWNSAMPLE_PHANTOMLIMB = True
TARGET_N_PL = 400

if DOWNSAMPLE_PHANTOMLIMB and ("PhantomLimb" in df["App"].unique()):
    pl = df[df["App"] == "PhantomLimb"]
    others = df[df["App"] != "PhantomLimb"]
    if len(pl) > TARGET_N_PL:
        pl = pl.sample(TARGET_N_PL, random_state=SEED)
    df = pd.concat([others, pl], ignore_index=True)
    apps = sorted(df["App"].unique().tolist())
    print("\n[INFO] Downsample PhantomLimb =", DOWNSAMPLE_PHANTOMLIMB,
          "| TARGET_N_PL =", TARGET_N_PL)
    print("[INFO] PhantomLimb N =", len(df[df["App"] == "PhantomLimb"]),
          "| Total N =", len(df))

# =========================
# Patch A: Label sanity + baselines
# =========================
print("\n=== Per-app label balance ===")
for app in apps:
    sub = df[df["App"] == app]
    print(f"\nAPP={app}  N={len(sub)}")
    print("  Spatial:", sub["Spatial"].value_counts().to_dict())
    print("  Temporal:", sub["Temporal"].value_counts().to_dict())

print("\n=== Majority baseline (per app) ===")
for app in apps:
    sub = df[df["App"] == app]
    for lab in ["Spatial", "Temporal"]:
        vc = sub[lab].value_counts()
        maj_acc = (vc.max() / vc.sum()) if len(vc) else np.nan
        print(f"APP={app:12s} label={lab:8s} majority_acc={maj_acc:.3f} dist={vc.to_dict()}")

# =========================
# 3) FEATURE COLUMN SELECTION
# =========================
ID_COLS = [c for c in ["GlobalID", "EntryID"] if c in df.columns]
DROP_COLS = ["App", "Spatial", "Temporal"] + ID_COLS
feature_cols = [c for c in df.columns if c not in DROP_COLS]

# Coerce features to numeric
for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("\n================= FEATURES =================")
print("Num features:", len(feature_cols))
print("First 25 feature cols:", feature_cols[:25])

# =========================
# 4) MODELS
# =========================
hgb_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", HistGradientBoostingClassifier(
        random_state=SEED,
        max_depth=6,
        learning_rate=0.05,
        max_iter=400
    ))
])

lr_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        random_state=SEED,
        max_iter=3000,
        class_weight="balanced",
        n_jobs=-1
    ))
])

MODELS = {
    "HGB": hgb_model,
    "LR_balanced": lr_model
}

# =========================
# 5) EVAL HELPERS
# =========================
def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred).tolist(),
        "pred_dist": dict(zip(*np.unique(y_pred, return_counts=True))),
    }

# ===============================================================
# Patch B: NaN-safe temporal aggregation for SEQ features
# ===============================================================
def _nanmean_safe(x, axis=0):
    with np.errstate(all="ignore"):
        return np.nanmean(x, axis=axis)

def _nanstd_safe(x, axis=0):
    with np.errstate(all="ignore"):
        return np.nanstd(x, axis=axis)

def _nanrange_safe(x, axis=0):
    with np.errstate(all="ignore"):
        mx = np.nanmax(x, axis=axis)
        mn = np.nanmin(x, axis=axis)
    return mx - mn

def window_is_valid(window, min_non_nan_ratio=0.4):
    total = window.size
    non_nan = np.isfinite(window).sum()
    return (non_nan / total) >= min_non_nan_ratio

def aggregate_window(window: np.ndarray,
                     use=("mean","std","delta","range","mean_abs_vel","vel_std")) -> np.ndarray:
    feats = []

    if "mean" in use:
        feats.append(_nanmean_safe(window, axis=0))
    if "std" in use:
        feats.append(_nanstd_safe(window, axis=0))
    if "delta" in use:
        feats.append(window[-1] - window[0])
    if "range" in use:
        feats.append(_nanrange_safe(window, axis=0))

    if "mean_abs_vel" in use or "vel_std" in use:
        diff = np.diff(window, axis=0)  # (T-1,F)
        if "mean_abs_vel" in use:
            feats.append(_nanmean_safe(np.abs(diff), axis=0))
        if "vel_std" in use:
            feats.append(_nanstd_safe(diff, axis=0))

    return np.concatenate(feats, axis=0)

def make_sequence_dataset(app_df, label_col, seq_len=5,
                          agg_use=("mean","std","delta","range","mean_abs_vel","vel_std"),
                          min_non_nan_ratio=0.4):
    """
    Converts frame-level features into sequence-level features by sliding window.
    Label is taken from center frame (like your earlier temporal labeling).
    """
    X = app_df[feature_cols].values.astype(float)
    y = app_df[label_col].values.astype(int)

    if len(app_df) < seq_len:
        return np.empty((0, 1)), np.empty((0,), dtype=int)

    X_seq = []
    y_seq = []
    center = seq_len // 2

    for i in range(len(app_df) - seq_len + 1):
        window = X[i:i+seq_len]

        # drop too-empty windows to avoid all-NaN aggregates
        if not window_is_valid(window, min_non_nan_ratio=min_non_nan_ratio):
            continue

        feat = aggregate_window(window, use=agg_use)
        X_seq.append(feat)
        y_seq.append(y[i + center])

    if len(X_seq) == 0:
        return np.empty((0, 1)), np.empty((0,), dtype=int)

    return np.vstack(X_seq), np.array(y_seq, dtype=int)

# =========================
# 6) LOAO: Spatial (frame-level)
# =========================
def run_loao_frame(label_col: str, model_name: str, model):
    results = []
    for test_app in apps:
        train_df = df[df["App"] != test_app].copy()
        test_df  = df[df["App"] == test_app].copy()

        X_train_full = train_df[feature_cols].values
        y_train_full = train_df[label_col].values

        X_test = test_df[feature_cols].values
        y_test = test_df[label_col].values

        # Inner val split for debugging (optional)
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_full, y_train_full,
            test_size=0.15,
            random_state=SEED,
            stratify=y_train_full
        )

        model.fit(X_tr, y_tr)
        yhat = model.predict(X_test)
        m = eval_binary(y_test, yhat)

        row = {
            "label": label_col,
            "task": "frame",
            "model": model_name,
            "held_out_app": test_app,
            "n_train": int(len(y_tr)),
            "n_val": int(len(y_val)),
            "n_test": int(len(y_test)),
            **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
            "cm": m["cm"],
            "pred_dist": m["pred_dist"],
        }

        print(f"[{label_col} | {model_name}] HELD-OUT={test_app:12s}  "
              f"acc={row['acc']:.3f} f1={row['f1']:.3f}  "
              f"prec={row['precision']:.3f} rec={row['recall']:.3f}  "
              f"n_test={row['n_test']}")
        results.append(row)

    return pd.DataFrame(results)

# =========================
# 7) LOAO: Temporal (sequence-level)
# =========================
def run_loao_sequence(label_col: str, model_name: str, model,
                      seq_len=5,
                      agg_use=("mean","std","delta","range","mean_abs_vel","vel_std"),
                      min_non_nan_ratio=0.4):
    results = []

    # Determine final seq feature length for printing
    seq_feature_dim = len(agg_use) * len(feature_cols)

    for test_app in apps:
        train_df = df[df["App"] != test_app].copy()
        test_df  = df[df["App"] == test_app].copy()

        # Build sequence datasets PER split (prevents any leakage)
        X_train_full, y_train_full = [], []
        for a in apps:
            if a == test_app:
                continue
            Xa, ya = make_sequence_dataset(
                df[df["App"] == a],
                label_col=label_col,
                seq_len=seq_len,
                agg_use=agg_use,
                min_non_nan_ratio=min_non_nan_ratio
            )
            if len(ya) > 0:
                X_train_full.append(Xa)
                y_train_full.append(ya)

        if len(X_train_full) == 0:
            print(f"[{label_col} | {model_name}] HELD-OUT={test_app}  -> No train sequences.")
            continue

        X_train_full = np.vstack(X_train_full)
        y_train_full = np.concatenate(y_train_full)

        X_test, y_test = make_sequence_dataset(
            test_df,
            label_col=label_col,
            seq_len=seq_len,
            agg_use=agg_use,
            min_non_nan_ratio=min_non_nan_ratio
        )

        if len(y_test) == 0:
            print(f"[{label_col} | {model_name}] HELD-OUT={test_app}  -> No test sequences.")
            continue

        # Inner val split
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_full, y_train_full,
            test_size=0.15,
            random_state=SEED,
            stratify=y_train_full
        )

        model.fit(X_tr, y_tr)
        yhat = model.predict(X_test)
        m = eval_binary(y_test, yhat)

        row = {
            "label": label_col,
            "task": f"sequence_len_{seq_len}",
            "model": model_name,
            "held_out_app": test_app,
            "seq_len": int(seq_len),
            "seq_feature_dim": int(seq_feature_dim),
            "n_train": int(len(y_tr)),
            "n_val": int(len(y_val)),
            "n_test": int(len(y_test)),
            **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
            "cm": m["cm"],
            "pred_dist": m["pred_dist"],
        }

        print(f"[{label_col} | {model_name}] HELD-OUT={test_app:12s} "
              f"(SEQ_LEN={seq_len}, seqF={seq_feature_dim})  "
              f"acc={row['acc']:.3f} f1={row['f1']:.3f} "
              f"prec={row['precision']:.3f} rec={row['recall']:.3f} "
              f"n_test={row['n_test']}")
        results.append(row)

    return pd.DataFrame(results)

# =========================
# 8) RUN EVERYTHING
# =========================
SEQ_LEN = 5
AGG_USE = ("mean","std","delta","range","mean_abs_vel","vel_std")  # 6 * F -> e.g., 6*23=138
MIN_NON_NAN_RATIO = 0.4

all_runs = []

print("\n================= LOAO: SPATIAL (FRAME) =================")
for model_name, model in MODELS.items():
    res = run_loao_frame("Spatial", model_name, model)
    all_runs.append(res)

print("\n================= LOAO: TEMPORAL (SEQUENCE) =================")
for model_name, model in MODELS.items():
    res = run_loao_sequence(
        "Temporal", model_name, model,
        seq_len=SEQ_LEN,
        agg_use=AGG_USE,
        min_non_nan_ratio=MIN_NON_NAN_RATIO
    )
    all_runs.append(res)

results_df = pd.concat(all_runs, ignore_index=True)

print("\n================ SUMMARY (mean over held-out apps) ================")
summary = results_df.groupby(["label", "task", "model"])[["acc", "f1", "precision", "recall"]].mean().reset_index()
print(summary)

OUT_PATH = "/content/loao_metrics_results_PATCHB.csv"
results_df.to_csv(OUT_PATH, index=False)
print("\nSaved results to:", OUT_PATH)


================= LOADED DATA =================
Loaded: /content/extracted_metrics_all_apps.csv | shape: (1744, 28)
Apps: ['Archery', 'PhantomLimb', 'PianoTiles', 'Puzzle', 'Sea', 'War']

[INFO] Downsample PhantomLimb = True | TARGET_N_PL = 400
[INFO] PhantomLimb N = 325 | Total N = 1744

=== Per-app label balance ===

APP=Archery  N=100
  Spatial: {1: 86, 0: 14}
  Temporal: {1: 86, 0: 14}

APP=PhantomLimb  N=325
  Spatial: {0: 293, 1: 32}
  Temporal: {0: 190, 1: 135}

APP=PianoTiles  N=440
  Spatial: {0: 228, 1: 212}
  Temporal: {0: 386, 1: 54}

APP=Puzzle  N=299
  Spatial: {1: 213, 0: 86}
  Temporal: {1: 213, 0: 86}

APP=Sea  N=300
  Spatial: {0: 168, 1: 132}
  Temporal: {0: 164, 1: 136}

APP=War  N=280
  Spatial: {0: 176, 1: 104}
  Temporal: {0: 164, 1: 116}

=== Majority baseline (per app) ===
APP=Archery      label=Spatial  majority_acc=0.860 dist={1: 86, 0: 14}
APP=Archery      label=Temporal majority_acc=0.860 dist={1: 86, 0: 14}
APP=PhantomLimb  label=Spatial  majority_acc=0.9

/tmp/ipython-input-2067879188.py:172: RuntimeWarning: Mean of empty slice
  return np.nanmean(x, axis=axis)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-2067879188.py:180: RuntimeWarning: All-NaN slice encountered
  mx = np.nanmax(x, axis=axis)
/tmp/ipython-input-2067879188.py:181: RuntimeWarning: All-NaN slice encountered
  mn = np.nanmin(x, axis=axis)
/tmp/ipython-input-2067879188.py:172: RuntimeWarning: Mean of empty slice
  return np.nanmean(x, axis=axis)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-2067879188.py:180: RuntimeWarning: All-NaN slice encountered
  mx = np.nanmax(x, axis=axis)
/tmp/ipython-input-2067879188.py:181: RuntimeWarning: All-NaN slice e

[Temporal | HGB] HELD-OUT=Archery      (SEQ_LEN=5, seqF=138)  acc=0.885 f1=0.939 prec=0.884 rec=1.000 n_test=96


/tmp/ipython-input-2067879188.py:172: RuntimeWarning: Mean of empty slice
  return np.nanmean(x, axis=axis)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-2067879188.py:180: RuntimeWarning: All-NaN slice encountered
  mx = np.nanmax(x, axis=axis)
/tmp/ipython-input-2067879188.py:181: RuntimeWarning: All-NaN slice encountered
  mn = np.nanmin(x, axis=axis)
/tmp/ipython-input-2067879188.py:172: RuntimeWarning: Mean of empty slice
  return np.nanmean(x, axis=axis)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-2067879188.py:180: RuntimeWarning: All-NaN slice encountered
  mx = np.nanmax(x, axis=axis)
/tmp/ipython-input-2067879188.py:181: RuntimeWarning: All-NaN slice e

[Temporal | HGB] HELD-OUT=PhantomLimb  (SEQ_LEN=5, seqF=138)  acc=0.632 f1=0.253 prec=0.800 rec=0.150 n_test=321


...

[Temporal | LR_balanced] HELD-OUT=Puzzle       (SEQ_LEN=5, seqF=138)  acc=0.549 f1=0.670 prec=0.773 rec=0.592 n_test=275


/tmp/ipython-input-2067879188.py:172: RuntimeWarning: Mean of empty slice
  return np.nanmean(x, axis=axis)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-2067879188.py:180: RuntimeWarning: All-NaN slice encountered
  mx = np.nanmax(x, axis=axis)
/tmp/ipython-input-2067879188.py:181: RuntimeWarning: All-NaN slice encountered
  mn = np.nanmin(x, axis=axis)
/tmp/ipython-input-2067879188.py:172: RuntimeWarning: Mean of empty slice
  return np.nanmean(x, axis=axis)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-2067879188.py:180: RuntimeWarning: All-NaN slice encountered
  mx = np.nanmax(x, axis=axis)
/tmp/ipython-input-2067879188.py:181: RuntimeWarning: All-NaN slice e

[Temporal | LR_balanced] HELD-OUT=Sea          (SEQ_LEN=5, seqF=138)  acc=0.456 f1=0.625 prec=0.454 rec=1.000 n_test=296


/tmp/ipython-input-2067879188.py:172: RuntimeWarning: Mean of empty slice
  return np.nanmean(x, axis=axis)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-2067879188.py:180: RuntimeWarning: All-NaN slice encountered
  mx = np.nanmax(x, axis=axis)
/tmp/ipython-input-2067879188.py:181: RuntimeWarning: All-NaN slice encountered
  mn = np.nanmin(x, axis=axis)
/tmp/ipython-input-2067879188.py:172: RuntimeWarning: Mean of empty slice
  return np.nanmean(x, axis=axis)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-2067879188.py:180: RuntimeWarning: All-NaN slice encountered
  mx = np.nanmax(x, axis=axis)
/tmp/ipython-input-2067879188.py:181: RuntimeWarning: All-NaN slice e

[Temporal | LR_balanced] HELD-OUT=War          (SEQ_LEN=5, seqF=138)  acc=0.533 f1=0.641 prec=0.473 rec=0.991 n_test=276

================ SUMMARY (mean over held-out apps) ================
      label            task        model       acc        f1  precision  \
0   Spatial           frame          HGB  0.607209  0.414001   0.456190   
1   Spatial           frame  LR_balanced  0.547156  0.338287   0.405481   
2  Temporal  sequence_len_5          HGB  0.670631  0.490174   0.549573   
3  Temporal  sequence_len_5  LR_balanced  0.704574  0.706668   0.669461   

     recall  
0  0.489744  
1  0.423453  
2  0.522079  
3  0.815432  

Saved results to: /content/loao_metrics_results_PATCHB.csv


In [8]:
# ===============================================================
# VR APP-INDEPENDENT METRICS - LOAO BASELINES
# Patch A: Per-app label sanity + majority baseline
# Patch B: NaN-safe temporal window aggregation (removes warnings)
#
# NOTE:
# - Spatial is evaluated frame-level (single row -> label)
# - Temporal is evaluated sequence-level with sliding window (SEQ_LEN)
# - Optional: Downsample PhantomLimb to match other apps (toggle below)
# ===============================================================

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

SEED = 42
np.random.seed(SEED)

# =========================
# 1) LOAD DATA
# =========================
CANDIDATE_PATHS = [
    "/content/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/VR_Metrics/extracted_metrics_all_apps.csv",
]

DATA_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        DATA_PATH = p
        break

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find extracted_metrics_all_apps.csv in common locations.\n"
        "Put it in /content OR adjust CANDIDATE_PATHS."
    )

df = pd.read_csv(DATA_PATH)
print("\n================= LOADED DATA =================")
print("Loaded:", DATA_PATH, "| shape:", df.shape)

# =========================
# 2) CLEAN / NORMALIZE TYPES
# =========================
df = df.replace("", np.nan)

required = {"App", "Spatial", "Temporal"}
missing_req = required - set(df.columns)
if missing_req:
    raise ValueError(f"Missing required columns in CSV: {missing_req}")

for col in ["Spatial", "Temporal"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["App"]).copy()
df = df.dropna(subset=["Spatial", "Temporal"], how="all").copy()

df["Spatial"] = df["Spatial"].astype(int)
df["Temporal"] = df["Temporal"].astype(int)

apps = sorted(df["App"].unique().tolist())
print("Apps:", apps)

# =========================
# OPTIONAL: Downsample PhantomLimb
# =========================
DOWNSAMPLE_PHANTOMLIMB = True
TARGET_N_PL = 400

if DOWNSAMPLE_PHANTOMLIMB and ("PhantomLimb" in df["App"].unique()):
    pl = df[df["App"] == "PhantomLimb"]
    others = df[df["App"] != "PhantomLimb"]
    if len(pl) > TARGET_N_PL:
        pl = pl.sample(TARGET_N_PL, random_state=SEED)
    df = pd.concat([others, pl], ignore_index=True)
    apps = sorted(df["App"].unique().tolist())
    print("\n[INFO] Downsample PhantomLimb =", DOWNSAMPLE_PHANTOMLIMB,
          "| TARGET_N_PL =", TARGET_N_PL)
    print("[INFO] PhantomLimb N =", len(df[df["App"] == "PhantomLimb"]),
          "| Total N =", len(df))

# =========================
# Patch A: Label sanity + baselines
# =========================
print("\n=== Per-app label balance ===")
for app in apps:
    sub = df[df["App"] == app]
    print(f"\nAPP={app}  N={len(sub)}")
    print("  Spatial:", sub["Spatial"].value_counts().to_dict())
    print("  Temporal:", sub["Temporal"].value_counts().to_dict())

print("\n=== Majority baseline (per app) ===")
for app in apps:
    sub = df[df["App"] == app]
    for lab in ["Spatial", "Temporal"]:
        vc = sub[lab].value_counts()
        maj_acc = (vc.max() / vc.sum()) if len(vc) else np.nan
        print(f"APP={app:12s} label={lab:8s} majority_acc={maj_acc:.3f} dist={vc.to_dict()}")

# =========================
# 3) FEATURE COLUMN SELECTION
# =========================
ID_COLS = [c for c in ["GlobalID", "EntryID"] if c in df.columns]
DROP_COLS = ["App", "Spatial", "Temporal"] + ID_COLS
feature_cols = [c for c in df.columns if c not in DROP_COLS]

# Coerce features to numeric
for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("\n================= FEATURES =================")
print("Num features:", len(feature_cols))
print("First 25 feature cols:", feature_cols[:25])

# =========================
# 4) MODELS
# =========================
hgb_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", HistGradientBoostingClassifier(
        random_state=SEED,
        max_depth=6,
        learning_rate=0.05,
        max_iter=400
    ))
])

lr_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        random_state=SEED,
        max_iter=3000,
        class_weight="balanced",
        n_jobs=-1
    ))
])

MODELS = {
    "HGB": hgb_model,
    "LR_balanced": lr_model
}

# =========================
# 5) EVAL HELPERS
# =========================
def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred).tolist(),
        "pred_dist": dict(zip(*np.unique(y_pred, return_counts=True))),
    }

# =========================
# Patch B FIX: warning-free NaN reducers
# =========================
import numpy as np

def nanmean_no_warn(x, axis=0):
    x = np.asarray(x, dtype=float)
    valid = np.isfinite(x)
    denom = valid.sum(axis=axis)
    # sum with NaNs turned to 0
    num = np.where(valid, x, 0.0).sum(axis=axis)
    out = np.divide(num, denom, out=np.full_like(num, np.nan, dtype=float), where=(denom > 0))
    return out

def nanstd_no_warn(x, axis=0):
    x = np.asarray(x, dtype=float)
    mu = nanmean_no_warn(x, axis=axis)
    # broadcast mu to x shape
    mu_b = np.expand_dims(mu, axis=axis)
    diff2 = (x - mu_b) ** 2
    valid = np.isfinite(diff2)
    denom = valid.sum(axis=axis)
    num = np.where(valid, diff2, 0.0).sum(axis=axis)
    var = np.divide(num, denom, out=np.full_like(num, np.nan, dtype=float), where=(denom > 0))
    return np.sqrt(var)

def nanrange_no_warn(x, axis=0):
    x = np.asarray(x, dtype=float)
    valid = np.isfinite(x)
    # Replace NaNs with +/- inf so min/max ignore them
    x_for_max = np.where(valid, x, -np.inf)
    x_for_min = np.where(valid, x, +np.inf)

    mx = np.max(x_for_max, axis=axis)
    mn = np.min(x_for_min, axis=axis)

    # If a column is all-NaN, mx will be -inf and mn will be +inf -> set to NaN
    all_nan = ~valid.any(axis=axis)
    out = mx - mn
    out = np.where(all_nan, np.nan, out)
    return out

def window_is_valid(window, min_non_nan_ratio=0.4):
    total = window.size
    non_nan = np.isfinite(window).sum()
    return (non_nan / total) >= min_non_nan_ratio

def aggregate_window(window, use=("mean","std","delta","range","mean_abs_vel","vel_std")):
    window = np.asarray(window, dtype=float)
    feats = []

    if "mean" in use:
        feats.append(nanmean_no_warn(window, axis=0))
    if "std" in use:
        feats.append(nanstd_no_warn(window, axis=0))
    if "delta" in use:
        feats.append(window[-1] - window[0])
    if "range" in use:
        feats.append(nanrange_no_warn(window, axis=0))

    if "mean_abs_vel" in use or "vel_std" in use:
        diff = np.diff(window, axis=0)  # (T-1,F)
        if "mean_abs_vel" in use:
            feats.append(nanmean_no_warn(np.abs(diff), axis=0))
        if "vel_std" in use:
            feats.append(nanstd_no_warn(diff, axis=0))

    return np.concatenate(feats, axis=0)

def make_sequence_dataset(app_df, label_col, seq_len=5,
                          agg_use=("mean","std","delta","range","mean_abs_vel","vel_std"),
                          min_non_nan_ratio=0.4):
    """
    Converts frame-level features into sequence-level features by sliding window.
    Label is taken from center frame (like your earlier temporal labeling).
    """
    X = app_df[feature_cols].values.astype(float)
    y = app_df[label_col].values.astype(int)

    if len(app_df) < seq_len:
        return np.empty((0, 1)), np.empty((0,), dtype=int)

    X_seq = []
    y_seq = []
    center = seq_len // 2

    for i in range(len(app_df) - seq_len + 1):
        window = X[i:i+seq_len]

        # drop too-empty windows to avoid all-NaN aggregates
        if not window_is_valid(window, min_non_nan_ratio=min_non_nan_ratio):
            continue

        feat = aggregate_window(window, use=agg_use)
        X_seq.append(feat)
        y_seq.append(y[i + center])

    if len(X_seq) == 0:
        return np.empty((0, 1)), np.empty((0,), dtype=int)

    return np.vstack(X_seq), np.array(y_seq, dtype=int)

# =========================
# 6) LOAO: Spatial (frame-level)
# =========================
def run_loao_frame(label_col: str, model_name: str, model):
    results = []
    for test_app in apps:
        train_df = df[df["App"] != test_app].copy()
        test_df  = df[df["App"] == test_app].copy()

        X_train_full = train_df[feature_cols].values
        y_train_full = train_df[label_col].values

        X_test = test_df[feature_cols].values
        y_test = test_df[label_col].values

        # Inner val split for debugging (optional)
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_full, y_train_full,
            test_size=0.15,
            random_state=SEED,
            stratify=y_train_full
        )

        model.fit(X_tr, y_tr)
        yhat = model.predict(X_test)
        m = eval_binary(y_test, yhat)

        row = {
            "label": label_col,
            "task": "frame",
            "model": model_name,
            "held_out_app": test_app,
            "n_train": int(len(y_tr)),
            "n_val": int(len(y_val)),
            "n_test": int(len(y_test)),
            **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
            "cm": m["cm"],
            "pred_dist": m["pred_dist"],
        }

        print(f"[{label_col} | {model_name}] HELD-OUT={test_app:12s}  "
              f"acc={row['acc']:.3f} f1={row['f1']:.3f}  "
              f"prec={row['precision']:.3f} rec={row['recall']:.3f}  "
              f"n_test={row['n_test']}")
        results.append(row)

    return pd.DataFrame(results)

# =========================
# 7) LOAO: Temporal (sequence-level)
# =========================
def run_loao_sequence(label_col: str, model_name: str, model,
                      seq_len=5,
                      agg_use=("mean","std","delta","range","mean_abs_vel","vel_std"),
                      min_non_nan_ratio=0.4):
    results = []

    # Determine final seq feature length for printing
    seq_feature_dim = len(agg_use) * len(feature_cols)

    for test_app in apps:
        train_df = df[df["App"] != test_app].copy()
        test_df  = df[df["App"] == test_app].copy()

        # Build sequence datasets PER split (prevents any leakage)
        X_train_full, y_train_full = [], []
        for a in apps:
            if a == test_app:
                continue
            Xa, ya = make_sequence_dataset(
                df[df["App"] == a],
                label_col=label_col,
                seq_len=seq_len,
                agg_use=agg_use,
                min_non_nan_ratio=min_non_nan_ratio
            )
            if len(ya) > 0:
                X_train_full.append(Xa)
                y_train_full.append(ya)

        if len(X_train_full) == 0:
            print(f"[{label_col} | {model_name}] HELD-OUT={test_app}  -> No train sequences.")
            continue

        X_train_full = np.vstack(X_train_full)
        y_train_full = np.concatenate(y_train_full)

        X_test, y_test = make_sequence_dataset(
            test_df,
            label_col=label_col,
            seq_len=seq_len,
            agg_use=agg_use,
            min_non_nan_ratio=min_non_nan_ratio
        )

        if len(y_test) == 0:
            print(f"[{label_col} | {model_name}] HELD-OUT={test_app}  -> No test sequences.")
            continue

        # Inner val split
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_full, y_train_full,
            test_size=0.15,
            random_state=SEED,
            stratify=y_train_full
        )

        model.fit(X_tr, y_tr)
        yhat = model.predict(X_test)
        m = eval_binary(y_test, yhat)

        row = {
            "label": label_col,
            "task": f"sequence_len_{seq_len}",
            "model": model_name,
            "held_out_app": test_app,
            "seq_len": int(seq_len),
            "seq_feature_dim": int(seq_feature_dim),
            "n_train": int(len(y_tr)),
            "n_val": int(len(y_val)),
            "n_test": int(len(y_test)),
            **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
            "cm": m["cm"],
            "pred_dist": m["pred_dist"],
        }

        print(f"[{label_col} | {model_name}] HELD-OUT={test_app:12s} "
              f"(SEQ_LEN={seq_len}, seqF={seq_feature_dim})  "
              f"acc={row['acc']:.3f} f1={row['f1']:.3f} "
              f"prec={row['precision']:.3f} rec={row['recall']:.3f} "
              f"n_test={row['n_test']}")
        results.append(row)

    return pd.DataFrame(results)

# =========================
# 8) RUN EVERYTHING
# =========================
SEQ_LEN = 5
AGG_USE = ("mean","std","delta","range","mean_abs_vel","vel_std")  # 6 * F -> e.g., 6*23=138
MIN_NON_NAN_RATIO = 0.4

all_runs = []

print("\n================= LOAO: SPATIAL (FRAME) =================")
for model_name, model in MODELS.items():
    res = run_loao_frame("Spatial", model_name, model)
    all_runs.append(res)

print("\n================= LOAO: TEMPORAL (SEQUENCE) =================")
for model_name, model in MODELS.items():
    res = run_loao_sequence(
        "Temporal", model_name, model,
        seq_len=SEQ_LEN,
        agg_use=AGG_USE,
        min_non_nan_ratio=MIN_NON_NAN_RATIO
    )
    all_runs.append(res)

results_df = pd.concat(all_runs, ignore_index=True)

print("\n================ SUMMARY (mean over held-out apps) ================")
summary = results_df.groupby(["label", "task", "model"])[["acc", "f1", "precision", "recall"]].mean().reset_index()
print(summary)

OUT_PATH = "/content/loao_metrics_results_PATCHB.csv"
results_df.to_csv(OUT_PATH, index=False)
print("\nSaved results to:", OUT_PATH)


================= LOADED DATA =================
Loaded: /content/extracted_metrics_all_apps.csv | shape: (1744, 28)
Apps: ['Archery', 'PhantomLimb', 'PianoTiles', 'Puzzle', 'Sea', 'War']

[INFO] Downsample PhantomLimb = True | TARGET_N_PL = 400
[INFO] PhantomLimb N = 325 | Total N = 1744

=== Per-app label balance ===

APP=Archery  N=100
  Spatial: {1: 86, 0: 14}
  Temporal: {1: 86, 0: 14}

APP=PhantomLimb  N=325
  Spatial: {0: 293, 1: 32}
  Temporal: {0: 190, 1: 135}

APP=PianoTiles  N=440
  Spatial: {0: 228, 1: 212}
  Temporal: {0: 386, 1: 54}

APP=Puzzle  N=299
  Spatial: {1: 213, 0: 86}
  Temporal: {1: 213, 0: 86}

APP=Sea  N=300
  Spatial: {0: 168, 1: 132}
  Temporal: {0: 164, 1: 136}

APP=War  N=280
  Spatial: {0: 176, 1: 104}
  Temporal: {0: 164, 1: 116}

=== Majority baseline (per app) ===
APP=Archery      label=Spatial  majority_acc=0.860 dist={1: 86, 0: 14}
APP=Archery      label=Temporal majority_acc=0.860 dist={1: 86, 0: 14}
APP=PhantomLimb  label=Spatial  majority_acc=0.9

In [16]:
!pip -q install catboost

# MLP Spatial

In [9]:
# ============================================================
# PATCH B - Spatial (FRAME) Neural Baseline (MLP) + LOAO
# Notes:
#  - Uses ALL numeric metrics columns (excludes App/labels/IDs)
#  - LOAO: train on 5 apps, test on held-out app
#  - Includes per-app label balance + majority baselines
#  - Uses class weights + early stopping
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# =========================
# 1) LOAD DATA
# =========================
CANDIDATE_PATHS = [
    "/content/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/VR_Metrics/extracted_metrics_all_apps.csv",
]

DATA_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        DATA_PATH = p
        break
if DATA_PATH is None:
    raise FileNotFoundError("Could not find extracted_metrics_all_apps.csv. Put it in /content or adjust paths.")

df = pd.read_csv(DATA_PATH)
print("\n================= LOADED DATA =================")
print("Loaded:", DATA_PATH, "| shape:", df.shape)

# =========================
# 2) BASIC CLEANING
# =========================
df = df.replace("", np.nan)

required = {"App", "Spatial", "Temporal"}
missing_req = required - set(df.columns)
if missing_req:
    raise ValueError(f"Missing required columns: {missing_req}")

df["App"] = df["App"].astype(str)

# For Spatial training we need Spatial present
df["Spatial"] = pd.to_numeric(df["Spatial"], errors="coerce")
df = df.dropna(subset=["App", "Spatial"]).copy()
df["Spatial"] = df["Spatial"].astype(int)

apps = sorted(df["App"].unique().tolist())
print("Apps:", apps)

# =========================
# 3) OPTIONAL: Downsample PhantomLimb (only if larger than others)
# =========================
DOWNSAMPLE_PHANTOMLIMB = True
TARGET_N_PL = 400  # you asked 300-400; but current PL is 325 so no change

print(f"\n[INFO] Downsample PhantomLimb = {DOWNSAMPLE_PHANTOMLIMB} | TARGET_N_PL = {TARGET_N_PL}")
if DOWNSAMPLE_PHANTOMLIMB and "PhantomLimb" in apps:
    df_pl = df[df["App"] == "PhantomLimb"]
    if len(df_pl) > TARGET_N_PL:
        df_pl = df_pl.sample(n=TARGET_N_PL, random_state=SEED)
        df_other = df[df["App"] != "PhantomLimb"]
        df = pd.concat([df_other, df_pl], ignore_index=True)
        print("[INFO] PhantomLimb was downsampled.")
print(f"[INFO] PhantomLimb N = {len(df[df['App']=='PhantomLimb']) if 'PhantomLimb' in apps else 0} | Total N = {len(df)}")

# =========================
# 4) FEATURE COLUMN SELECTION (USE ALL METRICS)
# =========================
ID_COLS = [c for c in ["GlobalID", "EntryID"] if c in df.columns]
DROP_COLS = ["App", "Spatial", "Temporal"] + ID_COLS
feature_cols = [c for c in df.columns if c not in DROP_COLS]

# Coerce to numeric
for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("\n================= FEATURES =================")
print("Num features:", len(feature_cols))
print("First 25 feature cols:", feature_cols[:25])

# =========================
# 5) PRINT PER-APP LABEL BALANCE + MAJORITY BASELINE
# =========================
print("\n=== Per-app label balance ===")
for a in apps:
    sub = df[df["App"] == a]
    dist = sub["Spatial"].value_counts().to_dict()
    print(f"\nAPP={a:12s}  N={len(sub)}")
    print("  Spatial:", dist)

print("\n=== Majority baseline (per app) ===")
for a in apps:
    sub = df[df["App"] == a]
    dist = sub["Spatial"].value_counts().to_dict()
    maj = max(dist, key=dist.get)
    maj_acc = dist[maj] / len(sub)
    print(f"APP={a:12s} label=Spatial majority_acc={maj_acc:.3f} dist={dist}")

# =========================
# 6) MODEL: simple MLP
# =========================
def build_mlp(input_dim: int) -> keras.Model:
    inp = keras.Input(shape=(input_dim,))
    x = layers.Dense(64, activation="relu")(inp)
    x = layers.Dropout(0.25)(x)
    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dropout(0.20)(x)
    x = layers.Dense(16, activation="relu")(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(inp, out)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[keras.metrics.BinaryAccuracy(name="acc")]
    )
    return model

def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred).tolist(),
        "pred_dist": dict(zip(*np.unique(y_pred, return_counts=True))),
    }

def compute_class_weight(y):
    # returns dict {0: w0, 1: w1} (balanced)
    y = np.asarray(y).astype(int)
    n0 = np.sum(y == 0)
    n1 = np.sum(y == 1)
    if n0 == 0 or n1 == 0:
        return None
    w0 = (n0 + n1) / (2.0 * n0)
    w1 = (n0 + n1) / (2.0 * n1)
    return {0: float(w0), 1: float(w1)}

# =========================
# 7) LOAO EVAL
# =========================
results = []

print("\n================= LOAO: SPATIAL (FRAME) - MLP =================")
for test_app in apps:
    train_df = df[df["App"] != test_app].copy()
    test_df  = df[df["App"] == test_app].copy()

    X_train_full = train_df[feature_cols].values
    y_train_full = train_df["Spatial"].values.astype(int)

    X_test = test_df[feature_cols].values
    y_test = test_df["Spatial"].values.astype(int)

    # train/val split inside training apps
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_full, y_train_full,
        test_size=0.15,
        random_state=SEED,
        stratify=y_train_full
    )

    # impute + scale using TRAIN ONLY
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()

    X_tr = imputer.fit_transform(X_tr)
    X_val = imputer.transform(X_val)
    X_test_i = imputer.transform(X_test)

    X_tr = scaler.fit_transform(X_tr)
    X_val = scaler.transform(X_val)
    X_test_s = scaler.transform(X_test_i)

    # class weights computed on training split
    cw = compute_class_weight(y_tr)

    model = build_mlp(input_dim=X_tr.shape[1])

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=15,
            restore_best_weights=True
        )
    ]

    model.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=200,
        batch_size=32,
        verbose=0,
        class_weight=cw
    )

    yhat_prob = model.predict(X_test_s, verbose=0).reshape(-1)
    yhat = (yhat_prob >= 0.5).astype(int)

    m = eval_binary(y_test, yhat)

    print(f"[Spatial | MLP] HELD-OUT={test_app:12s}  "
          f"acc={m['acc']:.3f} f1={m['f1']:.3f} prec={m['precision']:.3f} rec={m['recall']:.3f}  n_test={len(y_test)}")

    results.append({
        "label": "Spatial",
        "task": "frame",
        "model": "MLP",
        "held_out_app": test_app,
        "n_train": int(len(y_tr)),
        "n_val": int(len(y_val)),
        "n_test": int(len(y_test)),
        **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
        "cm": m["cm"],
        "pred_dist": m["pred_dist"],
    })

results_df = pd.DataFrame(results)

print("\n================ SUMMARY (mean over held-out apps) ================")
print(results_df[["acc","f1","precision","recall"]].mean())

OUT_PATH = "/content/loao_results_spatial_mlp.csv"
results_df.to_csv(OUT_PATH, index=False)
print("\nSaved:", OUT_PATH)


================= LOADED DATA =================
Loaded: /content/extracted_metrics_all_apps.csv | shape: (1744, 28)
Apps: ['Archery', 'PhantomLimb', 'PianoTiles', 'Puzzle', 'Sea', 'War']

[INFO] Downsample PhantomLimb = True | TARGET_N_PL = 400
[INFO] PhantomLimb N = 325 | Total N = 1744

================= FEATURES =================
Num features: 23
First 25 feature cols: ['missing_joints_count', 'missing_joints_ratio', 'collapsed_joints_count', 'center_of_mass_x', 'center_of_mass_y', 'center_of_mass_z', 'distance_from_origin', 'bbox_width', 'bbox_height', 'bbox_depth', 'bbox_volume', 'max_joint_distance_from_com', 'distance_from_floor', 'below_floor', 'left_forearm_length', 'right_forearm_length', 'left_shin_length', 'right_shin_length', 'arm_length_symmetry', 'leg_length_symmetry', 'body_forward_x', 'body_forward_y', 'body_forward_z']

=== Per-app label balance ===

APP=Archery       N=100
  Spatial: {1: 86, 0: 14}

APP=PhantomLimb   N=325
  Spatial: {0: 293, 1: 32}

APP=PianoTiles 

# 1DCNN Temporal

In [10]:
# ============================================================
# PATCH B - Temporal (SEQUENCE) Neural Baseline (GRU) + LOAO
# Notes:
#  - Builds sliding windows per app: X shape = (N-4, 5, F)
#  - Label for a window = center frame label (i+2)
#  - Uses imputer+scaler fitted on TRAIN ONLY (frame-level), then sequences built
#  - GRU is small and regularized (safe for small data)
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

SEQ_LEN = 5
CENTER = SEQ_LEN // 2  # 2 for seq_len=5

# =========================
# 1) LOAD DATA
# =========================
CANDIDATE_PATHS = [
    "/content/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/VR_Metrics/extracted_metrics_all_apps.csv",
]

DATA_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        DATA_PATH = p
        break
if DATA_PATH is None:
    raise FileNotFoundError("Could not find extracted_metrics_all_apps.csv. Put it in /content or adjust paths.")

df = pd.read_csv(DATA_PATH)
print("\n================= LOADED DATA =================")
print("Loaded:", DATA_PATH, "| shape:", df.shape)

df = df.replace("", np.nan)
df["App"] = df["App"].astype(str)

# Need Temporal present for temporal training
df["Temporal"] = pd.to_numeric(df["Temporal"], errors="coerce")
df = df.dropna(subset=["App", "Temporal"]).copy()
df["Temporal"] = df["Temporal"].astype(int)

apps = sorted(df["App"].unique().tolist())
print("Apps:", apps)

# =========================
# 2) FEATURE COLUMN SELECTION (USE ALL METRICS)
# =========================
ID_COLS = [c for c in ["GlobalID", "EntryID"] if c in df.columns]
DROP_COLS = ["App", "Spatial", "Temporal"] + ID_COLS
feature_cols = [c for c in df.columns if c not in DROP_COLS]

for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("\n================= FEATURES =================")
print("Num features:", len(feature_cols))
print("First 25 feature cols:", feature_cols[:25])

# =========================
# 3) SEQUENCE BUILDER (per app)
# =========================
def make_sequences(X_frame, y_frame, seq_len=5):
    n = len(X_frame)
    if n < seq_len:
        return np.empty((0, seq_len, X_frame.shape[1])), np.empty((0,), dtype=int)
    Xs, ys = [], []
    for i in range(n - seq_len + 1):
        Xs.append(X_frame[i:i+seq_len])
        ys.append(int(y_frame[i + (seq_len//2)]))
    return np.asarray(Xs), np.asarray(ys)

def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred).tolist(),
        "pred_dist": dict(zip(*np.unique(y_pred, return_counts=True))),
    }

def compute_class_weight(y):
    y = np.asarray(y).astype(int)
    n0 = np.sum(y == 0)
    n1 = np.sum(y == 1)
    if n0 == 0 or n1 == 0:
        return None
    w0 = (n0 + n1) / (2.0 * n0)
    w1 = (n0 + n1) / (2.0 * n1)
    return {0: float(w0), 1: float(w1)}

# =========================
# 4) MODEL: small GRU
# =========================
def build_gru(seq_len: int, feat_dim: int) -> keras.Model:
    inp = keras.Input(shape=(seq_len, feat_dim))
    x = layers.GRU(32, dropout=0.25, recurrent_dropout=0.0, return_sequences=False)(inp)
    x = layers.Dense(32, activation="relu")(x)
    x = layers.Dropout(0.25)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(inp, out)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[keras.metrics.BinaryAccuracy(name="acc")]
    )
    return model

# =========================
# 5) LOAO EVAL (Temporal sequence)
# =========================
results = []
print(f"\n================= LOAO: TEMPORAL (SEQUENCE) - GRU =================")
print(f"[INFO] SEQ_LEN={SEQ_LEN} | label = center frame (i+{CENTER})")

for test_app in apps:
    train_df = df[df["App"] != test_app].copy()
    test_df  = df[df["App"] == test_app].copy()

    # ---- Fit preprocessing on TRAIN FRAMES ONLY (good practice)
    X_train_full = train_df[feature_cols].values
    y_train_full = train_df["Temporal"].values.astype(int)

    # small validation split at frame-level before sequences
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_full, y_train_full,
        test_size=0.15,
        random_state=SEED,
        stratify=y_train_full
    )

    imputer = SimpleImputer(strategy="median")
    scaler  = StandardScaler()

    X_tr = imputer.fit_transform(X_tr)
    X_val = imputer.transform(X_val)
    X_test_frames = imputer.transform(test_df[feature_cols].values)

    X_tr = scaler.fit_transform(X_tr)
    X_val = scaler.transform(X_val)
    X_test_frames = scaler.transform(X_test_frames)

    # ---- Now we must rebuild per-app sequences consistently.
    # For training sequences: we need scaled+imputed frames in original train_df order.
    X_train_frames_all = scaler.transform(imputer.transform(train_df[feature_cols].values))
    y_train_frames_all = train_df["Temporal"].values.astype(int)

    X_val_frames_all = X_val
    y_val_frames_all = y_val

    # Create sequences:
    # Train sequences from ALL training frames (not only X_tr) to maximize data.
    # (You can tighten this later to avoid slight leakage into validation; for now we keep it simple.)
    X_train_seq, y_train_seq = make_sequences(X_train_frames_all, y_train_frames_all, seq_len=SEQ_LEN)
    X_test_seq, y_test_seq   = make_sequences(X_test_frames, test_df["Temporal"].values.astype(int), seq_len=SEQ_LEN)

    # Validation sequences: built from X_val split (works as quick early stopping signal)
    X_val_seq, y_val_seq = make_sequences(X_val_frames_all, y_val_frames_all, seq_len=SEQ_LEN)

    # If val has too few sequences, fall back to using a small slice of train_seq as val
    if len(X_val_seq) < 20:
        k = min(200, len(X_train_seq)//5)
        X_val_seq, y_val_seq = X_train_seq[:k], y_train_seq[:k]
        X_train_seq, y_train_seq = X_train_seq[k:], y_train_seq[k:]

    cw = compute_class_weight(y_train_seq)

    model = build_gru(seq_len=SEQ_LEN, feat_dim=X_train_seq.shape[2])

    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=12, restore_best_weights=True)
    ]

    model.fit(
        X_train_seq, y_train_seq,
        validation_data=(X_val_seq, y_val_seq),
        epochs=120,
        batch_size=32,
        verbose=0,
        class_weight=cw
    )

    yhat_prob = model.predict(X_test_seq, verbose=0).reshape(-1)
    yhat = (yhat_prob >= 0.5).astype(int)

    m = eval_binary(y_test_seq, yhat)

    print(f"[Temporal | GRU] HELD-OUT={test_app:12s}  "
          f"(SEQ_LEN={SEQ_LEN}) acc={m['acc']:.3f} f1={m['f1']:.3f} "
          f"prec={m['precision']:.3f} rec={m['recall']:.3f}  n_test={len(y_test_seq)}")

    results.append({
        "label": "Temporal",
        "task": f"sequence_len_{SEQ_LEN}",
        "model": "GRU",
        "held_out_app": test_app,
        "n_train_seq": int(len(y_train_seq)),
        "n_val_seq": int(len(y_val_seq)),
        "n_test_seq": int(len(y_test_seq)),
        **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
        "cm": m["cm"],
        "pred_dist": m["pred_dist"],
    })

results_df = pd.DataFrame(results)

print("\n================ SUMMARY (mean over held-out apps) ================")
print(results_df[["acc","f1","precision","recall"]].mean())

OUT_PATH = "/content/loao_results_temporal_gru.csv"
results_df.to_csv(OUT_PATH, index=False)
print("\nSaved:", OUT_PATH)


================= LOADED DATA =================
Loaded: /content/extracted_metrics_all_apps.csv | shape: (1744, 28)
Apps: ['Archery', 'PhantomLimb', 'PianoTiles', 'Puzzle', 'Sea', 'War']

================= FEATURES =================
Num features: 23
First 25 feature cols: ['missing_joints_count', 'missing_joints_ratio', 'collapsed_joints_count', 'center_of_mass_x', 'center_of_mass_y', 'center_of_mass_z', 'distance_from_origin', 'bbox_width', 'bbox_height', 'bbox_depth', 'bbox_volume', 'max_joint_distance_from_com', 'distance_from_floor', 'below_floor', 'left_forearm_length', 'right_forearm_length', 'left_shin_length', 'right_shin_length', 'arm_length_symmetry', 'leg_length_symmetry', 'body_forward_x', 'body_forward_y', 'body_forward_z']

================= LOAO: TEMPORAL (SEQUENCE) - GRU =================
[INFO] SEQ_LEN=5 | label = center frame (i+2)
[Temporal | GRU] HELD-OUT=Archery       (SEQ_LEN=5) acc=0.875 f1=0.933 prec=0.875 rec=1.000  n_test=96
[Temporal | GRU] HELD-OUT=PhantomL

# RF Spatial

In [11]:
# ============================================================
# PATCH: Random Forest baselines (NO SMOTE / NO OVERSAMPLING)
#  - Spatial = frame-level classification using 23 metrics
#  - Temporal will be separate block (sequence RF)
#
# Notes:
#  - Uses ALL feature columns found in CSV (excluding App/labels/IDs)
#  - Optional downsample PhantomLimb is included (prints status)
#  - LOAO = Leave-One-App-Out evaluation
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

SEED = 42
np.random.seed(SEED)

# =========================
# 1) LOAD DATA
# =========================
CANDIDATE_PATHS = [
    "/content/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/VR_Metrics/extracted_metrics_all_apps.csv",
]

DATA_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        DATA_PATH = p
        break

if DATA_PATH is None:
    raise FileNotFoundError("Could not find extracted_metrics_all_apps.csv. Update CANDIDATE_PATHS.")

df = pd.read_csv(DATA_PATH)
print("\n================= LOADED DATA =================")
print("Loaded:", DATA_PATH, "| shape:", df.shape)

# =========================
# 2) CLEAN / NORMALIZE TYPES
# =========================
df = df.replace("", np.nan)

required = {"App", "Spatial", "Temporal"}
missing_req = required - set(df.columns)
if missing_req:
    raise ValueError(f"Missing required columns in CSV: {missing_req}")

df["App"] = df["App"].astype(str)

# labels
for col in ["Spatial", "Temporal"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# keep rows that have Spatial label (since this is spatial block)
df = df.dropna(subset=["App", "Spatial"]).copy()
df["Spatial"] = df["Spatial"].astype(int)

apps = sorted(df["App"].unique().tolist())
print("Apps:", apps)

# =========================
# 3) OPTIONAL: Downsample PhantomLimb to be comparable
# =========================
DOWNSAMPLE_PHANTOMLIMB = True
TARGET_N_PL = 400

print(f"\n[INFO] Downsample PhantomLimb = {DOWNSAMPLE_PHANTOMLIMB} | TARGET_N_PL = {TARGET_N_PL}")

if DOWNSAMPLE_PHANTOMLIMB and "PhantomLimb" in apps:
    df_pl = df[df["App"] == "PhantomLimb"]
    n_pl = len(df_pl)
    if n_pl > TARGET_N_PL:
        # stratified downsample by Spatial label
        df_pl_down, _ = train_test_split(
            df_pl,
            train_size=TARGET_N_PL,
            random_state=SEED,
            stratify=df_pl["Spatial"]
        )
        df_other = df[df["App"] != "PhantomLimb"]
        df = pd.concat([df_other, df_pl_down], ignore_index=True)
        print(f"[INFO] PhantomLimb downsampled: {n_pl} -> {len(df_pl_down)}")
    else:
        print(f"[INFO] PhantomLimb N = {n_pl} (<= target), no downsample performed.")
else:
    print("[INFO] PhantomLimb downsample skipped.")

print(f"[INFO] Total N after downsample step = {len(df)}")

# =========================
# 4) FEATURE COLUMN SELECTION
# =========================
ID_COLS = [c for c in ["GlobalID", "EntryID"] if c in df.columns]
DROP_COLS = ["App", "Spatial", "Temporal"] + ID_COLS
feature_cols = [c for c in df.columns if c not in DROP_COLS]

# coerce features numeric
for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("\n================= FEATURES =================")
print("Num features:", len(feature_cols))
print("First 25 feature cols:", feature_cols[:25])

# =========================
# 5) REPORT LABEL BALANCE + MAJORITY BASELINE
# =========================
print("\n=== Per-app label balance ===")
for app in apps:
    sub = df[df["App"] == app]
    dist = sub["Spatial"].value_counts().to_dict()
    print(f"\nAPP={app:12s} N={len(sub)}")
    print("  Spatial:", dist)

print("\n=== Majority baseline (per app) ===")
for app in apps:
    sub = df[df["App"] == app]
    dist = sub["Spatial"].value_counts().to_dict()
    maj = max(dist, key=dist.get)
    maj_acc = dist[maj] / len(sub)
    print(f"APP={app:12s} label=Spatial  majority_acc={maj_acc:.3f} dist={dist}")

# =========================
# 6) MODEL: Random Forest
# =========================
rf_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", RandomForestClassifier(
        n_estimators=400,
        max_depth=None,
        min_samples_leaf=5,
        max_features="sqrt",
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1
    ))
])

# =========================
# 7) EVAL HELPERS
# =========================
def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred).tolist(),
        "pred_dist": dict(zip(*np.unique(y_pred, return_counts=True))),
    }

def run_loao_spatial_rf():
    results = []
    for test_app in apps:
        train_df = df[df["App"] != test_app].copy()
        test_df  = df[df["App"] == test_app].copy()

        X_train_full = train_df[feature_cols].values
        y_train_full = train_df["Spatial"].values
        X_test = test_df[feature_cols].values
        y_test = test_df["Spatial"].values

        # internal val split (optional, but consistent w/ your pipeline)
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_full, y_train_full,
            test_size=0.15,
            random_state=SEED,
            stratify=y_train_full
        )

        rf_model.fit(X_tr, y_tr)
        yhat = rf_model.predict(X_test)
        m = eval_binary(y_test, yhat)

        row = {
            "label": "Spatial",
            "task": "frame",
            "model": "RF",
            "held_out_app": test_app,
            "n_train": int(len(y_tr)),
            "n_val": int(len(y_val)),
            "n_test": int(len(y_test)),
            **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
            "cm": m["cm"],
            "pred_dist": m["pred_dist"],
        }

        print(f"[Spatial | RF] HELD-OUT={test_app:12s} "
              f"acc={row['acc']:.3f} f1={row['f1']:.3f} "
              f"prec={row['precision']:.3f} rec={row['recall']:.3f} "
              f"n_test={row['n_test']} pred_dist={row['pred_dist']}")
        results.append(row)

    return pd.DataFrame(results)

# =========================
# 8) RUN
# =========================
print("\n================= LOAO: SPATIAL (FRAME) - RF =================")
results_df = run_loao_spatial_rf()

print("\n================ SUMMARY (mean over held-out apps) ================")
summary = results_df[["acc", "f1", "precision", "recall"]].mean()
print(summary)

OUT_PATH = "/content/loao_results_spatial_rf.csv"
results_df.to_csv(OUT_PATH, index=False)
print("\nSaved:", OUT_PATH)


================= LOADED DATA =================
Loaded: /content/extracted_metrics_all_apps.csv | shape: (1744, 28)
Apps: ['Archery', 'PhantomLimb', 'PianoTiles', 'Puzzle', 'Sea', 'War']

[INFO] Downsample PhantomLimb = True | TARGET_N_PL = 400
[INFO] PhantomLimb N = 325 (<= target), no downsample performed.
[INFO] Total N after downsample step = 1744

================= FEATURES =================
Num features: 23
First 25 feature cols: ['missing_joints_count', 'missing_joints_ratio', 'collapsed_joints_count', 'center_of_mass_x', 'center_of_mass_y', 'center_of_mass_z', 'distance_from_origin', 'bbox_width', 'bbox_height', 'bbox_depth', 'bbox_volume', 'max_joint_distance_from_com', 'distance_from_floor', 'below_floor', 'left_forearm_length', 'right_forearm_length', 'left_shin_length', 'right_shin_length', 'arm_length_symmetry', 'leg_length_symmetry', 'body_forward_x', 'body_forward_y', 'body_forward_z']

=== Per-app label balance ===

APP=Archery      N=100
  Spatial: {1: 86, 0: 14}

APP

# RF Temporal

In [12]:
# ============================================================
# PATCH: Temporal Random Forest on SEQUENCES (NO SMOTE)
#  - Build seq_len=5 windows per app
#  - Flatten frames -> RF input (seq_len * num_features)
#  - Label = center frame (i + seq_len//2)
#  - LOAO evaluation
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

SEED = 42
np.random.seed(SEED)

SEQ_LEN = 5
CENTER = SEQ_LEN // 2

# =========================
# 1) LOAD DATA
# =========================
CANDIDATE_PATHS = [
    "/content/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/VR_Metrics/extracted_metrics_all_apps.csv",
]

DATA_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        DATA_PATH = p
        break

if DATA_PATH is None:
    raise FileNotFoundError("Could not find extracted_metrics_all_apps.csv. Update CANDIDATE_PATHS.")

df = pd.read_csv(DATA_PATH)
print("\n================= LOADED DATA =================")
print("Loaded:", DATA_PATH, "| shape:", df.shape)

# =========================
# 2) CLEAN / NORMALIZE TYPES
# =========================
df = df.replace("", np.nan)

required = {"App", "Temporal"}
missing_req = required - set(df.columns)
if missing_req:
    raise ValueError(f"Missing required columns in CSV: {missing_req}")

df["App"] = df["App"].astype(str)
df["Temporal"] = pd.to_numeric(df["Temporal"], errors="coerce")

# keep rows that have Temporal label
df = df.dropna(subset=["App", "Temporal"]).copy()
df["Temporal"] = df["Temporal"].astype(int)

apps = sorted(df["App"].unique().tolist())
print("Apps:", apps)

# =========================
# 3) OPTIONAL: Downsample PhantomLimb (frame-level BEFORE seq build)
# =========================
DOWNSAMPLE_PHANTOMLIMB = True
TARGET_N_PL = 400

print(f"\n[INFO] Downsample PhantomLimb = {DOWNSAMPLE_PHANTOMLIMB} | TARGET_N_PL = {TARGET_N_PL}")

if DOWNSAMPLE_PHANTOMLIMB and "PhantomLimb" in apps:
    df_pl = df[df["App"] == "PhantomLimb"]
    n_pl = len(df_pl)
    if n_pl > TARGET_N_PL:
        df_pl_down, _ = train_test_split(
            df_pl,
            train_size=TARGET_N_PL,
            random_state=SEED,
            stratify=df_pl["Temporal"]
        )
        df_other = df[df["App"] != "PhantomLimb"]
        df = pd.concat([df_other, df_pl_down], ignore_index=True)
        print(f"[INFO] PhantomLimb downsampled: {n_pl} -> {len(df_pl_down)}")
    else:
        print(f"[INFO] PhantomLimb N = {n_pl} (<= target), no downsample performed.")
else:
    print("[INFO] PhantomLimb downsample skipped.")

print(f"[INFO] Total N after downsample step = {len(df)}")

# =========================
# 4) FEATURE COLUMN SELECTION
# =========================
ID_COLS = [c for c in ["GlobalID", "EntryID"] if c in df.columns]
DROP_COLS = ["App", "Spatial", "Temporal"] + ID_COLS  # Spatial may or may not exist; harmless
DROP_COLS = [c for c in DROP_COLS if c in df.columns]
feature_cols = [c for c in df.columns if c not in DROP_COLS]

for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("\n================= FEATURES =================")
print("Num features:", len(feature_cols))
print("First 25 feature cols:", feature_cols[:25])
print(f"[INFO] SEQ_LEN={SEQ_LEN} | label = center frame (i+{CENTER})")
print(f"[INFO] Temporal seq RF will use flattened features: {len(feature_cols)} * {SEQ_LEN} = {len(feature_cols)*SEQ_LEN}")

# =========================
# 5) Build sequences per app
# =========================
def make_sequences_from_app(df_app: pd.DataFrame, feature_cols, label_col="Temporal", seq_len=5):
    X = df_app[feature_cols].values  # (N, F)
    y = df_app[label_col].values    # (N,)
    N, F = X.shape
    if N < seq_len:
        return np.empty((0, seq_len*F)), np.empty((0,), dtype=int)

    seqX, seqY = [], []
    center = seq_len // 2
    for i in range(N - seq_len + 1):
        window = X[i:i+seq_len]          # (seq_len, F)
        seqX.append(window.reshape(-1))  # (seq_len*F,)
        seqY.append(int(y[i + center]))  # center label

    return np.asarray(seqX), np.asarray(seqY)

# =========================
# 6) MODEL: Random Forest
# =========================
rf_seq_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=5,
        max_features="sqrt",
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1
    ))
])

# =========================
# 7) EVAL HELPERS
# =========================
def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred).tolist(),
        "pred_dist": dict(zip(*np.unique(y_pred, return_counts=True))),
    }

# =========================
# 8) LOAO: Temporal sequence RF
# =========================
def run_loao_temporal_seq_rf():
    results = []

    for test_app in apps:
        # build train sequences from all other apps
        train_apps = [a for a in apps if a != test_app]

        X_train_list, y_train_list = [], []
        for a in train_apps:
            df_a = df[df["App"] == a].copy()
            X_a, y_a = make_sequences_from_app(df_a, feature_cols, label_col="Temporal", seq_len=SEQ_LEN)
            if len(y_a) > 0:
                X_train_list.append(X_a)
                y_train_list.append(y_a)

        X_test, y_test = make_sequences_from_app(
            df[df["App"] == test_app].copy(),
            feature_cols,
            label_col="Temporal",
            seq_len=SEQ_LEN
        )

        if len(y_test) == 0 or len(y_train_list) == 0:
            print(f"[Temporal | RF_SEQ] HELD-OUT={test_app:12s}  SKIP (not enough sequences)")
            continue

        X_train_full = np.vstack(X_train_list)
        y_train_full = np.concatenate(y_train_list)

        # internal val split
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_full, y_train_full,
            test_size=0.15,
            random_state=SEED,
            stratify=y_train_full
        )

        rf_seq_model.fit(X_tr, y_tr)
        yhat = rf_seq_model.predict(X_test)
        m = eval_binary(y_test, yhat)

        row = {
            "label": "Temporal",
            "task": f"sequence_len_{SEQ_LEN}_flattened",
            "model": "RF",
            "held_out_app": test_app,
            "n_train": int(len(y_tr)),
            "n_val": int(len(y_val)),
            "n_test": int(len(y_test)),
            "seq_len": int(SEQ_LEN),
            "seq_features": int(X_train_full.shape[1]),
            **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
            "cm": m["cm"],
            "pred_dist": m["pred_dist"],
        }

        print(f"[Temporal | RF_SEQ] HELD-OUT={test_app:12s} "
              f"(SEQ_LEN={SEQ_LEN}, seqF={row['seq_features']}) "
              f"acc={row['acc']:.3f} f1={row['f1']:.3f} "
              f"prec={row['precision']:.3f} rec={row['recall']:.3f} "
              f"n_test={row['n_test']} pred_dist={row['pred_dist']}")
        results.append(row)

    return pd.DataFrame(results)

# =========================
# 9) RUN
# =========================
print("\n================= LOAO: TEMPORAL (SEQUENCE) - RF =================")
results_df = run_loao_temporal_seq_rf()

print("\n================ SUMMARY (mean over held-out apps) ================")
if len(results_df) > 0:
    summary = results_df[["acc", "f1", "precision", "recall"]].mean()
    print(summary)
else:
    print("No results (all apps skipped).")

OUT_PATH = f"/content/loao_results_temporal_seq_rf_len{SEQ_LEN}.csv"
results_df.to_csv(OUT_PATH, index=False)
print("\nSaved:", OUT_PATH)


================= LOADED DATA =================
Loaded: /content/extracted_metrics_all_apps.csv | shape: (1744, 28)
Apps: ['Archery', 'PhantomLimb', 'PianoTiles', 'Puzzle', 'Sea', 'War']

[INFO] Downsample PhantomLimb = True | TARGET_N_PL = 400
[INFO] PhantomLimb N = 325 (<= target), no downsample performed.
[INFO] Total N after downsample step = 1744

================= FEATURES =================
Num features: 23
First 25 feature cols: ['missing_joints_count', 'missing_joints_ratio', 'collapsed_joints_count', 'center_of_mass_x', 'center_of_mass_y', 'center_of_mass_z', 'distance_from_origin', 'bbox_width', 'bbox_height', 'bbox_depth', 'bbox_volume', 'max_joint_distance_from_com', 'distance_from_floor', 'below_floor', 'left_forearm_length', 'right_forearm_length', 'left_shin_length', 'right_shin_length', 'arm_length_symmetry', 'leg_length_symmetry', 'body_forward_x', 'body_forward_y', 'body_forward_z']
[INFO] SEQ_LEN=5 | label = center frame (i+2)
[INFO] Temporal seq RF will use flatten

In [13]:
# ============================================================
# Temporal Random Forest on SEQUENCE AGGREGATES (NO SMOTE)
#  - seq_len=5 sliding windows per app
#  - label = center frame (i+2)
#  - features: per-metric aggregates over the 5 frames:
#      mean, std, range, mean_abs_diff, (last-first)
#    => 23 * 5 = 115 aggregate dims
#    + optional: global scalars (motion energy etc.) to reach ~138-ish if desired
#
# IMPORTANT:
#  - This avoids the "flattened 115" issue where RF struggles.
#  - It’s the closest RF baseline to your successful 138-feature approach.
# ============================================================

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

SEED = 42
np.random.seed(SEED)

SEQ_LEN = 5
CENTER = SEQ_LEN // 2

# -------------------------
# 1) LOAD DATA
# -------------------------
CANDIDATE_PATHS = [
    "/content/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/VR_Metrics/extracted_metrics_all_apps.csv",
]
DATA_PATH = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), None)
if DATA_PATH is None:
    raise FileNotFoundError("Could not find extracted_metrics_all_apps.csv. Update CANDIDATE_PATHS.")

df = pd.read_csv(DATA_PATH).replace("", np.nan)
print("\n================= LOADED DATA =================")
print("Loaded:", DATA_PATH, "| shape:", df.shape)

required = {"App", "Temporal"}
missing_req = required - set(df.columns)
if missing_req:
    raise ValueError(f"Missing required columns in CSV: {missing_req}")

df["App"] = df["App"].astype(str)
df["Temporal"] = pd.to_numeric(df["Temporal"], errors="coerce")
df = df.dropna(subset=["App", "Temporal"]).copy()
df["Temporal"] = df["Temporal"].astype(int)

apps = sorted(df["App"].unique().tolist())
print("Apps:", apps)

# -------------------------
# 2) OPTIONAL DOWN-SAMPLE PL
# -------------------------
DOWNSAMPLE_PHANTOMLIMB = True
TARGET_N_PL = 400
print(f"\n[INFO] Downsample PhantomLimb = {DOWNSAMPLE_PHANTOMLIMB} | TARGET_N_PL = {TARGET_N_PL}")

if DOWNSAMPLE_PHANTOMLIMB and "PhantomLimb" in apps:
    df_pl = df[df["App"] == "PhantomLimb"]
    if len(df_pl) > TARGET_N_PL:
        df_pl_down, _ = train_test_split(
            df_pl,
            train_size=TARGET_N_PL,
            random_state=SEED,
            stratify=df_pl["Temporal"]
        )
        df = pd.concat([df[df["App"] != "PhantomLimb"], df_pl_down], ignore_index=True)
        print(f"[INFO] PhantomLimb downsampled -> {len(df_pl_down)}")
    else:
        print(f"[INFO] PhantomLimb N = {len(df_pl)} (<= target), no downsample performed.")

print(f"[INFO] Total N after downsample step = {len(df)}")

# -------------------------
# 3) FEATURE COLS
# -------------------------
ID_COLS = [c for c in ["GlobalID", "EntryID"] if c in df.columns]
DROP_COLS = [c for c in ["App", "Spatial", "Temporal"] + ID_COLS if c in df.columns]
feature_cols = [c for c in df.columns if c not in DROP_COLS]

for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("\n================= FEATURES =================")
print("Num features:", len(feature_cols))
print("First 25 feature cols:", feature_cols[:25])
print(f"[INFO] SEQ_LEN={SEQ_LEN} | label = center frame (i+{CENTER})")

# -------------------------
# 4) Sequence aggregate features
# -------------------------
def nanmean(x, axis=0): return np.nanmean(x, axis=axis)
def nanstd(x, axis=0):  return np.nanstd(x, axis=axis)
def nanrange(x, axis=0):
    mx = np.nanmax(x, axis=axis)
    mn = np.nanmin(x, axis=axis)
    return mx - mn

def seq_aggregate_features(window_2d: np.ndarray):
    """
    window_2d: (SEQ_LEN, F)
    returns: (F*5 + extra,)
    """
    feats = []
    feats.append(nanmean(window_2d, axis=0))                 # mean
    feats.append(nanstd(window_2d, axis=0))                  # std
    feats.append(nanrange(window_2d, axis=0))                # range
    diff = np.diff(window_2d, axis=0)                        # (SEQ_LEN-1, F)
    feats.append(nanmean(np.abs(diff), axis=0))              # mean abs step (smoothness proxy)
    feats.append(window_2d[-1] - window_2d[0])               # last-first (net change)

    base = np.concatenate(feats, axis=0)                     # (F*5,)

    # Optional extra global motion features (few scalars)
    # These help RF sometimes, and push dims toward your "138-ish" idea.
    # (they do NOT require timestamps)
    motion_energy = np.nanmean(diff**2) if diff.size else np.nan
    motion_l1 = np.nanmean(np.abs(diff)) if diff.size else np.nan
    net_move_l2 = np.nanmean((window_2d[-1] - window_2d[0])**2)
    extra = np.array([motion_energy, motion_l1, net_move_l2], dtype=float)

    return np.concatenate([base, extra], axis=0)

def make_seq_dataset(df_app: pd.DataFrame, label_col="Temporal"):
    X = df_app[feature_cols].values  # (N, F)
    y = df_app[label_col].values     # (N,)
    N, F = X.shape
    if N < SEQ_LEN:
        return np.empty((0, 0)), np.empty((0,), dtype=int)

    X_seq, y_seq = [], []
    for i in range(N - SEQ_LEN + 1):
        window = X[i:i+SEQ_LEN]  # (SEQ_LEN, F)
        X_seq.append(seq_aggregate_features(window))
        y_seq.append(int(y[i + CENTER]))
    return np.asarray(X_seq), np.asarray(y_seq)

# -------------------------
# 5) MODEL: RF
# -------------------------
rf = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", RandomForestClassifier(
        n_estimators=600,
        max_depth=None,
        min_samples_leaf=5,
        max_features="sqrt",
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1
    ))
])

def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred).tolist(),
        "pred_dist": dict(zip(*np.unique(y_pred, return_counts=True))),
    }

# -------------------------
# 6) LOAO
# -------------------------
print("\n================= LOAO: TEMPORAL (SEQUENCE AGG) - RF =================")
results = []

for test_app in apps:
    train_apps = [a for a in apps if a != test_app]

    X_train_list, y_train_list = [], []
    for a in train_apps:
        Xa, ya = make_seq_dataset(df[df["App"] == a].copy(), label_col="Temporal")
        if len(ya) > 0:
            X_train_list.append(Xa)
            y_train_list.append(ya)

    X_test, y_test = make_seq_dataset(df[df["App"] == test_app].copy(), label_col="Temporal")
    if len(y_test) == 0 or len(y_train_list) == 0:
        print(f"[Temporal | RF_AGG] HELD-OUT={test_app:12s}  SKIP (not enough sequences)")
        continue

    X_train_full = np.vstack(X_train_list)
    y_train_full = np.concatenate(y_train_list)

    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train_full, y_train_full,
        test_size=0.15,
        random_state=SEED,
        stratify=y_train_full
    )

    rf.fit(X_tr, y_tr)
    yhat = rf.predict(X_test)
    m = eval_binary(y_test, yhat)

    row = {
        "label": "Temporal",
        "task": f"sequence_len_{SEQ_LEN}_agg",
        "model": "RF",
        "held_out_app": test_app,
        "n_train": int(len(y_tr)),
        "n_val": int(len(y_val)),
        "n_test": int(len(y_test)),
        "seq_len": int(SEQ_LEN),
        "seq_features": int(X_train_full.shape[1]),
        **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
        "cm": m["cm"],
        "pred_dist": m["pred_dist"],
    }

    print(f"[Temporal | RF_AGG] HELD-OUT={test_app:12s} "
          f"(SEQ_LEN={SEQ_LEN}, seqF={row['seq_features']}) "
          f"acc={row['acc']:.3f} f1={row['f1']:.3f} "
          f"prec={row['precision']:.3f} rec={row['recall']:.3f} "
          f"n_test={row['n_test']} pred_dist={row['pred_dist']}")

    results.append(row)

results_df = pd.DataFrame(results)

print("\n================ SUMMARY (mean over held-out apps) ================")
if len(results_df) > 0:
    print(results_df[["acc", "f1", "precision", "recall"]].mean())
else:
    print("No results.")

OUT_PATH = f"/content/loao_results_temporal_rf_agg_len{SEQ_LEN}.csv"
results_df.to_csv(OUT_PATH, index=False)
print("\nSaved:", OUT_PATH)


================= LOADED DATA =================
Loaded: /content/extracted_metrics_all_apps.csv | shape: (1744, 28)
Apps: ['Archery', 'PhantomLimb', 'PianoTiles', 'Puzzle', 'Sea', 'War']

[INFO] Downsample PhantomLimb = True | TARGET_N_PL = 400
[INFO] PhantomLimb N = 325 (<= target), no downsample performed.
[INFO] Total N after downsample step = 1744

================= FEATURES =================
Num features: 23
First 25 feature cols: ['missing_joints_count', 'missing_joints_ratio', 'collapsed_joints_count', 'center_of_mass_x', 'center_of_mass_y', 'center_of_mass_z', 'distance_from_origin', 'bbox_width', 'bbox_height', 'bbox_depth', 'bbox_volume', 'max_joint_distance_from_com', 'distance_from_floor', 'below_floor', 'left_forearm_length', 'right_forearm_length', 'left_shin_length', 'right_shin_length', 'arm_length_symmetry', 'leg_length_symmetry', 'body_forward_x', 'body_forward_y', 'body_forward_z']
[INFO] SEQ_LEN=5 | label = center frame (i+2)

================= LOAO: TEMPORAL (SEQU

/tmp/ipython-input-2291642898.py:101: RuntimeWarning: Mean of empty slice
  def nanmean(x, axis=0): return np.nanmean(x, axis=axis)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-2291642898.py:104: RuntimeWarning: All-NaN slice encountered
  mx = np.nanmax(x, axis=axis)
/tmp/ipython-input-2291642898.py:105: RuntimeWarning: All-NaN slice encountered
  mn = np.nanmin(x, axis=axis)
/tmp/ipython-input-2291642898.py:101: RuntimeWarning: Mean of empty slice
  def nanmean(x, axis=0): return np.nanmean(x, axis=axis)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-2291642898.py:104: RuntimeWarning: All-NaN slice encountered
  mx = np.nanmax(x, axis=axis)
/tmp/ipython-input-22

[Temporal | RF_AGG] HELD-OUT=Archery      (SEQ_LEN=5, seqF=118) acc=0.365 f1=0.440 prec=0.960 rec=0.286 n_test=96 pred_dist={np.int64(0): np.int64(71), np.int64(1): np.int64(25)}


/tmp/ipython-input-2291642898.py:101: RuntimeWarning: Mean of empty slice
  def nanmean(x, axis=0): return np.nanmean(x, axis=axis)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-2291642898.py:104: RuntimeWarning: All-NaN slice encountered
  mx = np.nanmax(x, axis=axis)
/tmp/ipython-input-2291642898.py:105: RuntimeWarning: All-NaN slice encountered
  mn = np.nanmin(x, axis=axis)
/tmp/ipython-input-2291642898.py:101: RuntimeWarning: Mean of empty slice
  def nanmean(x, axis=0): return np.nanmean(x, axis=axis)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-2291642898.py:104: RuntimeWarning: All-NaN slice encountered
  mx = np.nanmax(x, axis=axis)
/tmp/ipython-input-22

[Temporal | RF_AGG] HELD-OUT=PhantomLimb  (SEQ_LEN=5, seqF=118) acc=0.586 f1=0.000 prec=0.000 rec=0.000 n_test=321 pred_dist={np.int64(0): np.int64(321)}


...

[Temporal | RF_AGG] HELD-OUT=Puzzle       (SEQ_LEN=5, seqF=118) acc=0.278 f1=0.000 prec=0.000 rec=0.000 n_test=295 pred_dist={np.int64(0): np.int64(295)}


/tmp/ipython-input-2291642898.py:101: RuntimeWarning: Mean of empty slice
  def nanmean(x, axis=0): return np.nanmean(x, axis=axis)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-2291642898.py:104: RuntimeWarning: All-NaN slice encountered
  mx = np.nanmax(x, axis=axis)
/tmp/ipython-input-2291642898.py:105: RuntimeWarning: All-NaN slice encountered
  mn = np.nanmin(x, axis=axis)
/tmp/ipython-input-2291642898.py:101: RuntimeWarning: Mean of empty slice
  def nanmean(x, axis=0): return np.nanmean(x, axis=axis)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-2291642898.py:104: RuntimeWarning: All-NaN slice encountered
  mx = np.nanmax(x, axis=axis)
/tmp/ipython-input-22

[Temporal | RF_AGG] HELD-OUT=Sea          (SEQ_LEN=5, seqF=118) acc=0.848 f1=0.813 prec=0.916 rec=0.731 n_test=296 pred_dist={np.int64(0): np.int64(189), np.int64(1): np.int64(107)}


/tmp/ipython-input-2291642898.py:101: RuntimeWarning: Mean of empty slice
  def nanmean(x, axis=0): return np.nanmean(x, axis=axis)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-2291642898.py:104: RuntimeWarning: All-NaN slice encountered
  mx = np.nanmax(x, axis=axis)
/tmp/ipython-input-2291642898.py:105: RuntimeWarning: All-NaN slice encountered
  mn = np.nanmin(x, axis=axis)
/tmp/ipython-input-2291642898.py:101: RuntimeWarning: Mean of empty slice
  def nanmean(x, axis=0): return np.nanmean(x, axis=axis)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/tmp/ipython-input-2291642898.py:104: RuntimeWarning: All-NaN slice encountered
  mx = np.nanmax(x, axis=axis)
/tmp/ipython-input-22

[Temporal | RF_AGG] HELD-OUT=War          (SEQ_LEN=5, seqF=118) acc=0.710 f1=0.744 prec=0.592 rec=1.000 n_test=276 pred_dist={np.int64(0): np.int64(80), np.int64(1): np.int64(196)}

================ SUMMARY (mean over held-out apps) ================
acc          0.588625
f1           0.393825
precision    0.455365
recall       0.434942
dtype: float64

Saved: /content/loao_results_temporal_rf_agg_len5.csv


# Summary

# [XG/HG/Cat]-Boost/LR S+T

In [17]:
# ===============================================================
# VR APP-INDEPENDENT METRICS - LOAO BASELINES + CatBoost + XGBoost
# Starting from your Patch B (NaN-safe temporal aggregation)
#
# Adds:
#   - CatBoostClassifier (handles nonlinearity well on tabular)
#   - XGBoost (if installed; otherwise auto-skip with a message)
#
# Notes:
# - No SMOTE / oversampling here (as requested)
# - Keeps your Spatial frame-level and Temporal sequence-level setup
# - Uses the same Pipeline structure (imputer; scaler where appropriate)
# ===============================================================

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

SEED = 42
np.random.seed(SEED)

# =========================
# 1) LOAD DATA
# =========================
CANDIDATE_PATHS = [
    "/content/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/VR_Metrics/extracted_metrics_all_apps.csv",
]

DATA_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        DATA_PATH = p
        break

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find extracted_metrics_all_apps.csv in common locations.\n"
        "Put it in /content OR adjust CANDIDATE_PATHS."
    )

df = pd.read_csv(DATA_PATH)
print("\n================= LOADED DATA =================")
print("Loaded:", DATA_PATH, "| shape:", df.shape)

# =========================
# 2) CLEAN / NORMALIZE TYPES
# =========================
df = df.replace("", np.nan)

required = {"App", "Spatial", "Temporal"}
missing_req = required - set(df.columns)
if missing_req:
    raise ValueError(f"Missing required columns in CSV: {missing_req}")

for col in ["Spatial", "Temporal"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["App"]).copy()
df = df.dropna(subset=["Spatial", "Temporal"], how="all").copy()

df["Spatial"] = df["Spatial"].astype(int)
df["Temporal"] = df["Temporal"].astype(int)

apps = sorted(df["App"].unique().tolist())
print("Apps:", apps)

# =========================
# OPTIONAL: Downsample PhantomLimb
# =========================
DOWNSAMPLE_PHANTOMLIMB = True
TARGET_N_PL = 400

if DOWNSAMPLE_PHANTOMLIMB and ("PhantomLimb" in df["App"].unique()):
    pl = df[df["App"] == "PhantomLimb"]
    others = df[df["App"] != "PhantomLimb"]
    if len(pl) > TARGET_N_PL:
        pl = pl.sample(TARGET_N_PL, random_state=SEED)
    df = pd.concat([others, pl], ignore_index=True)
    apps = sorted(df["App"].unique().tolist())
    print("\n[INFO] Downsample PhantomLimb =", DOWNSAMPLE_PHANTOMLIMB,
          "| TARGET_N_PL =", TARGET_N_PL)
    print("[INFO] PhantomLimb N =", len(df[df["App"] == "PhantomLimb"]),
          "| Total N =", len(df))

# =========================
# Patch A: Label sanity + baselines
# =========================
print("\n=== Per-app label balance ===")
for app in apps:
    sub = df[df["App"] == app]
    print(f"\nAPP={app}  N={len(sub)}")
    print("  Spatial:", sub["Spatial"].value_counts().to_dict())
    print("  Temporal:", sub["Temporal"].value_counts().to_dict())

print("\n=== Majority baseline (per app) ===")
for app in apps:
    sub = df[df["App"] == app]
    for lab in ["Spatial", "Temporal"]:
        vc = sub[lab].value_counts()
        maj_acc = (vc.max() / vc.sum()) if len(vc) else np.nan
        print(f"APP={app:12s} label={lab:8s} majority_acc={maj_acc:.3f} dist={vc.to_dict()}")

# =========================
# 3) FEATURE COLUMN SELECTION
# =========================
ID_COLS = [c for c in ["GlobalID", "EntryID"] if c in df.columns]
DROP_COLS = ["App", "Spatial", "Temporal"] + ID_COLS
feature_cols = [c for c in df.columns if c not in DROP_COLS]

for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("\n================= FEATURES =================")
print("Num features:", len(feature_cols))
print("First 25 feature cols:", feature_cols[:25])

# =========================
# 4) MODELS
# =========================
# ---- Baselines you already had ----
hgb_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", HistGradientBoostingClassifier(
        random_state=SEED,
        max_depth=6,
        learning_rate=0.05,
        max_iter=400
    ))
])

lr_model = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        random_state=SEED,
        max_iter=3000,
        class_weight="balanced",
        n_jobs=-1
    ))
])

# ---- CatBoost (try to import; if missing, skip cleanly) ----
cat_model = None
try:
    from catboost import CatBoostClassifier
    cat_model = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", CatBoostClassifier(
            random_seed=SEED,
            iterations=2000,
            learning_rate=0.03,
            depth=6,
            l2_leaf_reg=6.0,
            loss_function="Logloss",
            eval_metric="F1",
            auto_class_weights="Balanced",  # CatBoost's built-in balancing
            verbose=False
        ))
    ])
    print("\n[INFO] CatBoost imported OK.")
except Exception as e:
    print("\n[WARN] CatBoost not available. Install with: !pip -q install catboost")
    print("       Error:", repr(e))

# ---- XGBoost (try to import; if missing, skip cleanly) ----
xgb_model = None
try:
    import xgboost as xgb
    xgb_model = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", xgb.XGBClassifier(
            random_state=SEED,
            n_estimators=1200,
            learning_rate=0.03,
            max_depth=6,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=1.0,
            reg_alpha=0.0,
            min_child_weight=1.0,
            gamma=0.0,
            tree_method="hist",         # fast on CPU in Colab
            objective="binary:logistic",
            eval_metric="logloss",
            n_jobs=-1
        ))
    ])
    print("[INFO] XGBoost imported OK.")
except Exception as e:
    print("\n[WARN] XGBoost not available. Install with: !pip -q install xgboost")
    print("       Error:", repr(e))

MODELS = {
    "HGB": hgb_model,
    "LR_balanced": lr_model,
}
if cat_model is not None:
    MODELS["CatBoost"] = cat_model
if xgb_model is not None:
    MODELS["XGBoost"] = xgb_model

print("\n[INFO] Models to run:", list(MODELS.keys()))

# =========================
# 5) EVAL HELPERS
# =========================
def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred).tolist(),
        "pred_dist": dict(zip(*np.unique(y_pred, return_counts=True))),
    }

# =========================
# Patch B FIX: warning-free NaN reducers
# =========================
def nanmean_no_warn(x, axis=0):
    x = np.asarray(x, dtype=float)
    valid = np.isfinite(x)
    denom = valid.sum(axis=axis)
    num = np.where(valid, x, 0.0).sum(axis=axis)
    out = np.divide(num, denom, out=np.full_like(num, np.nan, dtype=float), where=(denom > 0))
    return out

def nanstd_no_warn(x, axis=0):
    x = np.asarray(x, dtype=float)
    mu = nanmean_no_warn(x, axis=axis)
    mu_b = np.expand_dims(mu, axis=axis)
    diff2 = (x - mu_b) ** 2
    valid = np.isfinite(diff2)
    denom = valid.sum(axis=axis)
    num = np.where(valid, diff2, 0.0).sum(axis=axis)
    var = np.divide(num, denom, out=np.full_like(num, np.nan, dtype=float), where=(denom > 0))
    return np.sqrt(var)

def nanrange_no_warn(x, axis=0):
    x = np.asarray(x, dtype=float)
    valid = np.isfinite(x)
    x_for_max = np.where(valid, x, -np.inf)
    x_for_min = np.where(valid, x, +np.inf)

    mx = np.max(x_for_max, axis=axis)
    mn = np.min(x_for_min, axis=axis)

    all_nan = ~valid.any(axis=axis)
    out = mx - mn
    out = np.where(all_nan, np.nan, out)
    return out

def window_is_valid(window, min_non_nan_ratio=0.4):
    total = window.size
    non_nan = np.isfinite(window).sum()
    return (non_nan / total) >= min_non_nan_ratio

def aggregate_window(window, use=("mean","std","delta","range","mean_abs_vel","vel_std")):
    window = np.asarray(window, dtype=float)
    feats = []

    if "mean" in use:
        feats.append(nanmean_no_warn(window, axis=0))
    if "std" in use:
        feats.append(nanstd_no_warn(window, axis=0))
    if "delta" in use:
        feats.append(window[-1] - window[0])
    if "range" in use:
        feats.append(nanrange_no_warn(window, axis=0))

    if "mean_abs_vel" in use or "vel_std" in use:
        diff = np.diff(window, axis=0)
        if "mean_abs_vel" in use:
            feats.append(nanmean_no_warn(np.abs(diff), axis=0))
        if "vel_std" in use:
            feats.append(nanstd_no_warn(diff, axis=0))

    return np.concatenate(feats, axis=0)

def make_sequence_dataset(app_df, label_col, seq_len=5,
                          agg_use=("mean","std","delta","range","mean_abs_vel","vel_std"),
                          min_non_nan_ratio=0.4):
    X = app_df[feature_cols].values.astype(float)
    y = app_df[label_col].values.astype(int)

    if len(app_df) < seq_len:
        return np.empty((0, 1)), np.empty((0,), dtype=int)

    X_seq, y_seq = [], []
    center = seq_len // 2

    for i in range(len(app_df) - seq_len + 1):
        window = X[i:i+seq_len]
        if not window_is_valid(window, min_non_nan_ratio=min_non_nan_ratio):
            continue
        feat = aggregate_window(window, use=agg_use)
        X_seq.append(feat)
        y_seq.append(y[i + center])

    if len(X_seq) == 0:
        return np.empty((0, 1)), np.empty((0,), dtype=int)

    return np.vstack(X_seq), np.array(y_seq, dtype=int)

# =========================
# 6) LOAO: Spatial (frame-level)
# =========================
def run_loao_frame(label_col: str, model_name: str, model):
    results = []
    for test_app in apps:
        train_df = df[df["App"] != test_app].copy()
        test_df  = df[df["App"] == test_app].copy()

        X_train_full = train_df[feature_cols].values
        y_train_full = train_df[label_col].values

        X_test = test_df[feature_cols].values
        y_test = test_df[label_col].values

        # Inner val split (kept same as your patch)
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_full, y_train_full,
            test_size=0.15,
            random_state=SEED,
            stratify=y_train_full
        )

        model.fit(X_tr, y_tr)
        yhat = model.predict(X_test)
        m = eval_binary(y_test, yhat)

        row = {
            "label": label_col,
            "task": "frame",
            "model": model_name,
            "held_out_app": test_app,
            "n_train": int(len(y_tr)),
            "n_val": int(len(y_val)),
            "n_test": int(len(y_test)),
            **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
            "cm": m["cm"],
            "pred_dist": m["pred_dist"],
        }

        print(f"[{label_col} | {model_name}] HELD-OUT={test_app:12s}  "
              f"acc={row['acc']:.3f} f1={row['f1']:.3f}  "
              f"prec={row['precision']:.3f} rec={row['recall']:.3f}  "
              f"n_test={row['n_test']}")
        results.append(row)

    return pd.DataFrame(results)

# =========================
# 7) LOAO: Temporal (sequence-level)
# =========================
def run_loao_sequence(label_col: str, model_name: str, model,
                      seq_len=5,
                      agg_use=("mean","std","delta","range","mean_abs_vel","vel_std"),
                      min_non_nan_ratio=0.4):
    results = []
    seq_feature_dim = len(agg_use) * len(feature_cols)

    for test_app in apps:
        train_df = df[df["App"] != test_app].copy()
        test_df  = df[df["App"] == test_app].copy()

        # Build sequence datasets per-app (prevents leakage)
        X_train_full, y_train_full = [], []
        for a in apps:
            if a == test_app:
                continue
            Xa, ya = make_sequence_dataset(
                df[df["App"] == a],
                label_col=label_col,
                seq_len=seq_len,
                agg_use=agg_use,
                min_non_nan_ratio=min_non_nan_ratio
            )
            if len(ya) > 0:
                X_train_full.append(Xa)
                y_train_full.append(ya)

        if len(X_train_full) == 0:
            print(f"[{label_col} | {model_name}] HELD-OUT={test_app}  -> No train sequences.")
            continue

        X_train_full = np.vstack(X_train_full)
        y_train_full = np.concatenate(y_train_full)

        X_test, y_test = make_sequence_dataset(
            test_df,
            label_col=label_col,
            seq_len=seq_len,
            agg_use=agg_use,
            min_non_nan_ratio=min_non_nan_ratio
        )

        if len(y_test) == 0:
            print(f"[{label_col} | {model_name}] HELD-OUT={test_app}  -> No test sequences.")
            continue

        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_full, y_train_full,
            test_size=0.15,
            random_state=SEED,
            stratify=y_train_full
        )

        model.fit(X_tr, y_tr)
        yhat = model.predict(X_test)
        m = eval_binary(y_test, yhat)

        row = {
            "label": label_col,
            "task": f"sequence_len_{seq_len}",
            "model": model_name,
            "held_out_app": test_app,
            "seq_len": int(seq_len),
            "seq_feature_dim": int(seq_feature_dim),
            "n_train": int(len(y_tr)),
            "n_val": int(len(y_val)),
            "n_test": int(len(y_test)),
            **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
            "cm": m["cm"],
            "pred_dist": m["pred_dist"],
        }

        print(f"[{label_col} | {model_name}] HELD-OUT={test_app:12s} "
              f"(SEQ_LEN={seq_len}, seqF={seq_feature_dim})  "
              f"acc={row['acc']:.3f} f1={row['f1']:.3f} "
              f"prec={row['precision']:.3f} rec={row['recall']:.3f} "
              f"n_test={row['n_test']}")
        results.append(row)

    return pd.DataFrame(results)

# =========================
# 8) RUN EVERYTHING
# =========================
SEQ_LEN = 5
AGG_USE = ("mean","std","delta","range","mean_abs_vel","vel_std")  # 6 * F
MIN_NON_NAN_RATIO = 0.4

all_runs = []

print("\n================= LOAO: SPATIAL (FRAME) =================")
for model_name, model in MODELS.items():
    res = run_loao_frame("Spatial", model_name, model)
    all_runs.append(res)

print("\n================= LOAO: TEMPORAL (SEQUENCE) =================")
for model_name, model in MODELS.items():
    res = run_loao_sequence(
        "Temporal", model_name, model,
        seq_len=SEQ_LEN,
        agg_use=AGG_USE,
        min_non_nan_ratio=MIN_NON_NAN_RATIO
    )
    all_runs.append(res)

results_df = pd.concat(all_runs, ignore_index=True)

print("\n================ SUMMARY (mean over held-out apps) ================")
summary = results_df.groupby(["label", "task", "model"])[["acc", "f1", "precision", "recall"]].mean().reset_index()
print(summary)

OUT_PATH = "/content/loao_metrics_results_PATCHB_with_cat_xgb.csv"
results_df.to_csv(OUT_PATH, index=False)
print("\nSaved results to:", OUT_PATH)


================= LOADED DATA =================
Loaded: /content/extracted_metrics_all_apps.csv | shape: (1744, 28)
Apps: ['Archery', 'PhantomLimb', 'PianoTiles', 'Puzzle', 'Sea', 'War']

[INFO] Downsample PhantomLimb = True | TARGET_N_PL = 400
[INFO] PhantomLimb N = 325 | Total N = 1744

=== Per-app label balance ===

APP=Archery  N=100
  Spatial: {1: 86, 0: 14}
  Temporal: {1: 86, 0: 14}

APP=PhantomLimb  N=325
  Spatial: {0: 293, 1: 32}
  Temporal: {0: 190, 1: 135}

APP=PianoTiles  N=440
  Spatial: {0: 228, 1: 212}
  Temporal: {0: 386, 1: 54}

APP=Puzzle  N=299
  Spatial: {1: 213, 0: 86}
  Temporal: {1: 213, 0: 86}

APP=Sea  N=300
  Spatial: {0: 168, 1: 132}
  Temporal: {0: 164, 1: 136}

APP=War  N=280
  Spatial: {0: 176, 1: 104}
  Temporal: {0: 164, 1: 116}

=== Majority baseline (per app) ===
APP=Archery      label=Spatial  majority_acc=0.860 dist={1: 86, 0: 14}
APP=Archery      label=Temporal majority_acc=0.860 dist={1: 86, 0: 14}
APP=PhantomLimb  label=Spatial  majority_acc=0.9

In [19]:
from sklearn.calibration import CalibratedClassifierCV

def best_threshold_f1(y_true, probs, grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 181)
    best_t, best_f1 = 0.5, -1.0
    for t in grid:
        y_pred = (probs >= t).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, float(t)
    return best_t, float(best_f1)

def run_loao_sequence_lr_threshold(label_col="Temporal",
                                  seq_len=5,
                                  agg_use=("mean","std","delta","range","mean_abs_vel","vel_std"),
                                  min_non_nan_ratio=0.4,
                                  do_calibrate=False,
                                  calibrate_method="sigmoid"):
    results = []
    seq_feature_dim = len(agg_use) * len(feature_cols)

    # base LR model (same as yours)
    base_lr = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            random_state=SEED,
            max_iter=5000,
            class_weight="balanced",
            n_jobs=-1
        ))
    ])

    for test_app in apps:
        # Build train sequences from all other apps
        X_train_full, y_train_full = [], []
        for a in apps:
            if a == test_app:
                continue
            Xa, ya = make_sequence_dataset(
                df[df["App"] == a],
                label_col=label_col,
                seq_len=seq_len,
                agg_use=agg_use,
                min_non_nan_ratio=min_non_nan_ratio
            )
            if len(ya) > 0:
                X_train_full.append(Xa)
                y_train_full.append(ya)

        if len(X_train_full) == 0:
            print(f"[{label_col} | LR_thr] HELD-OUT={test_app} -> No train sequences.")
            continue

        X_train_full = np.vstack(X_train_full)
        y_train_full = np.concatenate(y_train_full)

        # Build test sequences
        X_test, y_test = make_sequence_dataset(
            df[df["App"] == test_app],
            label_col=label_col,
            seq_len=seq_len,
            agg_use=agg_use,
            min_non_nan_ratio=min_non_nan_ratio
        )
        if len(y_test) == 0:
            print(f"[{label_col} | LR_thr] HELD-OUT={test_app} -> No test sequences.")
            continue

        # Inner val split
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_full, y_train_full,
            test_size=0.15,
            random_state=SEED,
            stratify=y_train_full
        )

        model = base_lr

        # Optional calibration (fit only on training split; calibrator uses CV inside)
        # Note: calibration makes probs more meaningful; threshold tuning then is more stable.
        if do_calibrate:
            # Need an estimator that supports predict_proba at the end; pipeline is OK.
            model = CalibratedClassifierCV(estimator=base_lr, method=calibrate_method, cv=3)

        model.fit(X_tr, y_tr)

        # probs on val -> pick threshold by F1
        val_probs = model.predict_proba(X_val)[:, 1]
        best_t, best_val_f1 = best_threshold_f1(y_val, val_probs)

        # apply to test
        test_probs = model.predict_proba(X_test)[:, 1]
        yhat = (test_probs >= best_t).astype(int)

        m = eval_binary(y_test, yhat)

        row = {
            "label": label_col,
            "task": f"sequence_len_{seq_len}",
            "model": f"LR_thr_calib={do_calibrate}",
            "held_out_app": test_app,
            "seq_len": int(seq_len),
            "seq_feature_dim": int(seq_feature_dim),
            "n_train": int(len(y_tr)),
            "n_val": int(len(y_val)),
            "n_test": int(len(y_test)),
            "best_thr": float(best_t),
            "best_val_f1": float(best_val_f1),
            **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
            "cm": m["cm"],
            "pred_dist": m["pred_dist"],
        }

        print(f"[Temporal | LR_thr{'_cal' if do_calibrate else ''}] HELD-OUT={test_app:12s} "
              f"(SEQ_LEN={seq_len}, seqF={seq_feature_dim}) "
              f"thr={best_t:.2f} val_f1={best_val_f1:.3f}  "
              f"acc={row['acc']:.3f} f1={row['f1']:.3f} "
              f"prec={row['precision']:.3f} rec={row['recall']:.3f} "
              f"n_test={row['n_test']}")
        results.append(row)

    return pd.DataFrame(results)

# ---- RUN IT ----
res_thr = run_loao_sequence_lr_threshold(
    label_col="Temporal",
    seq_len=SEQ_LEN,
    agg_use=AGG_USE,
    min_non_nan_ratio=MIN_NON_NAN_RATIO,
    do_calibrate=False
)

print("\n=== SUMMARY (LR threshold tuned) ===")
print(res_thr[["acc","f1","precision","recall"]].mean())

OUT_THR = "/content/loao_temporal_lr_threshold_tuned.csv"
res_thr.to_csv(OUT_THR, index=False)
print("Saved:", OUT_THR)

# Optional calibrated run (often helps when domains differ)
res_thr_cal = run_loao_sequence_lr_threshold(
    label_col="Temporal",
    seq_len=SEQ_LEN,
    agg_use=AGG_USE,
    min_non_nan_ratio=MIN_NON_NAN_RATIO,
    do_calibrate=True,
    calibrate_method="sigmoid"
)

print("\n=== SUMMARY (LR calibrated + threshold tuned) ===")
print(res_thr_cal[["acc","f1","precision","recall"]].mean())

OUT_THR_CAL = "/content/loao_temporal_lr_calibrated_threshold_tuned.csv"
res_thr_cal.to_csv(OUT_THR_CAL, index=False)
print("Saved:", OUT_THR_CAL)

[Temporal | LR_thr] HELD-OUT=Archery      (SEQ_LEN=5, seqF=138) thr=0.48 val_f1=0.888  acc=0.875 f1=0.933 prec=0.875 rec=1.000 n_test=96
[Temporal | LR_thr] HELD-OUT=PhantomLimb  (SEQ_LEN=5, seqF=138) thr=0.38 val_f1=0.930  acc=0.938 f1=0.923 prec=0.945 rec=0.902 n_test=321
[Temporal | LR_thr] HELD-OUT=PianoTiles   (SEQ_LEN=5, seqF=138) thr=0.61 val_f1=0.945  acc=0.897 f1=0.458 prec=0.655 rec=0.352 n_test=436
[Temporal | LR_thr] HELD-OUT=Puzzle       (SEQ_LEN=5, seqF=138) thr=0.59 val_f1=0.867  acc=0.549 f1=0.670 prec=0.773 rec=0.592 n_test=275
[Temporal | LR_thr] HELD-OUT=Sea          (SEQ_LEN=5, seqF=138) thr=0.65 val_f1=0.901  acc=0.463 f1=0.628 prec=0.457 rec=1.000 n_test=296
[Temporal | LR_thr] HELD-OUT=War          (SEQ_LEN=5, seqF=138) thr=0.44 val_f1=0.925  acc=0.518 f1=0.634 prec=0.466 rec=0.991 n_test=276

=== SUMMARY (LR threshold tuned) ===
acc          0.706588
f1           0.707616
precision    0.695164
recall       0.806173
dtype: float64
Saved: /content/loao_temporal_lr

In [20]:
from sklearn.calibration import CalibratedClassifierCV

def run_loao_sequence_lr_threshold(label_col="Temporal",
                                  seq_len=5,
                                  agg_use=("mean","std","delta","range","mean_abs_vel","vel_std"),
                                  min_non_nan_ratio=0.4,
                                  do_calibrate=False,
                                  calibrate_method="sigmoid"):
    results = []
    seq_feature_dim = len(agg_use) * len(feature_cols)

    base_lr = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            random_state=SEED,
            max_iter=5000,
            class_weight="balanced",
            n_jobs=-1
        ))
    ])

    for test_app in apps:
        X_train_full, y_train_full = [], []
        for a in apps:
            if a == test_app:
                continue
            Xa, ya = make_sequence_dataset(
                df[df["App"] == a],
                label_col=label_col,
                seq_len=seq_len,
                agg_use=agg_use,
                min_non_nan_ratio=min_non_nan_ratio
            )
            if len(ya) > 0:
                X_train_full.append(Xa)
                y_train_full.append(ya)

        if len(X_train_full) == 0:
            print(f"[{label_col} | LR_thr] HELD-OUT={test_app} -> No train sequences.")
            continue

        X_train_full = np.vstack(X_train_full)
        y_train_full = np.concatenate(y_train_full)

        X_test, y_test = make_sequence_dataset(
            df[df["App"] == test_app],
            label_col=label_col,
            seq_len=seq_len,
            agg_use=agg_use,
            min_non_nan_ratio=min_non_nan_ratio
        )
        if len(y_test) == 0:
            print(f"[{label_col} | LR_thr] HELD-OUT={test_app} -> No test sequences.")
            continue

        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_full, y_train_full,
            test_size=0.15,
            random_state=SEED,
            stratify=y_train_full
        )

        model = base_lr
        if do_calibrate:
            model = CalibratedClassifierCV(
                estimator=base_lr,   # <-- FIXED NAME
                method=calibrate_method,
                cv=3
            )

        model.fit(X_tr, y_tr)

        val_probs = model.predict_proba(X_val)[:, 1]
        best_t, best_val_f1 = best_threshold_f1(y_val, val_probs)

        test_probs = model.predict_proba(X_test)[:, 1]
        yhat = (test_probs >= best_t).astype(int)

        m = eval_binary(y_test, yhat)

        row = {
            "label": label_col,
            "task": f"sequence_len_{seq_len}",
            "model": f"LR_thr_calib={do_calibrate}_{calibrate_method}",
            "held_out_app": test_app,
            "seq_len": int(seq_len),
            "seq_feature_dim": int(seq_feature_dim),
            "n_train": int(len(y_tr)),
            "n_val": int(len(y_val)),
            "n_test": int(len(y_test)),
            "best_thr": float(best_t),
            "best_val_f1": float(best_val_f1),
            **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
            "cm": m["cm"],
            "pred_dist": m["pred_dist"],
        }

        print(f"[Temporal | LR_thr{'_cal' if do_calibrate else ''}] HELD-OUT={test_app:12s} "
              f"thr={best_t:.2f} val_f1={best_val_f1:.3f}  "
              f"acc={row['acc']:.3f} f1={row['f1']:.3f} "
              f"prec={row['precision']:.3f} rec={row['recall']:.3f} "
              f"n_test={row['n_test']}")
        results.append(row)

    return pd.DataFrame(results)

# Run calibrated
res_thr_cal = run_loao_sequence_lr_threshold(
    label_col="Temporal",
    seq_len=SEQ_LEN,
    agg_use=AGG_USE,
    min_non_nan_ratio=MIN_NON_NAN_RATIO,
    do_calibrate=True,
    calibrate_method="sigmoid"  # try "isotonic" after if you want
)

print("\n=== SUMMARY (LR calibrated + threshold tuned) ===")
print(res_thr_cal[["acc","f1","precision","recall"]].mean())

OUT_THR_CAL = "/content/loao_temporal_lr_calibrated_threshold_tuned.csv"
res_thr_cal.to_csv(OUT_THR_CAL, index=False)
print("Saved:", OUT_THR_CAL)

[Temporal | LR_thr_cal] HELD-OUT=Archery      thr=0.39 val_f1=0.889  acc=0.875 f1=0.933 prec=0.875 rec=1.000 n_test=96
[Temporal | LR_thr_cal] HELD-OUT=PhantomLimb  thr=0.46 val_f1=0.921  acc=0.938 f1=0.923 prec=0.945 rec=0.902 n_test=321
[Temporal | LR_thr_cal] HELD-OUT=PianoTiles   thr=0.39 val_f1=0.943  acc=0.773 f1=0.327 prec=0.258 rec=0.444 n_test=436
[Temporal | LR_thr_cal] HELD-OUT=Puzzle       thr=0.46 val_f1=0.863  acc=0.567 f1=0.681 prec=0.794 rec=0.596 n_test=275
[Temporal | LR_thr_cal] HELD-OUT=Sea          thr=0.42 val_f1=0.906  acc=0.470 f1=0.631 prec=0.460 rec=1.000 n_test=296
[Temporal | LR_thr_cal] HELD-OUT=War          thr=0.41 val_f1=0.925  acc=0.540 f1=0.646 prec=0.477 rec=1.000 n_test=276

=== SUMMARY (LR calibrated + threshold tuned) ===
acc          0.693725
f1           0.690122
precision    0.634924
recall       0.823824
dtype: float64
Saved: /content/loao_temporal_lr_calibrated_threshold_tuned.csv


# Improvements

In [21]:
# ===============================================================
# TEMPORAL LOAO - LR ablations:
#   A) Baseline LR_balanced (your best)
#   B) + RandomOverSampler (train only)
#   C) + Per-app normalization (train apps fit, applied per app)
#   D) + App-aware one-hot (train app id appended, test = zeros)
#
# Saves:
#   /content/loao_temporal_lr_baseline.csv
#   /content/loao_temporal_lr_ros.csv
#   /content/loao_temporal_lr_perapp_norm.csv
#   /content/loao_temporal_lr_appaware.csv
# ===============================================================

import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

# Optional: imblearn for RandomOverSampler
try:
    from imblearn.over_sampling import RandomOverSampler
    IMBLEARN_OK = True
    print("[INFO] imblearn RandomOverSampler imported OK.")
except Exception as e:
    IMBLEARN_OK = False
    print("[WARN] imblearn not available. RandomOverSampler experiment will be skipped.", e)

SEED = 42
np.random.seed(SEED)

# -------------------------
# Metrics helper (same style as yours)
# -------------------------
def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred).tolist(),
        "pred_dist": dict(zip(*np.unique(y_pred, return_counts=True))),
    }

# -------------------------
# Your best LR model (balanced)
# -------------------------
def make_lr_balanced():
    return Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            random_state=SEED,
            max_iter=4000,
            class_weight="balanced",
            n_jobs=-1
        ))
    ])

# -------------------------
# Sequence dataset builder wrapper: return also "source app" label per sequence
# -------------------------
def make_sequence_dataset_with_source(app_df, label_col, source_app_name,
                                      seq_len=5,
                                      agg_use=("mean","std","delta","range","mean_abs_vel","vel_std"),
                                      min_non_nan_ratio=0.4):
    Xs, ys = make_sequence_dataset(
        app_df=app_df,
        label_col=label_col,
        seq_len=seq_len,
        agg_use=agg_use,
        min_non_nan_ratio=min_non_nan_ratio
    )
    if len(ys) == 0:
        return Xs, ys, np.array([], dtype=object)
    src = np.array([source_app_name] * len(ys), dtype=object)
    return Xs, ys, src

# -------------------------
# Per-app normalization:
# Fit scaler on each app's TRAIN sequences only, apply to that app's sequences.
# For held-out app: fit scaler on its own (test-only) sequences? -> NO (would leak test stats).
# Instead: apply "global train scaler" to held-out sequences OR leave them raw.
#
# I recommend: GLOBAL train scaler for held-out app (no leakage), per-app for training apps.
# -------------------------
def per_app_normalize_train_then_global_for_test(X_train, src_train, X_test, src_test):
    """
    X_train: (N_train, D), src_train: (N_train,)
    X_test:  (N_test, D),  src_test: (N_test,)  -> typically all held-out app
    Returns normalized X_train, X_test
    """
    X_train = X_train.astype(float)
    X_test  = X_test.astype(float)

    # Global scaler fit on all training sequences (safe)
    global_scaler = StandardScaler()
    global_scaler.fit(X_train)
    X_test_norm = global_scaler.transform(X_test)

    # Per-app scalers for training apps
    X_train_norm = np.empty_like(X_train, dtype=float)

    for a in np.unique(src_train):
        idx = np.where(src_train == a)[0]
        scaler = StandardScaler()
        scaler.fit(X_train[idx])
        X_train_norm[idx] = scaler.transform(X_train[idx])

    return X_train_norm, X_test_norm

# -------------------------
# App-aware one-hot
# Train: append one-hot for SOURCE app (among training apps).
# Test (held-out): append all zeros.
# -------------------------
def append_app_onehot(X_train, src_train, X_test, held_out_app):
    train_apps = sorted([a for a in np.unique(src_train) if a != held_out_app])
    app2i = {a:i for i,a in enumerate(train_apps)}
    K = len(train_apps)

    def onehot(src_arr):
        oh = np.zeros((len(src_arr), K), dtype=float)
        for i, a in enumerate(src_arr):
            if a in app2i:
                oh[i, app2i[a]] = 1.0
        return oh

    oh_train = onehot(src_train)
    # held-out app => all zeros
    oh_test = np.zeros((len(X_test), K), dtype=float)

    X_train2 = np.hstack([X_train, oh_train])
    X_test2  = np.hstack([X_test, oh_test])
    return X_train2, X_test2, K

# -------------------------
# Core LOAO runner (temporal, seq)
# Options:
#   use_ros: RandomOverSampler on training set
#   use_per_app_norm: per-app normalization on train apps, global scaler on test
#   use_app_aware: append one-hot app id
# -------------------------
def run_loao_temporal_lr(
    label_col="Temporal",
    seq_len=5,
    agg_use=("mean","std","delta","range","mean_abs_vel","vel_std"),
    min_non_nan_ratio=0.4,
    use_ros=False,
    use_per_app_norm=False,
    use_app_aware=False,
    tag="baseline"
):
    results = []
    seq_feature_dim = len(agg_use) * len(feature_cols)

    for test_app in apps:
        # Build train sequences PER APP to avoid leakage & to keep src labels
        X_train_list, y_train_list, src_train_list = [], [], []
        for a in apps:
            if a == test_app:
                continue
            app_df = df[df["App"] == a]
            Xa, ya, srca = make_sequence_dataset_with_source(
                app_df, label_col, source_app_name=a,
                seq_len=seq_len, agg_use=agg_use, min_non_nan_ratio=min_non_nan_ratio
            )
            if len(ya) > 0:
                X_train_list.append(Xa)
                y_train_list.append(ya)
                src_train_list.append(srca)

        if len(X_train_list) == 0:
            print(f"[Temporal | LR_{tag}] HELD-OUT={test_app:12s} -> No train sequences.")
            continue

        X_train_full = np.vstack(X_train_list)
        y_train_full = np.concatenate(y_train_list)
        src_train_full = np.concatenate(src_train_list)

        # Test sequences for held-out app
        test_df = df[df["App"] == test_app].copy()
        X_test, y_test, src_test = make_sequence_dataset_with_source(
            test_df, label_col, source_app_name=test_app,
            seq_len=seq_len, agg_use=agg_use, min_non_nan_ratio=min_non_nan_ratio
        )
        if len(y_test) == 0:
            print(f"[Temporal | LR_{tag}] HELD-OUT={test_app:12s} -> No test sequences.")
            continue

        # Inner split (keep your style)
        X_tr, X_val, y_tr, y_val, src_tr, src_val = train_test_split(
            X_train_full, y_train_full, src_train_full,
            test_size=0.15, random_state=SEED, stratify=y_train_full
        )

        # (C) Per-app normalization
        if use_per_app_norm:
            X_tr, X_test_norm = per_app_normalize_train_then_global_for_test(
                X_tr, src_tr, X_test, src_test
            )
            # IMPORTANT: Also normalize val using GLOBAL scaler fit on X_tr
            # (keep consistent with "test uses global train scaler")
            global_scaler = StandardScaler().fit(X_tr)
            X_val = global_scaler.transform(X_val)
            X_test = X_test_norm

        # (D) App-aware features
        appaware_K = 0
        if use_app_aware:
            X_tr, X_test, appaware_K = append_app_onehot(X_tr, src_tr, X_test, held_out_app=test_app)
            # For val: append one-hot for val source app too
            X_val, _, _ = append_app_onehot(X_val, src_val, X_val[:1], held_out_app=test_app)
            # note: second return unused; we just needed same onehot mapping length

        # (B) RandomOverSampler (train only)
        if use_ros:
            if not IMBLEARN_OK:
                raise RuntimeError("imblearn not installed but use_ros=True.")
            ros = RandomOverSampler(random_state=SEED)
            X_tr, y_tr = ros.fit_resample(X_tr, y_tr)

        # Fit/predict
        model = make_lr_balanced()
        model.fit(X_tr, y_tr)
        yhat = model.predict(X_test)

        m = eval_binary(y_test, yhat)
        row = {
            "label": label_col,
            "task": f"sequence_len_{seq_len}",
            "model": f"LR_{tag}",
            "held_out_app": test_app,
            "seq_len": int(seq_len),
            "seq_feature_dim": int(seq_feature_dim),
            "appaware_K": int(appaware_K),
            "use_ros": bool(use_ros),
            "use_per_app_norm": bool(use_per_app_norm),
            "use_app_aware": bool(use_app_aware),
            "n_train": int(len(y_tr)),
            "n_val": int(len(y_val)),
            "n_test": int(len(y_test)),
            **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
            "cm": m["cm"],
            "pred_dist": m["pred_dist"],
        }

        print(f"[Temporal | LR_{tag}] HELD-OUT={test_app:12s} "
              f"(SEQ_LEN={seq_len}, seqF={seq_feature_dim}) "
              f"acc={row['acc']:.3f} f1={row['f1']:.3f} "
              f"prec={row['precision']:.3f} rec={row['recall']:.3f} "
              f"n_test={row['n_test']}")

        results.append(row)

    return pd.DataFrame(results)

# -------------------------
# Run the three requested experiments (+ baseline for comparison)
# -------------------------
SEQ_LEN = 5
AGG_USE = ("mean","std","delta","range","mean_abs_vel","vel_std")
MIN_NON_NAN_RATIO = 0.4

print("\n================= TEMPORAL LOAO: BASELINE LR =================")
res_base = run_loao_temporal_lr(
    label_col="Temporal",
    seq_len=SEQ_LEN,
    agg_use=AGG_USE,
    min_non_nan_ratio=MIN_NON_NAN_RATIO,
    use_ros=False,
    use_per_app_norm=False,
    use_app_aware=False,
    tag="baseline"
)
base_path = "/content/loao_temporal_lr_baseline.csv"
res_base.to_csv(base_path, index=False)
print("Saved:", base_path)

print("\n================= TEMPORAL LOAO: + RandomOverSampler =================")
if IMBLEARN_OK:
    res_ros = run_loao_temporal_lr(
        label_col="Temporal",
        seq_len=SEQ_LEN,
        agg_use=AGG_USE,
        min_non_nan_ratio=MIN_NON_NAN_RATIO,
        use_ros=True,
        use_per_app_norm=False,
        use_app_aware=False,
        tag="ROS"
    )
    ros_path = "/content/loao_temporal_lr_ros.csv"
    res_ros.to_csv(ros_path, index=False)
    print("Saved:", ros_path)
else:
    res_ros = None

print("\n================= TEMPORAL LOAO: + Per-app normalization =================")
res_norm = run_loao_temporal_lr(
    label_col="Temporal",
    seq_len=SEQ_LEN,
    agg_use=AGG_USE,
    min_non_nan_ratio=MIN_NON_NAN_RATIO,
    use_ros=False,
    use_per_app_norm=True,
    use_app_aware=False,
    tag="PerAppNorm"
)
norm_path = "/content/loao_temporal_lr_perapp_norm.csv"
res_norm.to_csv(norm_path, index=False)
print("Saved:", norm_path)

print("\n================= TEMPORAL LOAO: + App-aware one-hot =================")
res_appaware = run_loao_temporal_lr(
    label_col="Temporal",
    seq_len=SEQ_LEN,
    agg_use=AGG_USE,
    min_non_nan_ratio=MIN_NON_NAN_RATIO,
    use_ros=False,
    use_per_app_norm=False,
    use_app_aware=True,
    tag="AppAware"
)
aa_path = "/content/loao_temporal_lr_appaware.csv"
res_appaware.to_csv(aa_path, index=False)
print("Saved:", aa_path)

# -------------------------
# Summary table
# -------------------------
all_res = [res_base, res_norm, res_appaware]
if res_ros is not None:
    all_res.append(res_ros)

all_res_df = pd.concat(all_res, ignore_index=True)
summary = all_res_df.groupby(["model"])[["acc","f1","precision","recall"]].mean().sort_values("f1", ascending=False)
print("\n================ SUMMARY (mean over held-out apps) ================")
print(summary)

[INFO] imblearn RandomOverSampler imported OK.

================= TEMPORAL LOAO: BASELINE LR =================
[Temporal | LR_baseline] HELD-OUT=Archery      (SEQ_LEN=5, seqF=138) acc=0.875 f1=0.933 prec=0.875 rec=1.000 n_test=96
[Temporal | LR_baseline] HELD-OUT=PhantomLimb  (SEQ_LEN=5, seqF=138) acc=0.941 f1=0.927 prec=0.952 rec=0.902 n_test=321
[Temporal | LR_baseline] HELD-OUT=PianoTiles   (SEQ_LEN=5, seqF=138) acc=0.874 f1=0.444 prec=0.489 rec=0.407 n_test=436
[Temporal | LR_baseline] HELD-OUT=Puzzle       (SEQ_LEN=5, seqF=138) acc=0.549 f1=0.670 prec=0.773 rec=0.592 n_test=275
[Temporal | LR_baseline] HELD-OUT=Sea          (SEQ_LEN=5, seqF=138) acc=0.456 f1=0.625 prec=0.454 rec=1.000 n_test=296
[Temporal | LR_baseline] HELD-OUT=War          (SEQ_LEN=5, seqF=138) acc=0.533 f1=0.641 prec=0.473 rec=0.991 n_test=276
Saved: /content/loao_temporal_lr_baseline.csv

================= TEMPORAL LOAO: + RandomOverSampler =================
[Temporal | LR_ROS] HELD-OUT=Archery      (SEQ_LEN=5

/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1101: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1106: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1126: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1101: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1106: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1126: RuntimeWarning: invalid value encountered in divide
  new_unno

[Temporal | LR_PerAppNorm] HELD-OUT=Archery      (SEQ_LEN=5, seqF=138) acc=0.875 f1=0.933 prec=0.875 rec=1.000 n_test=96


/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1101: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1106: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1126: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1101: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1106: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1126: RuntimeWarning: invalid value encountered in divide
  new_unno

[Temporal | LR_PerAppNorm] HELD-OUT=PhantomLimb  (SEQ_LEN=5, seqF=138) acc=0.738 f1=0.750 prec=0.621 rec=0.947 n_test=321


...

[Temporal | LR_PerAppNorm] HELD-OUT=Puzzle       (SEQ_LEN=5, seqF=138) acc=0.738 f1=0.849 prec=0.768 rec=0.948 n_test=275


/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1101: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1106: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1126: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1101: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1106: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1126: RuntimeWarning: invalid value encountered in divide
  new_unno

[Temporal | LR_PerAppNorm] HELD-OUT=Sea          (SEQ_LEN=5, seqF=138) acc=0.453 f1=0.623 prec=0.453 rec=1.000 n_test=296


/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1101: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1106: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1126: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1101: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1106: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1126: RuntimeWarning: invalid value encountered in divide
  new_unno

[Temporal | LR_PerAppNorm] HELD-OUT=War          (SEQ_LEN=5, seqF=138) acc=0.743 f1=0.742 prec=0.642 rec=0.879 n_test=276
Saved: /content/loao_temporal_lr_perapp_norm.csv

================= TEMPORAL LOAO: + App-aware one-hot =================
[Temporal | LR_AppAware] HELD-OUT=Archery      (SEQ_LEN=5, seqF=138) acc=0.875 f1=0.933 prec=0.875 rec=1.000 n_test=96
[Temporal | LR_AppAware] HELD-OUT=PhantomLimb  (SEQ_LEN=5, seqF=138) acc=0.769 f1=0.773 prec=0.653 rec=0.947 n_test=321
[Temporal | LR_AppAware] HELD-OUT=PianoTiles   (SEQ_LEN=5, seqF=138) acc=0.704 f1=0.416 prec=0.275 rec=0.852 n_test=436
[Temporal | LR_AppAware] HELD-OUT=Puzzle       (SEQ_LEN=5, seqF=138) acc=0.444 f1=0.514 prec=0.794 rec=0.380 n_test=275
[Temporal | LR_AppAware] HELD-OUT=Sea          (SEQ_LEN=5, seqF=138) acc=0.449 f1=0.615 prec=0.450 rec=0.970 n_test=296
[Temporal | LR_AppAware] HELD-OUT=War          (SEQ_LEN=5, seqF=138) acc=0.692 f1=0.728 prec=0.579 rec=0.983 n_test=276
Saved: /content/loao_temporal_lr_appaw

# Transformer Tempoal

In [2]:
# ===============================================================
# LOAO Transformer (tabular-only) for YOUR VR metrics
# - Spatial: frame-level (T=1)  -> input (N, 1, F)
# - Temporal: sequence-level (T=SEQ_LEN) with your sliding-window + NaN-safe agg (optional)
#   Option A (recommended): raw sequence tokens (N, SEQ_LEN, F)  [Transformer sees time]
#   Label: center frame (i + SEQ_LEN//2), same as your setup
#
# This code:
# 1) Loads extracted_metrics_all_apps.csv (same as before)
# 2) Builds LOAO splits by App
# 3) Creates sequences for Temporal (RAW sequence, no agg)
# 4) Trains a small Transformer with focal loss + early stopping
# 5) Reports per-app acc/prec/rec/f1 + mean summary
# ===============================================================

import os
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.utils import shuffle

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# -------------------------
# 1) LOAD DATA
# -------------------------
CANDIDATE_PATHS = [
    "/content/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/VR_Metrics/extracted_metrics_all_apps.csv",
]
DATA_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        DATA_PATH = p
        break
if DATA_PATH is None:
    raise FileNotFoundError("Could not find extracted_metrics_all_apps.csv")

df = pd.read_csv(DATA_PATH)
df = df.replace("", np.nan)

required = {"App", "Spatial", "Temporal"}
missing_req = required - set(df.columns)
if missing_req:
    raise ValueError(f"Missing required columns: {missing_req}")

for col in ["Spatial", "Temporal"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["App"]).copy()
df = df.dropna(subset=["Spatial", "Temporal"], how="all").copy()
df["Spatial"] = df["Spatial"].astype(int)
df["Temporal"] = df["Temporal"].astype(int)

apps = sorted(df["App"].unique().tolist())
print("\n================= LOADED DATA =================")
print("Loaded:", DATA_PATH, "| shape:", df.shape)
print("Apps:", apps)

# Optional: downsample PhantomLimb
DOWNSAMPLE_PHANTOMLIMB = True
TARGET_N_PL = 400
if DOWNSAMPLE_PHANTOMLIMB and ("PhantomLimb" in df["App"].unique()):
    pl = df[df["App"] == "PhantomLimb"]
    others = df[df["App"] != "PhantomLimb"]
    if len(pl) > TARGET_N_PL:
        pl = pl.sample(TARGET_N_PL, random_state=SEED)
    df = pd.concat([others, pl], ignore_index=True)
    apps = sorted(df["App"].unique().tolist())
    print("\n[INFO] Downsample PhantomLimb =", DOWNSAMPLE_PHANTOMLIMB, "| TARGET_N_PL =", TARGET_N_PL)
    print("[INFO] PhantomLimb N =", len(df[df["App"] == "PhantomLimb"]), "| Total N =", len(df))

# -------------------------
# 2) FEATURES
# -------------------------
ID_COLS = [c for c in ["GlobalID", "EntryID"] if c in df.columns]
DROP_COLS = ["App", "Spatial", "Temporal"] + ID_COLS
feature_cols = [c for c in df.columns if c not in DROP_COLS]

for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

F = len(feature_cols)
print("\n================= FEATURES =================")
print("Num features:", F)
print("First 25 feature cols:", feature_cols[:25])

# -------------------------
# 3) METRICS
# -------------------------
def compute_metrics(y_true, y_pred):
    return (
        float(accuracy_score(y_true, y_pred)),
        float(precision_score(y_true, y_pred, zero_division=0)),
        float(recall_score(y_true, y_pred, zero_division=0)),
        float(f1_score(y_true, y_pred, zero_division=0)),
    )

def print_eval(prefix, y_true, y_pred):
    acc, prec, rec, f1 = compute_metrics(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred).tolist()
    return acc, prec, rec, f1, cm

# -------------------------
# 4) FOCAL LOSS (same spirit as yours)
# -------------------------
import tensorflow.keras.backend as K
def focal_loss(alpha=0.25, gamma=4.0):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        bce = K.binary_crossentropy(y_true, y_pred)
        p_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        loss = alpha * K.pow((1 - p_t), gamma) * bce
        return loss
    return loss

# -------------------------
# 5) TRANSFORMER ENCODER + MODEL
# Small + stable (don’t overdo capacity on 1.7k rows)
# -------------------------
from tensorflow.keras.layers import Input, Dense, Dropout, LayerNormalization, MultiHeadAttention, GlobalAveragePooling1D
from tensorflow.keras.models import Model

def transformer_encoder(inputs, num_heads=2, key_dim=32, ff_dim=128, dropout_rate=0.1):
    attn = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(inputs, inputs)
    attn = Dropout(dropout_rate)(attn)
    x = LayerNormalization(epsilon=1e-6)(inputs + attn)

    ffn = Dense(ff_dim, activation="relu")(x)
    ffn = Dense(inputs.shape[-1])(ffn)
    ffn = Dropout(dropout_rate)(ffn)
    out = LayerNormalization(epsilon=1e-6)(x + ffn)
    return out

def create_tabular_transformer(tabular_input_shape, embed_dim=64,
                              num_heads=2, key_dim=32, ff_dim=128,
                              depth=2, dropout_rate=0.1,
                              lr=1e-4):
    inp = Input(shape=tabular_input_shape, name="tabular_input")  # (T, F)

    # Token embedding per timestep
    x = Dense(embed_dim, activation="relu")(inp)

    for _ in range(depth):
        x = transformer_encoder(x, num_heads=num_heads, key_dim=key_dim, ff_dim=ff_dim, dropout_rate=dropout_rate)

    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(dropout_rate)(x)
    out = Dense(1, activation="sigmoid")(x)

    model = Model(inp, out)
    opt = tf.keras.optimizers.Adam(learning_rate=lr)
    model.compile(optimizer=opt, loss=focal_loss(), metrics=["accuracy"])
    return model

# -------------------------
# 6) DATA BUILDERS
# -------------------------
def impute_per_split(X_train, X_test):
    """Median impute using ONLY training split statistics."""
    med = np.nanmedian(X_train, axis=0)
    med = np.where(np.isfinite(med), med, 0.0)

    X_train_f = np.where(np.isfinite(X_train), X_train, med)
    X_test_f  = np.where(np.isfinite(X_test),  med, X_test)  # careful broadcast shape
    # X_test may be 3D; handle properly
    if X_test.ndim == 3:
        X_test_f = np.where(np.isfinite(X_test), X_test, med[None, None, :])
    else:
        X_test_f = np.where(np.isfinite(X_test), X_test, med)

    return X_train_f, X_test_f, med

def standardize_per_split(X_train, X_test, eps=1e-8):
    """Standardize using ONLY training split statistics."""
    mu = X_train.mean(axis=0)
    sd = X_train.std(axis=0)
    sd = np.where(sd < eps, 1.0, sd)

    if X_train.ndim == 3:
        X_train_z = (X_train - mu[None, None, :]) / sd[None, None, :]
        X_test_z  = (X_test  - mu[None, None, :]) / sd[None, None, :]
    else:
        X_train_z = (X_train - mu) / sd
        X_test_z  = (X_test  - mu) / sd

    return X_train_z, X_test_z

def make_raw_sequences(app_df, label_col, seq_len=5):
    """
    RAW sequence tokens:
      X_seq: (N_seq, seq_len, F)
      y_seq: (N_seq,) label from center frame
    """
    X = app_df[feature_cols].values.astype(float)
    y = app_df[label_col].values.astype(int)

    if len(app_df) < seq_len:
        return np.empty((0, seq_len, F), dtype=float), np.empty((0,), dtype=int)

    center = seq_len // 2
    Xs, ys = [], []
    for i in range(len(app_df) - seq_len + 1):
        window = X[i:i+seq_len]
        Xs.append(window)
        ys.append(y[i + center])

    return np.stack(Xs, axis=0), np.array(ys, dtype=int)

# -------------------------
# 7) LOAO RUNNERS
# -------------------------
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

def train_and_eval_transformer_loao_frame(label_col="Spatial",
                                         epochs=25, batch_size=32):
    """
    Frame-level: treat each sample as a 1-token sequence (T=1).
    Input shape: (1, F)
    """
    rows = []
    for test_app in apps:
        train_df = df[df["App"] != test_app].copy()
        test_df  = df[df["App"] == test_app].copy()

        X_train_full = train_df[feature_cols].values.astype(float)
        y_train_full = train_df[label_col].values.astype(int)
        X_test = test_df[feature_cols].values.astype(float)
        y_test = test_df[label_col].values.astype(int)

        # inner val split
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_full, y_train_full,
            test_size=0.15, random_state=SEED, stratify=y_train_full
        )

        # impute + standardize using training only
        X_tr, X_val, _ = impute_per_split(X_tr, X_val)
        X_tr, X_val = standardize_per_split(X_tr, X_val)

        # apply same stats to test using training med/mu/sd:
        # easiest: re-run impute & standardize with train stats by concatenation trick
        X_tr2, X_test2, _ = impute_per_split(X_tr, X_test)  # X_tr already imputed but ok
        X_tr2, X_test2 = standardize_per_split(X_tr2, X_test2)

        # reshape to (N, 1, F)
        X_tr3  = X_tr2[:, None, :]
        X_val3 = X_val[:, None, :]
        X_te3  = X_test2[:, None, :]

        model = create_tabular_transformer(tabular_input_shape=(1, F),
                                           embed_dim=64, num_heads=2, key_dim=32,
                                           ff_dim=128, depth=2, dropout_rate=0.1, lr=1e-4)

        callbacks = [
            EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True, verbose=0),
            ReduceLROnPlateau(monitor="val_accuracy", factor=0.2, patience=2, min_lr=1e-7, verbose=0),
        ]

        model.fit(X_tr3, y_tr, validation_data=(X_val3, y_val),
                  epochs=epochs, batch_size=batch_size, verbose=0, callbacks=callbacks)

        y_prob = model.predict(X_te3, verbose=0).reshape(-1)
        y_pred = (y_prob >= 0.5).astype(int)

        acc, prec, rec, f1, cm = print_eval("", y_test, y_pred)
        print(f"[{label_col} | T-Frame] HELD-OUT={test_app:12s} acc={acc:.3f} f1={f1:.3f} prec={prec:.3f} rec={rec:.3f} n_test={len(y_test)}")

        rows.append({
            "label": label_col, "task":"frame", "model":"Transformer_frame",
            "held_out_app": test_app, "n_test": int(len(y_test)),
            "acc": acc, "precision": prec, "recall": rec, "f1": f1, "cm": cm
        })

        tf.keras.backend.clear_session()

    return pd.DataFrame(rows)

def train_and_eval_transformer_loao_temporal_raw(label_col="Temporal",
                                                 seq_len=5,
                                                 epochs=25, batch_size=32):
    """
    Temporal: RAW sequences (N, seq_len, F) -> Transformer
    Label from center frame like your pipeline.
    """
    rows = []
    for test_app in apps:
        # build train sequences from all other apps
        X_train_list, y_train_list = [], []
        for a in apps:
            if a == test_app:
                continue
            Xa, ya = make_raw_sequences(df[df["App"] == a], label_col=label_col, seq_len=seq_len)
            if len(ya) > 0:
                X_train_list.append(Xa)
                y_train_list.append(ya)

        if len(X_train_list) == 0:
            print(f"[{label_col} | T-SeqRaw] HELD-OUT={test_app} -> no train seqs")
            continue

        X_train_full = np.concatenate(X_train_list, axis=0).astype(float)  # (N, T, F)
        y_train_full = np.concatenate(y_train_list, axis=0).astype(int)

        # test sequences
        X_test, y_test = make_raw_sequences(df[df["App"] == test_app], label_col=label_col, seq_len=seq_len)
        if len(y_test) == 0:
            print(f"[{label_col} | T-SeqRaw] HELD-OUT={test_app} -> no test seqs")
            continue

        # inner val split (stratify on y)
        idx = np.arange(len(y_train_full))
        tr_idx, val_idx = train_test_split(idx, test_size=0.15, random_state=SEED, stratify=y_train_full)

        X_tr = X_train_full[tr_idx]
        y_tr = y_train_full[tr_idx]
        X_val = X_train_full[val_idx]
        y_val = y_train_full[val_idx]

        # impute + standardize using ONLY X_tr stats, feature-wise (across all timesteps)
        X_tr_flat = X_tr.reshape(-1, F)  # (N*T, F)
        X_val_flat = X_val.reshape(-1, F)
        X_te_flat = X_test.reshape(-1, F)

        X_tr_flat, X_val_flat, med = impute_per_split(X_tr_flat, X_val_flat)
        # impute test using same med
        X_te_flat = np.where(np.isfinite(X_te_flat), X_te_flat, med)

        # standardize using X_tr_flat stats
        mu = X_tr_flat.mean(axis=0)
        sd = X_tr_flat.std(axis=0)
        sd = np.where(sd < 1e-8, 1.0, sd)

        X_tr_flat = (X_tr_flat - mu) / sd
        X_val_flat = (X_val_flat - mu) / sd
        X_te_flat = (X_te_flat - mu) / sd

        # reshape back
        X_tr2 = X_tr_flat.reshape(-1, seq_len, F)
        X_val2 = X_val_flat.reshape(-1, seq_len, F)
        X_te2 = X_te_flat.reshape(-1, seq_len, F)

        model = create_tabular_transformer(tabular_input_shape=(seq_len, F),
                                           embed_dim=64, num_heads=2, key_dim=32,
                                           ff_dim=128, depth=2, dropout_rate=0.1, lr=1e-4)

        callbacks = [
            EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True, verbose=0),
            ReduceLROnPlateau(monitor="val_accuracy", factor=0.2, patience=2, min_lr=1e-7, verbose=0),
        ]

        model.fit(X_tr2, y_tr, validation_data=(X_val2, y_val),
                  epochs=epochs, batch_size=batch_size, verbose=0, callbacks=callbacks)

        y_prob = model.predict(X_te2, verbose=0).reshape(-1)
        y_pred = (y_prob >= 0.5).astype(int)

        acc, prec, rec, f1, cm = print_eval("", y_test, y_pred)
        print(f"[{label_col} | T-SeqRaw] HELD-OUT={test_app:12s} (SEQ_LEN={seq_len}) "
              f"acc={acc:.3f} f1={f1:.3f} prec={prec:.3f} rec={rec:.3f} n_test={len(y_test)}")

        rows.append({
            "label": label_col, "task": f"sequence_raw_len_{seq_len}", "model":"Transformer_seqraw",
            "held_out_app": test_app, "seq_len": int(seq_len), "n_test": int(len(y_test)),
            "acc": acc, "precision": prec, "recall": rec, "f1": f1, "cm": cm
        })

        tf.keras.backend.clear_session()

    return pd.DataFrame(rows)

# -------------------------
# 8) RUN BOTH (Spatial + Temporal)
# -------------------------
SEQ_LEN = 5
EPOCHS = 25
BATCH = 32

print("\n================= LOAO: SPATIAL (FRAME) - TRANSFORMER =================")
res_sp = train_and_eval_transformer_loao_frame(label_col="Spatial", epochs=EPOCHS, batch_size=BATCH)
out_sp = "/content/loao_spatial_transformer_frame.csv"
res_sp.to_csv(out_sp, index=False)
print("Saved:", out_sp)

print("\n================= LOAO: TEMPORAL (RAW SEQ) - TRANSFORMER =================")
res_tmp = train_and_eval_transformer_loao_temporal_raw(label_col="Temporal", seq_len=SEQ_LEN, epochs=EPOCHS, batch_size=BATCH)
out_tmp = f"/content/loao_temporal_transformer_seqraw_len{SEQ_LEN}.csv"
res_tmp.to_csv(out_tmp, index=False)
print("Saved:", out_tmp)

print("\n================ SUMMARY =================")
all_df = pd.concat([res_sp, res_tmp], ignore_index=True)
summary = all_df.groupby(["label","task","model"])[["acc","precision","recall","f1"]].mean().reset_index()
print(summary)

out_all = "/content/loao_transformer_spatial_temporal.csv"
all_df.to_csv(out_all, index=False)
print("Saved:", out_all)


================= LOADED DATA =================
Loaded: /content/extracted_metrics_all_apps.csv | shape: (1744, 28)
Apps: ['Archery', 'PhantomLimb', 'PianoTiles', 'Puzzle', 'Sea', 'War']

[INFO] Downsample PhantomLimb = True | TARGET_N_PL = 400
[INFO] PhantomLimb N = 325 | Total N = 1744

================= FEATURES =================
Num features: 23
First 25 feature cols: ['missing_joints_count', 'missing_joints_ratio', 'collapsed_joints_count', 'center_of_mass_x', 'center_of_mass_y', 'center_of_mass_z', 'distance_from_origin', 'bbox_width', 'bbox_height', 'bbox_depth', 'bbox_volume', 'max_joint_distance_from_com', 'distance_from_floor', 'below_floor', 'left_forearm_length', 'right_forearm_length', 'left_shin_length', 'right_shin_length', 'arm_length_symmetry', 'leg_length_symmetry', 'body_forward_x', 'body_forward_y', 'body_forward_z']

================= LOAO: SPATIAL (FRAME) - TRANSFORMER =================
[Spatial | T-Frame] HELD-OUT=Archery      acc=0.140 f1=0.000 prec=0.000 rec=0

In [4]:
# ===============================================================
# VR APP-INDEPENDENT METRICS - LOAO (Spatial + Temporal Transformer)
# FULL SCRIPT (single block)
#
# - Loads extracted_metrics_all_apps.csv
# - Optional: downsample PhantomLimb
# - Spatial: "Transformer" (implemented as token-per-feature; otherwise T=1 is meaningless)
# - Temporal: raw sequences (SEQ_LEN) -> Transformer with per-sequence normalization
#
# Notes:
# - Temporal sequence dataset uses RAW windows (no aggregation) and center-frame label (i+SEQ_LEN//2)
# - No SMOTE/ROS here (you can add later)
# ===============================================================

import os
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# =========================
# 1) LOAD DATA
# =========================
CANDIDATE_PATHS = [
    "/content/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/VR_Metrics/extracted_metrics_all_apps.csv",
]

DATA_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        DATA_PATH = p
        break

if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not find extracted_metrics_all_apps.csv in common locations.\n"
        "Put it in /content OR adjust CANDIDATE_PATHS."
    )

df = pd.read_csv(DATA_PATH)
print("\n================= LOADED DATA =================")
print("Loaded:", DATA_PATH, "| shape:", df.shape)

# =========================
# 2) CLEAN / NORMALIZE TYPES
# =========================
df = df.replace("", np.nan)

required = {"App", "Spatial", "Temporal"}
missing_req = required - set(df.columns)
if missing_req:
    raise ValueError(f"Missing required columns in CSV: {missing_req}")

for col in ["Spatial", "Temporal"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=["App"]).copy()
df = df.dropna(subset=["Spatial", "Temporal"], how="all").copy()

df["Spatial"] = df["Spatial"].astype(int)
df["Temporal"] = df["Temporal"].astype(int)

apps = sorted(df["App"].unique().tolist())
print("Apps:", apps)

# =========================
# OPTIONAL: Downsample PhantomLimb
# =========================
DOWNSAMPLE_PHANTOMLIMB = True
TARGET_N_PL = 400

if DOWNSAMPLE_PHANTOMLIMB and ("PhantomLimb" in df["App"].unique()):
    pl = df[df["App"] == "PhantomLimb"]
    others = df[df["App"] != "PhantomLimb"]
    if len(pl) > TARGET_N_PL:
        pl = pl.sample(TARGET_N_PL, random_state=SEED)
    df = pd.concat([others, pl], ignore_index=True)
    apps = sorted(df["App"].unique().tolist())
    print("\n[INFO] Downsample PhantomLimb =", DOWNSAMPLE_PHANTOMLIMB,
          "| TARGET_N_PL =", TARGET_N_PL)
    print("[INFO] PhantomLimb N =", len(df[df["App"] == "PhantomLimb"]),
          "| Total N =", len(df))

# =========================
# 3) FEATURE COLUMN SELECTION
# =========================
ID_COLS = [c for c in ["GlobalID", "EntryID"] if c in df.columns]
DROP_COLS = ["App", "Spatial", "Temporal"] + ID_COLS
feature_cols = [c for c in df.columns if c not in DROP_COLS]

for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("\n================= FEATURES =================")
print("Num features:", len(feature_cols))
print("First 25 feature cols:", feature_cols[:25])

# =========================
# 4) METRICS
# =========================
def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred).tolist(),
        "pred_dist": dict(zip(*np.unique(y_pred, return_counts=True))),
    }

# ===============================================================
# 5) TRANSFORMER BUILDERS
# ===============================================================

from tensorflow.keras.layers import (
    Input, Dense, Dropout, LayerNormalization, MultiHeadAttention,
    GlobalAveragePooling1D, Lambda, Reshape
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

def transformer_encoder(inputs, num_heads=2, key_dim=32, ff_dim=128, dropout_rate=0.1):
    attn = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(inputs, inputs)
    attn = Dropout(dropout_rate)(attn)
    x = LayerNormalization(epsilon=1e-6)(inputs + attn)

    ffn = Dense(ff_dim, activation="relu")(x)
    ffn = Dense(inputs.shape[-1])(ffn)
    ffn = Dropout(dropout_rate)(ffn)
    return LayerNormalization(epsilon=1e-6)(x + ffn)

# ----- Spatial transformer (token-per-feature) -----
# This avoids the "T=1 token" issue. We treat each feature as a token.
def create_spatial_transformer_featuretokens(F,
                                             embed_dim=32, depth=2,
                                             num_heads=2, key_dim=16, ff_dim=64,
                                             dropout=0.1, lr=1e-3):
    inp = Input(shape=(F,), name="x_frame")  # (F,)

    # Replace NaNs safely inside TF
    x = Lambda(lambda t: tf.where(tf.math.is_finite(t), t, tf.zeros_like(t)))(inp)

    # Make tokens: (F, 1)
    x = Reshape((F, 1))(x)

    # Token embedding: (F, embed_dim)
    x = Dense(embed_dim, activation="relu")(x)

    # Transformer over feature tokens
    for _ in range(depth):
        x = transformer_encoder(x, num_heads=num_heads, key_dim=key_dim, ff_dim=ff_dim, dropout_rate=dropout)

    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(dropout)(x)
    out = Dense(1, activation="sigmoid")(x)

    model = Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

# ----- Temporal transformer (raw seq) with per-sequence normalization -----
def create_temporal_transformer_seqnorm(seq_len, F,
                                       embed_dim=64, depth=2,
                                       num_heads=2, key_dim=32, ff_dim=128,
                                       dropout=0.1, lr=1e-4):
    inp = Input(shape=(seq_len, F), name="x_seq")  # (T,F)

    # Replace NaNs with 0 first
    x = Lambda(lambda t: tf.where(tf.math.is_finite(t), t, tf.zeros_like(t)))(inp)

    # Per-sequence normalization across time (no test leakage: uses only the sequence)
    def seq_norm(z):
        mu = tf.reduce_mean(z, axis=1, keepdims=True)
        sd = tf.math.reduce_std(z, axis=1, keepdims=True)
        sd = tf.where(sd < 1e-6, tf.ones_like(sd), sd)
        return (z - mu) / sd

    x = Lambda(seq_norm, name="seq_norm")(x)

    # Token embedding over time tokens
    x = Dense(embed_dim, activation="relu")(x)

    for _ in range(depth):
        x = transformer_encoder(x, num_heads=num_heads, key_dim=key_dim, ff_dim=ff_dim, dropout_rate=dropout)

    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(dropout)(x)
    out = Dense(1, activation="sigmoid")(x)

    model = Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(lr),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

# ===============================================================
# 6) DATASET BUILDERS
# ===============================================================

def make_raw_sequence_dataset(app_df, label_col, seq_len=5):
    """
    Raw sliding windows: X_seq shape (Nseq, seq_len, F)
    Label = center frame (i + seq_len//2)
    """
    X = app_df[feature_cols].values.astype(np.float32)
    y = app_df[label_col].values.astype(int)

    if len(app_df) < seq_len:
        return np.empty((0, seq_len, len(feature_cols)), dtype=np.float32), np.empty((0,), dtype=int)

    X_seq, y_seq = [], []
    center = seq_len // 2

    for i in range(len(app_df) - seq_len + 1):
        window = X[i:i+seq_len]
        X_seq.append(window)
        y_seq.append(y[i + center])

    return np.stack(X_seq, axis=0), np.array(y_seq, dtype=int)

# ===============================================================
# 7) LOAO RUNNERS
# ===============================================================

def run_loao_spatial_transformer():
    results = []
    F = len(feature_cols)

    for test_app in apps:
        train_df = df[df["App"] != test_app].copy()
        test_df  = df[df["App"] == test_app].copy()

        X_train_full = train_df[feature_cols].values.astype(np.float32)
        y_train_full = train_df["Spatial"].values.astype(int)

        X_test = test_df[feature_cols].values.astype(np.float32)
        y_test = test_df["Spatial"].values.astype(int)

        # Split train/val
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_full, y_train_full,
            test_size=0.15,
            random_state=SEED,
            stratify=y_train_full
        )

        # Impute + scale using TRAIN only (outside TF for stability)
        imp = SimpleImputer(strategy="median")
        sca = StandardScaler()

        X_tr = sca.fit_transform(imp.fit_transform(X_tr)).astype(np.float32)
        X_val = sca.transform(imp.transform(X_val)).astype(np.float32)
        X_test2 = sca.transform(imp.transform(X_test)).astype(np.float32)

        # Class weights from TRAIN only
        cw = compute_class_weight(class_weight="balanced", classes=np.array([0, 1]), y=y_tr)
        class_weight = {0: float(cw[0]), 1: float(cw[1])}

        model = create_spatial_transformer_featuretokens(F=F)

        callbacks = [
            EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True, verbose=0),
            ReduceLROnPlateau(monitor="val_accuracy", factor=0.2, patience=2, min_lr=1e-7, verbose=0),
        ]

        model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=25,
            batch_size=32,
            verbose=0,
            callbacks=callbacks,
            class_weight=class_weight
        )

        y_prob = model.predict(X_test2, verbose=0).reshape(-1)
        y_pred = (y_prob >= 0.5).astype(int)

        m = eval_binary(y_test, y_pred)

        row = {
            "label": "Spatial",
            "task": "frame",
            "model": "Transformer_frame_featuretokens",
            "held_out_app": test_app,
            "n_train": int(len(y_tr)),
            "n_val": int(len(y_val)),
            "n_test": int(len(y_test)),
            **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
            "cm": m["cm"],
            "pred_dist": m["pred_dist"],
        }

        print(f"[Spatial | T-Frame] HELD-OUT={test_app:12s} "
              f"acc={row['acc']:.3f} f1={row['f1']:.3f} "
              f"prec={row['precision']:.3f} rec={row['recall']:.3f} "
              f"n_test={row['n_test']}")
        results.append(row)

    out = pd.DataFrame(results)
    out_path = "/content/loao_spatial_transformer_frame.csv"
    out.to_csv(out_path, index=False)
    print("Saved:", out_path)
    return out

def run_loao_temporal_transformer(seq_len=5):
    results = []
    F = len(feature_cols)

    for test_app in apps:
        # Build train sequences from all other apps (no leakage)
        X_train_list, y_train_list = [], []
        for a in apps:
            if a == test_app:
                continue
            Xa, ya = make_raw_sequence_dataset(df[df["App"] == a], label_col="Temporal", seq_len=seq_len)
            if len(ya) > 0:
                X_train_list.append(Xa)
                y_train_list.append(ya)

        if len(X_train_list) == 0:
            print(f"[Temporal | T-SeqRaw] HELD-OUT={test_app} -> No train sequences.")
            continue

        X_train_full = np.concatenate(X_train_list, axis=0)
        y_train_full = np.concatenate(y_train_list, axis=0)

        # Test sequences
        test_df = df[df["App"] == test_app].copy()
        X_test, y_test = make_raw_sequence_dataset(test_df, label_col="Temporal", seq_len=seq_len)

        if len(y_test) == 0:
            print(f"[Temporal | T-SeqRaw] HELD-OUT={test_app} -> No test sequences.")
            continue

        # Train/val split on sequences
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_full, y_train_full,
            test_size=0.15,
            random_state=SEED,
            stratify=y_train_full
        )

        # Impute + scale per-feature using TRAIN ONLY (flatten -> fit -> reshape)
        imp = SimpleImputer(strategy="median")
        sca = StandardScaler()

        def fit_transform_seq(X, fit=False):
            # X: (N,T,F)
            N, T, F_ = X.shape
            X2 = X.reshape(N*T, F_)
            if fit:
                X2 = imp.fit_transform(X2)
                X2 = sca.fit_transform(X2)
            else:
                X2 = imp.transform(X2)
                X2 = sca.transform(X2)
            return X2.reshape(N, T, F_).astype(np.float32)

        X_tr2 = fit_transform_seq(X_tr, fit=True)
        X_val2 = fit_transform_seq(X_val, fit=False)
        X_te2  = fit_transform_seq(X_test, fit=False)

        # Class weights from TRAIN ONLY
        cw = compute_class_weight(class_weight="balanced", classes=np.array([0, 1]), y=y_tr)
        class_weight = {0: float(cw[0]), 1: float(cw[1])}

        model = create_temporal_transformer_seqnorm(seq_len=seq_len, F=F)

        callbacks = [
            EarlyStopping(monitor="val_accuracy", patience=5, restore_best_weights=True, verbose=0),
            ReduceLROnPlateau(monitor="val_accuracy", factor=0.2, patience=2, min_lr=1e-7, verbose=0),
        ]

        model.fit(
            X_tr2, y_tr,
            validation_data=(X_val2, y_val),
            epochs=25,
            batch_size=32,
            verbose=0,
            callbacks=callbacks,
            class_weight=class_weight
        )

        y_prob = model.predict(X_te2, verbose=0).reshape(-1)
        y_pred = (y_prob >= 0.5).astype(int)

        m = eval_binary(y_test, y_pred)

        row = {
            "label": "Temporal",
            "task": f"sequence_raw_len_{seq_len}",
            "model": "Transformer_seqraw_seqnorm",
            "held_out_app": test_app,
            "seq_len": int(seq_len),
            "n_train": int(len(y_tr)),
            "n_val": int(len(y_val)),
            "n_test": int(len(y_test)),
            **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
            "cm": m["cm"],
            "pred_dist": m["pred_dist"],
        }

        print(f"[Temporal | T-SeqRaw] HELD-OUT={test_app:12s} "
              f"(SEQ_LEN={seq_len}) acc={row['acc']:.3f} f1={row['f1']:.3f} "
              f"prec={row['precision']:.3f} rec={row['recall']:.3f} "
              f"n_test={row['n_test']}")
        results.append(row)

    out = pd.DataFrame(results)
    out_path = f"/content/loao_temporal_transformer_seqraw_len{seq_len}.csv"
    out.to_csv(out_path, index=False)
    print("Saved:", out_path)
    return out

# ===============================================================
# 8) RUN
# ===============================================================
SEQ_LEN = 5

print("\n================= LOAO: SPATIAL (FRAME) - TRANSFORMER =================")
spatial_df = run_loao_spatial_transformer()

print("\n================= LOAO: TEMPORAL (RAW SEQ) - TRANSFORMER =================")
temporal_df = run_loao_temporal_transformer(seq_len=SEQ_LEN)

results_df = pd.concat([spatial_df, temporal_df], ignore_index=True)

print("\n================ SUMMARY =================")
summary = results_df.groupby(["label", "task", "model"])[["acc", "precision", "recall", "f1"]].mean().reset_index()
print(summary)

OUT_PATH = "/content/loao_transformer_spatial_temporal.csv"
results_df.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH)


================= LOADED DATA =================
Loaded: /content/extracted_metrics_all_apps.csv | shape: (1744, 28)
Apps: ['Archery', 'PhantomLimb', 'PianoTiles', 'Puzzle', 'Sea', 'War']

[INFO] Downsample PhantomLimb = True | TARGET_N_PL = 400
[INFO] PhantomLimb N = 325 | Total N = 1744

================= FEATURES =================
Num features: 23
First 25 feature cols: ['missing_joints_count', 'missing_joints_ratio', 'collapsed_joints_count', 'center_of_mass_x', 'center_of_mass_y', 'center_of_mass_z', 'distance_from_origin', 'bbox_width', 'bbox_height', 'bbox_depth', 'bbox_volume', 'max_joint_distance_from_com', 'distance_from_floor', 'below_floor', 'left_forearm_length', 'right_forearm_length', 'left_shin_length', 'right_shin_length', 'arm_length_symmetry', 'leg_length_symmetry', 'body_forward_x', 'body_forward_y', 'body_forward_z']

================= LOAO: SPATIAL (FRAME) - TRANSFORMER =================
[Spatial | T-Frame] HELD-OUT=Archery      acc=0.140 f1=0.000 prec=0.000 rec=0

[Spatial | T-Frame] HELD-OUT=PhantomLimb  acc=0.578 f1=0.055 prec=0.035 rec=0.125 n_test=325
[Spatial | T-Frame] HELD-OUT=PianoTiles   acc=0.518 f1=0.000 prec=0.000 rec=0.000 n_test=440
[Spatial | T-Frame] HELD-OUT=Puzzle       acc=0.261 f1=0.000 prec=0.000 rec=0.000 n_test=299
[Spatial | T-Frame] HELD-OUT=Sea          acc=0.560 f1=0.000 prec=0.000 rec=0.000 n_test=300
[Spatial | T-Frame] HELD-OUT=War          acc=0.421 f1=0.557 prec=0.389 rec=0.981 n_test=280
Saved: /content/loao_spatial_transformer_frame.csv

================= LOAO: TEMPORAL (RAW SEQ) - TRANSFORMER =================
[Temporal | T-SeqRaw] HELD-OUT=Archery      (SEQ_LEN=5) acc=0.271 f1=0.352 prec=0.792 rec=0.226 n_test=96
[Temporal | T-SeqRaw] HELD-OUT=PhantomLimb  (SEQ_LEN=5) acc=0.536 f1=0.259 prec=0.382 rec=0.195 n_test=321
[Temporal | T-SeqRaw] HELD-OUT=PianoTiles   (SEQ_LEN=5) acc=0.569 f1=0.324 prec=0.201 rec=0.833 n_test=436
[Temporal | T-SeqRaw] HELD-OUT=Puzzle       (SEQ_LEN=5) acc=0.739 f1=0.816 prec=0.830 re

In [5]:
# ===============================================================
# LOAO TEMPORAL (RAW SEQ) - TRANSFORMER with stability fixes
# - Uses raw sequences: (SEQ_LEN, F=23)
# - Per-fold scaler fit on TRAIN only (no leakage)
# - Class weights
# - Fixed predict batch_size to reduce TF retracing
# - clear_session() each fold
# ===============================================================

import os, random
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# -------------------------
# 1) LOAD DATA
# -------------------------
CANDIDATE_PATHS = [
    "/content/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/VR_Metrics/extracted_metrics_all_apps.csv",
]
DATA_PATH = next((p for p in CANDIDATE_PATHS if os.path.exists(p)), None)
if DATA_PATH is None:
    raise FileNotFoundError("Could not find extracted_metrics_all_apps.csv")

df = pd.read_csv(DATA_PATH).replace("", np.nan)

required = {"App", "Temporal"}
missing_req = required - set(df.columns)
if missing_req:
    raise ValueError(f"Missing required columns: {missing_req}")

df["Temporal"] = pd.to_numeric(df["Temporal"], errors="coerce")
df = df.dropna(subset=["App", "Temporal"]).copy()
df["Temporal"] = df["Temporal"].astype(int)

# optional downsample PhantomLimb
DOWNSAMPLE_PHANTOMLIMB = True
TARGET_N_PL = 400
if DOWNSAMPLE_PHANTOMLIMB and ("PhantomLimb" in df["App"].unique()):
    pl = df[df["App"] == "PhantomLimb"]
    others = df[df["App"] != "PhantomLimb"]
    if len(pl) > TARGET_N_PL:
        pl = pl.sample(TARGET_N_PL, random_state=SEED)
    df = pd.concat([others, pl], ignore_index=True)

apps = sorted(df["App"].unique().tolist())
print("\n================= LOADED DATA =================")
print("Loaded:", DATA_PATH, "| shape:", df.shape)
print("Apps:", apps)

# -------------------------
# 2) FEATURE COLS
# -------------------------
ID_COLS = [c for c in ["GlobalID", "EntryID"] if c in df.columns]
DROP_COLS = ["App", "Spatial", "Temporal"] + ID_COLS
feature_cols = [c for c in df.columns if c not in DROP_COLS]

for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("\n================= FEATURES =================")
print("Num features:", len(feature_cols))
print("First 25 feature cols:", feature_cols[:25])

# -------------------------
# 3) METRICS
# -------------------------
def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred).tolist(),
        "pred_dist": dict(zip(*np.unique(y_pred, return_counts=True))),
    }

# -------------------------
# 4) SEQUENCE DATA (RAW)
# -------------------------
def make_raw_sequence_dataset(app_df, label_col="Temporal", seq_len=5, min_non_nan_ratio=0.4):
    X = app_df[feature_cols].values.astype(float)
    y = app_df[label_col].values.astype(int)

    if len(app_df) < seq_len:
        return np.empty((0, seq_len, len(feature_cols))), np.empty((0,), dtype=int)

    X_seq, y_seq = [], []
    center = seq_len // 2

    for i in range(len(app_df) - seq_len + 1):
        window = X[i:i+seq_len]
        valid = np.isfinite(window).sum()
        total = window.size
        if (valid / total) < min_non_nan_ratio:
            continue
        X_seq.append(window)
        y_seq.append(y[i + center])

    if len(X_seq) == 0:
        return np.empty((0, seq_len, len(feature_cols))), np.empty((0,), dtype=int)

    return np.stack(X_seq, axis=0), np.array(y_seq, dtype=int)

# -------------------------
# 5) TRANSFORMER MODEL (SMALL + STABLE)
# -------------------------
from tensorflow.keras.layers import Input, Dense, Dropout, LayerNormalization, MultiHeadAttention, GlobalAveragePooling1D
from tensorflow.keras.models import Model

def transformer_encoder(x, num_heads=2, key_dim=16, ff_dim=64, dropout=0.15):
    attn = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(x, x)
    attn = Dropout(dropout)(attn)
    x = LayerNormalization(epsilon=1e-6)(x + attn)

    ffn = Dense(ff_dim, activation="relu")(x)
    ffn = Dense(x.shape[-1])(ffn)
    ffn = Dropout(dropout)(ffn)
    return LayerNormalization(epsilon=1e-6)(x + ffn)

def build_temporal_transformer(seq_len, feat_dim,
                              embed_dim=32,
                              depth=2,
                              num_heads=2,
                              key_dim=16,
                              ff_dim=64,
                              dropout=0.15,
                              lr=2e-4):
    inp = Input(shape=(seq_len, feat_dim), name="seq_input")
    x = Dense(embed_dim, activation="relu")(inp)

    for _ in range(depth):
        x = transformer_encoder(x, num_heads=num_heads, key_dim=key_dim, ff_dim=ff_dim, dropout=dropout)

    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(dropout)(x)
    out = Dense(1, activation="sigmoid")(x)

    model = Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

# -------------------------
# 6) LOAO RUN
# -------------------------
def run_loao_temporal_transformer(seq_len=5, min_non_nan_ratio=0.4,
                                 epochs=30, batch_size=64, pred_batch_size=256):
    results = []

    for test_app in apps:
        tf.keras.backend.clear_session()  # IMPORTANT for memory + retracing

        # Build sequences per app (no leakage)
        X_train_list, y_train_list = [], []
        for a in apps:
            if a == test_app:
                continue
            Xa, ya = make_raw_sequence_dataset(
                df[df["App"] == a],
                label_col="Temporal",
                seq_len=seq_len,
                min_non_nan_ratio=min_non_nan_ratio
            )
            if len(ya) > 0:
                X_train_list.append(Xa)
                y_train_list.append(ya)

        if len(X_train_list) == 0:
            print(f"[Temporal | T-SeqRaw] HELD-OUT={test_app} -> no train sequences")
            continue

        X_train_full = np.concatenate(X_train_list, axis=0)
        y_train_full = np.concatenate(y_train_list, axis=0)

        X_test, y_test = make_raw_sequence_dataset(
            df[df["App"] == test_app],
            label_col="Temporal",
            seq_len=seq_len,
            min_non_nan_ratio=min_non_nan_ratio
        )
        if len(y_test) == 0:
            print(f"[Temporal | T-SeqRaw] HELD-OUT={test_app} -> no test sequences")
            continue

        # ---------
        # Impute + scale (fit on TRAIN only)
        # ---------
        # Flatten to (N*T, F) for imputer/scaler, then reshape back
        Ntr, T, F = X_train_full.shape
        Nte = X_test.shape[0]

        imputer = SimpleImputer(strategy="median")
        scaler = StandardScaler()

        Xtr_flat = X_train_full.reshape(Ntr*T, F)
        Xte_flat = X_test.reshape(Nte*T, F)

        Xtr_flat = imputer.fit_transform(Xtr_flat)
        Xte_flat = imputer.transform(Xte_flat)

        Xtr_flat = scaler.fit_transform(Xtr_flat)
        Xte_flat = scaler.transform(Xte_flat)

        X_train_full = Xtr_flat.reshape(Ntr, T, F)
        X_test = Xte_flat.reshape(Nte, T, F)

        # inner split
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_full, y_train_full,
            test_size=0.15,
            random_state=SEED,
            stratify=y_train_full
        )

        # class weights
        cw = compute_class_weight(class_weight="balanced", classes=np.array([0,1]), y=y_tr)
        class_weight = {0: float(cw[0]), 1: float(cw[1])}

        model = build_temporal_transformer(seq_len=T, feat_dim=F)

        callbacks = [
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=5,
                restore_best_weights=True,
                verbose=0
            ),
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss",
                factor=0.5,
                patience=2,
                min_lr=1e-6,
                verbose=0
            )
        ]

        model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=batch_size,
            verbose=0,
            class_weight=class_weight,
        )

        probs = model.predict(X_test, batch_size=pred_batch_size, verbose=0).reshape(-1)
        yhat = (probs >= 0.5).astype(int)

        m = eval_binary(y_test, yhat)

        print(f"[Temporal | T-SeqRaw] HELD-OUT={test_app:12s} (SEQ_LEN={seq_len}) "
              f"acc={m['acc']:.3f} f1={m['f1']:.3f} prec={m['precision']:.3f} rec={m['recall']:.3f} "
              f"n_test={len(y_test)}")

        results.append({
            "label": "Temporal",
            "task": f"sequence_raw_len_{seq_len}",
            "model": "Transformer_seqraw_stable",
            "held_out_app": test_app,
            "seq_len": int(seq_len),
            "n_train": int(len(y_tr)),
            "n_val": int(len(y_val)),
            "n_test": int(len(y_test)),
            "acc": m["acc"],
            "f1": m["f1"],
            "precision": m["precision"],
            "recall": m["recall"],
            "cm": m["cm"],
            "pred_dist": m["pred_dist"],
        })

    return pd.DataFrame(results)

SEQ_LEN = 5
MIN_NON_NAN_RATIO = 0.4
res_df = run_loao_temporal_transformer(seq_len=SEQ_LEN, min_non_nan_ratio=MIN_NON_NAN_RATIO)

print("\n================ SUMMARY (mean over held-out apps) ================")
print(res_df[["acc","f1","precision","recall"]].mean())

OUT = "/content/loao_temporal_transformer_seqraw_len5_STABLE.csv"
res_df.to_csv(OUT, index=False)
print("\nSaved:", OUT)


================= LOADED DATA =================
Loaded: /content/extracted_metrics_all_apps.csv | shape: (1744, 28)
Apps: ['Archery', 'PhantomLimb', 'PianoTiles', 'Puzzle', 'Sea', 'War']

================= FEATURES =================
Num features: 23
First 25 feature cols: ['missing_joints_count', 'missing_joints_ratio', 'collapsed_joints_count', 'center_of_mass_x', 'center_of_mass_y', 'center_of_mass_z', 'distance_from_origin', 'bbox_width', 'bbox_height', 'bbox_depth', 'bbox_volume', 'max_joint_distance_from_com', 'distance_from_floor', 'below_floor', 'left_forearm_length', 'right_forearm_length', 'left_shin_length', 'right_shin_length', 'arm_length_symmetry', 'leg_length_symmetry', 'body_forward_x', 'body_forward_y', 'body_forward_z']
[Temporal | T-SeqRaw] HELD-OUT=Archery      (SEQ_LEN=5) acc=0.875 f1=0.933 prec=0.875 rec=1.000 n_test=96


[Temporal | T-SeqRaw] HELD-OUT=PhantomLimb  (SEQ_LEN=5) acc=0.586 f1=0.000 prec=0.000 rec=0.000 n_test=321
[Temporal | T-SeqRaw] HELD-OUT=PianoTiles   (SEQ_LEN=5) acc=0.851 f1=0.425 prec=0.407 rec=0.444 n_test=436
[Temporal | T-SeqRaw] HELD-OUT=Puzzle       (SEQ_LEN=5) acc=0.778 f1=0.875 prec=0.777 rec=1.000 n_test=275
[Temporal | T-SeqRaw] HELD-OUT=Sea          (SEQ_LEN=5) acc=0.571 f1=0.099 prec=1.000 rec=0.052 n_test=296
[Temporal | T-SeqRaw] HELD-OUT=War          (SEQ_LEN=5) acc=0.757 f1=0.660 prec=0.802 rec=0.560 n_test=276

================ SUMMARY (mean over held-out apps) ================
acc          0.736327
f1           0.498674
precision    0.643604
recall       0.509505
dtype: float64

Saved: /content/loao_temporal_transformer_seqraw_len5_STABLE.csv


In [6]:
# ===============================================================
# VR APP-INDEPENDENT METRICS - LOAO TEMPORAL (RAW SEQ) TRANSFORMER
# Improvements:
#   (1) RAW SEQ input: (SEQ_LEN, F) = (5, 23)
#   (2) NaN-safe preprocessing (median impute on train only)
#   (3) StandardScaler fitted on TRAIN only (flattened tokens)
#   (4) Class weights (per-fold) to reduce majority collapse
#   (5) Threshold tuning on VAL (maximize F1), then evaluate on TEST
#   (6) Reduce TF retracing: tf.function(reduce_retracing=True) predict
#
# Outputs:
#   - Per held-out app: acc/f1/prec/rec + best threshold
#   - Mean metrics across held-out apps
#   - CSV saved to /content/loao_temporal_transformer_seqraw_len5_IMPROVED.csv
# ===============================================================

import os
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.layers import (
    Input, Dense, Dropout, LayerNormalization,
    MultiHeadAttention, GlobalAveragePooling1D
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# -------------------------
# 1) LOAD DATA
# -------------------------
CANDIDATE_PATHS = [
    "/content/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/VR_Metrics/extracted_metrics_all_apps.csv",
]
DATA_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        DATA_PATH = p
        break
if DATA_PATH is None:
    raise FileNotFoundError("Could not find extracted_metrics_all_apps.csv in common locations.")

df = pd.read_csv(DATA_PATH)
print("\n================= LOADED DATA =================")
print("Loaded:", DATA_PATH, "| shape:", df.shape)

# -------------------------
# 2) CLEAN / TYPES
# -------------------------
df = df.replace("", np.nan)

required = {"App", "Temporal"}
missing_req = required - set(df.columns)
if missing_req:
    raise ValueError(f"Missing required columns: {missing_req}")

df["Temporal"] = pd.to_numeric(df["Temporal"], errors="coerce")
df = df.dropna(subset=["App", "Temporal"]).copy()
df["Temporal"] = df["Temporal"].astype(int)

apps = sorted(df["App"].unique().tolist())
print("Apps:", apps)

# OPTIONAL: Downsample PhantomLimb
DOWNSAMPLE_PHANTOMLIMB = True
TARGET_N_PL = 400
if DOWNSAMPLE_PHANTOMLIMB and ("PhantomLimb" in df["App"].unique()):
    pl = df[df["App"] == "PhantomLimb"]
    others = df[df["App"] != "PhantomLimb"]
    if len(pl) > TARGET_N_PL:
        pl = pl.sample(TARGET_N_PL, random_state=SEED)
    df = pd.concat([others, pl], ignore_index=True)
    apps = sorted(df["App"].unique().tolist())
    print("\n[INFO] Downsample PhantomLimb =", DOWNSAMPLE_PHANTOMLIMB, "| TARGET_N_PL =", TARGET_N_PL)
    print("[INFO] PhantomLimb N =", len(df[df["App"] == "PhantomLimb"]), "| Total N =", len(df))

# -------------------------
# 3) FEATURE COLS (23)
# -------------------------
ID_COLS = [c for c in ["GlobalID", "EntryID"] if c in df.columns]
DROP_COLS = ["App", "Spatial", "Temporal"] + ID_COLS
feature_cols = [c for c in df.columns if c not in DROP_COLS]

for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("\n================= FEATURES =================")
print("Num features:", len(feature_cols))
print("First 25 feature cols:", feature_cols[:25])

# -------------------------
# 4) HELPERS
# -------------------------
def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred).tolist(),
        "pred_dist": dict(zip(*np.unique(y_pred, return_counts=True))),
    }

def make_raw_sequence_dataset(app_df, label_col="Temporal", seq_len=5):
    """
    Raw seq: returns X of shape (N_seq, seq_len, F) using consecutive rows.
    Label is center frame: i + seq_len//2
    """
    X = app_df[feature_cols].values.astype(float)
    y = app_df[label_col].values.astype(int)

    if len(app_df) < seq_len:
        return np.empty((0, seq_len, len(feature_cols))), np.empty((0,), dtype=int)

    X_seq, y_seq = [], []
    center = seq_len // 2
    for i in range(len(app_df) - seq_len + 1):
        X_seq.append(X[i:i+seq_len])
        y_seq.append(y[i + center])

    return np.stack(X_seq, axis=0), np.array(y_seq, dtype=int)

def build_transformer_seq_model(seq_len, n_feat,
                                d_model=64, num_heads=2, key_dim=32,
                                ff_dim=128, depth=2, dropout=0.15,
                                lr=1e-4):
    """
    Input: (seq_len, n_feat)
    We'll project features -> d_model, then transformer blocks, then GAP + head
    """
    inp = Input(shape=(seq_len, n_feat), name="seq_in")
    x = Dense(d_model, activation="relu")(inp)

    for _ in range(depth):
        attn = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(x, x)
        attn = Dropout(dropout)(attn)
        x = LayerNormalization(epsilon=1e-6)(x + attn)

        ffn = Dense(ff_dim, activation="relu")(x)
        ffn = Dense(d_model)(ffn)
        ffn = Dropout(dropout)(ffn)
        x = LayerNormalization(epsilon=1e-6)(x + ffn)

    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(dropout)(x)
    out = Dense(1, activation="sigmoid", name="out")(x)

    model = Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

def tune_threshold_on_val(y_val, p_val, grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 19)  # coarse but stable
    best_thr, best_f1 = 0.5, -1.0
    for t in grid:
        yhat = (p_val >= t).astype(int)
        f1 = f1_score(y_val, yhat, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = float(t)
    return best_thr, float(best_f1)

# -------------------------
# 5) TF retracing reduction: fast predict
# -------------------------
tf.config.run_functions_eagerly(False)

@tf.function(reduce_retracing=True)
def fast_predict(model, x):
    return model(x, training=False)

# -------------------------
# 6) LOAO run: RAW SEQ Transformer + val threshold tuning
# -------------------------
def run_loao_temporal_transformer_seqraw(
    label_col="Temporal",
    seq_len=5,
    d_model=64,
    num_heads=2,
    key_dim=32,
    ff_dim=128,
    depth=2,
    dropout=0.15,
    lr=1e-4,
    epochs=20,
    batch_size=64,
    val_size=0.15
):
    results = []

    print("\n================= LOAO: TEMPORAL (RAW SEQ) - TRANSFORMER (IMPROVED) =================")
    print(f"[INFO] SEQ_LEN={seq_len} | raw tokens = (T={seq_len}, F={len(feature_cols)})")

    for test_app in apps:
        # ---- build train/test (no leakage)
        train_apps = [a for a in apps if a != test_app]

        X_train_list, y_train_list = [], []
        for a in train_apps:
            Xa, ya = make_raw_sequence_dataset(df[df["App"] == a], label_col=label_col, seq_len=seq_len)
            if len(ya) > 0:
                X_train_list.append(Xa)
                y_train_list.append(ya)

        if len(X_train_list) == 0:
            print(f"[{label_col} | T-SeqRaw] HELD-OUT={test_app} -> No train sequences.")
            continue

        X_train_full = np.concatenate(X_train_list, axis=0)
        y_train_full = np.concatenate(y_train_list, axis=0)

        X_test, y_test = make_raw_sequence_dataset(df[df["App"] == test_app], label_col=label_col, seq_len=seq_len)
        if len(y_test) == 0:
            print(f"[{label_col} | T-SeqRaw] HELD-OUT={test_app} -> No test sequences.")
            continue

        # ---- train/val split (stratified)
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_full, y_train_full,
            test_size=val_size,
            random_state=SEED,
            stratify=y_train_full
        )

        # ---- Impute + scale (fit ONLY on train)
        # We fit imputer/scaler over flattened tokens to treat each feature consistently.
        n_feat = X_tr.shape[-1]

        imp = SimpleImputer(strategy="median")
        sca = StandardScaler()

        X_tr_flat = X_tr.reshape(-1, n_feat)
        X_val_flat = X_val.reshape(-1, n_feat)
        X_te_flat  = X_test.reshape(-1, n_feat)

        X_tr_flat = imp.fit_transform(X_tr_flat)
        X_val_flat = imp.transform(X_val_flat)
        X_te_flat  = imp.transform(X_te_flat)

        X_tr_flat = sca.fit_transform(X_tr_flat)
        X_val_flat = sca.transform(X_val_flat)
        X_te_flat  = sca.transform(X_te_flat)

        X_tr = X_tr_flat.reshape(-1, seq_len, n_feat).astype(np.float32)
        X_val = X_val_flat.reshape(-1, seq_len, n_feat).astype(np.float32)
        X_te  = X_te_flat.reshape(-1, seq_len, n_feat).astype(np.float32)

        # ---- class weights (per-fold)
        classes = np.array([0, 1])
        cw = compute_class_weight(class_weight="balanced", classes=classes, y=y_tr)
        class_weight = {0: float(cw[0]), 1: float(cw[1])}

        # ---- model
        model = build_transformer_seq_model(
            seq_len=seq_len, n_feat=n_feat,
            d_model=d_model, num_heads=num_heads, key_dim=key_dim,
            ff_dim=ff_dim, depth=depth, dropout=dropout, lr=lr
        )

        early = EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True, verbose=0)
        rlrop = ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-6, verbose=0)

        model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=epochs,
            batch_size=batch_size,
            verbose=0,
            class_weight=class_weight,
            callbacks=[early, rlrop]
        )

        # ---- predict probs via tf.function (reduces retracing spam)
        p_val = fast_predict(model, tf.convert_to_tensor(X_val)).numpy().reshape(-1)
        p_te  = fast_predict(model, tf.convert_to_tensor(X_te)).numpy().reshape(-1)

        thr, val_f1 = tune_threshold_on_val(y_val, p_val)
        yhat = (p_te >= thr).astype(int)

        m = eval_binary(y_test, yhat)
        row = {
            "label": label_col,
            "task": f"sequence_raw_len_{seq_len}",
            "model": "Transformer_seqraw_improved",
            "held_out_app": test_app,
            "seq_len": int(seq_len),
            "F": int(n_feat),
            "n_train": int(len(y_tr)),
            "n_val": int(len(y_val)),
            "n_test": int(len(y_test)),
            "thr": float(thr),
            "val_f1_best": float(val_f1),
            **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
            "cm": m["cm"],
            "pred_dist": m["pred_dist"],
        }

        print(f"[Temporal | T-SeqRaw+thr] HELD-OUT={test_app:12s} (SEQ_LEN={seq_len}) "
              f"thr={thr:.2f} val_f1={val_f1:.3f}  "
              f"acc={row['acc']:.3f} f1={row['f1']:.3f} "
              f"prec={row['precision']:.3f} rec={row['recall']:.3f}  "
              f"n_test={row['n_test']}")
        results.append(row)

        # cleanup between folds
        tf.keras.backend.clear_session()

    return pd.DataFrame(results)

# -------------------------
# 7) RUN
# -------------------------
SEQ_LEN = 5

res = run_loao_temporal_transformer_seqraw(
    label_col="Temporal",
    seq_len=SEQ_LEN,
    d_model=64,
    num_heads=2,
    key_dim=32,
    ff_dim=128,
    depth=2,
    dropout=0.15,
    lr=1e-4,
    epochs=20,
    batch_size=64,
    val_size=0.15
)

print("\n================ SUMMARY (mean over held-out apps) ================")
if len(res) > 0:
    print(res[["acc","f1","precision","recall"]].mean())
else:
    print("No results (no sequences found).")

OUT = "/content/loao_temporal_transformer_seqraw_len5_IMPROVED.csv"
res.to_csv(OUT, index=False)
print("\nSaved:", OUT)


================= LOADED DATA =================
Loaded: /content/extracted_metrics_all_apps.csv | shape: (1744, 28)
Apps: ['Archery', 'PhantomLimb', 'PianoTiles', 'Puzzle', 'Sea', 'War']

[INFO] Downsample PhantomLimb = True | TARGET_N_PL = 400
[INFO] PhantomLimb N = 325 | Total N = 1744

================= FEATURES =================
Num features: 23
First 25 feature cols: ['missing_joints_count', 'missing_joints_ratio', 'collapsed_joints_count', 'center_of_mass_x', 'center_of_mass_y', 'center_of_mass_z', 'distance_from_origin', 'bbox_width', 'bbox_height', 'bbox_depth', 'bbox_volume', 'max_joint_distance_from_com', 'distance_from_floor', 'below_floor', 'left_forearm_length', 'right_forearm_length', 'left_shin_length', 'right_shin_length', 'arm_length_symmetry', 'leg_length_symmetry', 'body_forward_x', 'body_forward_y', 'body_forward_z']

================= LOAO: TEMPORAL (RAW SEQ) - TRANSFORMER (IMPROVED) =================
[INFO] SEQ_LEN=5 | raw tokens = (T=5, F=23)
[Temporal | T-SeqRa

[Temporal | T-SeqRaw+thr] HELD-OUT=PianoTiles   (SEQ_LEN=5) thr=0.35 val_f1=0.951  acc=0.725 f1=0.464 prec=0.306 rec=0.963  n_test=436
[Temporal | T-SeqRaw+thr] HELD-OUT=Puzzle       (SEQ_LEN=5) thr=0.85 val_f1=0.869  acc=0.278 f1=0.000 prec=0.000 rec=0.000  n_test=295
[Temporal | T-SeqRaw+thr] HELD-OUT=Sea          (SEQ_LEN=5) thr=0.65 val_f1=0.919  acc=0.547 f1=0.000 prec=0.000 rec=0.000  n_test=296
[Temporal | T-SeqRaw+thr] HELD-OUT=War          (SEQ_LEN=5) thr=0.50 val_f1=0.940  acc=0.667 f1=0.716 prec=0.558 rec=1.000  n_test=276

================ SUMMARY (mean over held-out apps) ================
acc          0.623799
f1           0.397733
precision    0.456429
recall       0.520143
dtype: float64

Saved: /content/loao_temporal_transformer_seqraw_len5_IMPROVED.csv


In [7]:
# ===============================================================
# VR APP-INDEPENDENT METRICS - LOAO TEMPORAL (RAW SEQ) TRANSFORMER
# Stable version:
#   - RAW SEQ input: (SEQ_LEN, F) = (5, 23)
#   - Train-only median impute + StandardScaler
#   - tf.data pipelines w/ fixed batch size + drop_remainder=True
#   - Class weights (per fold)
#   - Threshold tuned on VAL (maximize F1), eval on TEST
#   - NO tf.function wrapper => avoids retracing spam
# ===============================================================

import os
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.layers import (
    Input, Dense, Dropout, LayerNormalization,
    MultiHeadAttention, GlobalAveragePooling1D
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# -------------------------
# 1) LOAD DATA
# -------------------------
CANDIDATE_PATHS = [
    "/content/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/VR_Metrics/extracted_metrics_all_apps.csv",
]
DATA_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        DATA_PATH = p
        break
if DATA_PATH is None:
    raise FileNotFoundError("Could not find extracted_metrics_all_apps.csv in common locations.")

df = pd.read_csv(DATA_PATH)
print("\n================= LOADED DATA =================")
print("Loaded:", DATA_PATH, "| shape:", df.shape)

# -------------------------
# 2) CLEAN / TYPES
# -------------------------
df = df.replace("", np.nan)

required = {"App", "Temporal"}
missing_req = required - set(df.columns)
if missing_req:
    raise ValueError(f"Missing required columns: {missing_req}")

df["Temporal"] = pd.to_numeric(df["Temporal"], errors="coerce")
df = df.dropna(subset=["App", "Temporal"]).copy()
df["Temporal"] = df["Temporal"].astype(int)

apps = sorted(df["App"].unique().tolist())
print("Apps:", apps)

# OPTIONAL: Downsample PhantomLimb
DOWNSAMPLE_PHANTOMLIMB = True
TARGET_N_PL = 400
if DOWNSAMPLE_PHANTOMLIMB and ("PhantomLimb" in df["App"].unique()):
    pl = df[df["App"] == "PhantomLimb"]
    others = df[df["App"] != "PhantomLimb"]
    if len(pl) > TARGET_N_PL:
        pl = pl.sample(TARGET_N_PL, random_state=SEED)
    df = pd.concat([others, pl], ignore_index=True)
    apps = sorted(df["App"].unique().tolist())
    print("\n[INFO] Downsample PhantomLimb =", DOWNSAMPLE_PHANTOMLIMB, "| TARGET_N_PL =", TARGET_N_PL)
    print("[INFO] PhantomLimb N =", len(df[df["App"] == "PhantomLimb"]), "| Total N =", len(df))

# -------------------------
# 3) FEATURE COLS (23)
# -------------------------
ID_COLS = [c for c in ["GlobalID", "EntryID"] if c in df.columns]
DROP_COLS = ["App", "Spatial", "Temporal"] + ID_COLS
feature_cols = [c for c in df.columns if c not in DROP_COLS]

for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("\n================= FEATURES =================")
print("Num features:", len(feature_cols))
print("First 25 feature cols:", feature_cols[:25])

# -------------------------
# 4) HELPERS
# -------------------------
def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred).tolist(),
        "pred_dist": dict(zip(*np.unique(y_pred, return_counts=True))),
    }

def make_raw_sequence_dataset(app_df, label_col="Temporal", seq_len=5):
    """
    Raw seq: returns X of shape (N_seq, seq_len, F) using consecutive rows.
    Label is center frame: i + seq_len//2
    """
    X = app_df[feature_cols].values.astype(float)
    y = app_df[label_col].values.astype(int)

    if len(app_df) < seq_len:
        return np.empty((0, seq_len, len(feature_cols))), np.empty((0,), dtype=int)

    X_seq, y_seq = [], []
    center = seq_len // 2
    for i in range(len(app_df) - seq_len + 1):
        X_seq.append(X[i:i+seq_len])
        y_seq.append(y[i + center])

    return np.stack(X_seq, axis=0), np.array(y_seq, dtype=int)

def tune_threshold_on_val(y_val, p_val, grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 19)
    best_thr, best_f1 = 0.5, -1.0
    for t in grid:
        yhat = (p_val >= t).astype(int)
        f1 = f1_score(y_val, yhat, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = float(t)
    return best_thr, float(best_f1)

def build_transformer_seq_model(seq_len, n_feat,
                                d_model=64, num_heads=2, key_dim=32,
                                ff_dim=128, depth=2, dropout=0.15,
                                lr=1e-4):
    inp = Input(shape=(seq_len, n_feat), name="seq_in")
    x = Dense(d_model, activation="relu")(inp)

    for _ in range(depth):
        attn = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(x, x)
        attn = Dropout(dropout)(attn)
        x = LayerNormalization(epsilon=1e-6)(x + attn)

        ffn = Dense(ff_dim, activation="relu")(x)
        ffn = Dense(d_model)(ffn)
        ffn = Dropout(dropout)(ffn)
        x = LayerNormalization(epsilon=1e-6)(x + ffn)

    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(dropout)(x)
    out = Dense(1, activation="sigmoid", name="out")(x)

    model = Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

def make_tf_dataset(X, y, batch_size, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(y), 5000), seed=SEED, reshuffle_each_iteration=True)
    # drop_remainder=True => consistent batch shapes, helps stability
    ds = ds.batch(batch_size, drop_remainder=True).prefetch(tf.data.AUTOTUNE)
    return ds

# -------------------------
# 5) LOAO: Temporal raw-seq Transformer (stable)
# -------------------------
def run_loao_temporal_transformer_seqraw(
    label_col="Temporal",
    seq_len=5,
    d_model=64,
    num_heads=2,
    key_dim=32,
    ff_dim=128,
    depth=2,
    dropout=0.15,
    lr=1e-4,
    epochs=25,
    batch_size=64,
    val_size=0.15,
):
    results = []

    print("\n================= LOAO: TEMPORAL (RAW SEQ) - TRANSFORMER (STABLE) =================")
    print(f"[INFO] SEQ_LEN={seq_len} | raw tokens = (T={seq_len}, F={len(feature_cols)})")

    for test_app in apps:
        train_apps = [a for a in apps if a != test_app]

        # Build train sequences (no leakage)
        X_train_list, y_train_list = [], []
        for a in train_apps:
            Xa, ya = make_raw_sequence_dataset(df[df["App"] == a], label_col=label_col, seq_len=seq_len)
            if len(ya) > 0:
                X_train_list.append(Xa)
                y_train_list.append(ya)

        if len(X_train_list) == 0:
            print(f"[{label_col} | T-SeqRaw] HELD-OUT={test_app} -> No train sequences.")
            continue

        X_train_full = np.concatenate(X_train_list, axis=0)
        y_train_full = np.concatenate(y_train_list, axis=0)

        # Build test sequences
        X_test, y_test = make_raw_sequence_dataset(df[df["App"] == test_app], label_col=label_col, seq_len=seq_len)
        if len(y_test) == 0:
            print(f"[{label_col} | T-SeqRaw] HELD-OUT={test_app} -> No test sequences.")
            continue

        # Train/val split
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_full, y_train_full,
            test_size=val_size,
            random_state=SEED,
            stratify=y_train_full
        )

        n_feat = X_tr.shape[-1]

        # Train-only impute + scale (flatten tokens)
        imp = SimpleImputer(strategy="median")
        sca = StandardScaler()

        X_tr_flat = X_tr.reshape(-1, n_feat)
        X_val_flat = X_val.reshape(-1, n_feat)
        X_te_flat  = X_test.reshape(-1, n_feat)

        X_tr_flat = imp.fit_transform(X_tr_flat)
        X_val_flat = imp.transform(X_val_flat)
        X_te_flat  = imp.transform(X_te_flat)

        X_tr_flat = sca.fit_transform(X_tr_flat)
        X_val_flat = sca.transform(X_val_flat)
        X_te_flat  = sca.transform(X_te_flat)

        X_tr = X_tr_flat.reshape(-1, seq_len, n_feat).astype(np.float32)
        X_val = X_val_flat.reshape(-1, seq_len, n_feat).astype(np.float32)
        X_te  = X_te_flat.reshape(-1, seq_len, n_feat).astype(np.float32)

        # Class weights
        classes = np.array([0, 1])
        cw = compute_class_weight(class_weight="balanced", classes=classes, y=y_tr)
        class_weight = {0: float(cw[0]), 1: float(cw[1])}

        # Build model
        model = build_transformer_seq_model(
            seq_len=seq_len, n_feat=n_feat,
            d_model=d_model, num_heads=num_heads, key_dim=key_dim,
            ff_dim=ff_dim, depth=depth, dropout=dropout, lr=lr
        )

        early = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=0)
        rlrop = ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-6, verbose=0)

        # tf.data datasets (fixed batch size, drop_remainder=True)
        train_ds = make_tf_dataset(X_tr, y_tr, batch_size=batch_size, shuffle=True)
        val_ds   = make_tf_dataset(X_val, y_val, batch_size=batch_size, shuffle=False)

        model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=epochs,
            verbose=0,
            class_weight=class_weight,
            callbacks=[early, rlrop]
        )

        # Predict probabilities (no tf.function wrapper)
        p_val = model.predict(X_val, batch_size=batch_size, verbose=0).reshape(-1)
        p_te  = model.predict(X_te,  batch_size=batch_size, verbose=0).reshape(-1)

        thr, val_f1 = tune_threshold_on_val(y_val, p_val)
        yhat = (p_te >= thr).astype(int)

        m = eval_binary(y_test, yhat)
        row = {
            "label": label_col,
            "task": f"sequence_raw_len_{seq_len}",
            "model": "Transformer_seqraw_STABLE",
            "held_out_app": test_app,
            "seq_len": int(seq_len),
            "F": int(n_feat),
            "n_train": int(len(y_tr)),
            "n_val": int(len(y_val)),
            "n_test": int(len(y_test)),
            "thr": float(thr),
            "val_f1_best": float(val_f1),
            **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
            "cm": m["cm"],
            "pred_dist": m["pred_dist"],
        }

        print(f"[Temporal | T-SeqRaw+thr] HELD-OUT={test_app:12s} (SEQ_LEN={seq_len}) "
              f"thr={thr:.2f} val_f1={val_f1:.3f}  "
              f"acc={row['acc']:.3f} f1={row['f1']:.3f} "
              f"prec={row['precision']:.3f} rec={row['recall']:.3f}  "
              f"n_test={row['n_test']}")
        results.append(row)

        tf.keras.backend.clear_session()

    return pd.DataFrame(results)

# -------------------------
# 6) RUN
# -------------------------
SEQ_LEN = 5
res = run_loao_temporal_transformer_seqraw(
    label_col="Temporal",
    seq_len=SEQ_LEN,
    d_model=64,
    num_heads=2,
    key_dim=32,
    ff_dim=128,
    depth=2,
    dropout=0.15,
    lr=1e-4,
    epochs=25,
    batch_size=64,
    val_size=0.15,
)

print("\n================ SUMMARY (mean over held-out apps) ================")
if len(res) > 0:
    print(res[["acc","f1","precision","recall"]].mean())
else:
    print("No results (no sequences found).")

OUT = "/content/loao_temporal_transformer_seqraw_len5_STABLE.csv"
res.to_csv(OUT, index=False)
print("\nSaved:", OUT)


================= LOADED DATA =================
Loaded: /content/extracted_metrics_all_apps.csv | shape: (1744, 28)
Apps: ['Archery', 'PhantomLimb', 'PianoTiles', 'Puzzle', 'Sea', 'War']

[INFO] Downsample PhantomLimb = True | TARGET_N_PL = 400
[INFO] PhantomLimb N = 325 | Total N = 1744

================= FEATURES =================
Num features: 23
First 25 feature cols: ['missing_joints_count', 'missing_joints_ratio', 'collapsed_joints_count', 'center_of_mass_x', 'center_of_mass_y', 'center_of_mass_z', 'distance_from_origin', 'bbox_width', 'bbox_height', 'bbox_depth', 'bbox_volume', 'max_joint_distance_from_com', 'distance_from_floor', 'below_floor', 'left_forearm_length', 'right_forearm_length', 'left_shin_length', 'right_shin_length', 'arm_length_symmetry', 'leg_length_symmetry', 'body_forward_x', 'body_forward_y', 'body_forward_z']

================= LOAO: TEMPORAL (RAW SEQ) - TRANSFORMER (STABLE) =================
[INFO] SEQ_LEN=5 | raw tokens = (T=5, F=23)
[Temporal | T-SeqRaw+

In [9]:
# ===============================================================
# TEMPORAL LOAO: Transformer on RAW SEQ (T=5, F=23)
# Fixes:
#  - NO drop_remainder for val/test (critical for threshold tuning)
#  - AdamW weight decay
#  - optional label smoothing
#  - deterministic-ish seeds
# ===============================================================

import os
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.layers import Input, Dense, Dropout, LayerNormalization, MultiHeadAttention, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# (Optional) helps reproducibility
os.environ["TF_DETERMINISTIC_OPS"] = "1"

# -------------------------
# 1) LOAD DATA
# -------------------------
CANDIDATE_PATHS = [
    "/content/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/VR_Metrics/extracted_metrics_all_apps.csv",
]
DATA_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        DATA_PATH = p
        break
if DATA_PATH is None:
    raise FileNotFoundError("Could not find extracted_metrics_all_apps.csv in common locations.")

df = pd.read_csv(DATA_PATH)
print("\n================= LOADED DATA =================")
print("Loaded:", DATA_PATH, "| shape:", df.shape)

# -------------------------
# 2) CLEAN / TYPES
# -------------------------
df = df.replace("", np.nan)
df = df.dropna(subset=["App"]).copy()

df["Temporal"] = pd.to_numeric(df["Temporal"], errors="coerce")
df = df.dropna(subset=["Temporal"]).copy()
df["Temporal"] = df["Temporal"].astype(int)

apps = sorted(df["App"].unique().tolist())
print("Apps:", apps)

# OPTIONAL: Downsample PhantomLimb
DOWNSAMPLE_PHANTOMLIMB = True
TARGET_N_PL = 400
if DOWNSAMPLE_PHANTOMLIMB and ("PhantomLimb" in df["App"].unique()):
    pl = df[df["App"] == "PhantomLimb"]
    others = df[df["App"] != "PhantomLimb"]
    if len(pl) > TARGET_N_PL:
        pl = pl.sample(TARGET_N_PL, random_state=SEED)
    df = pd.concat([others, pl], ignore_index=True)
    apps = sorted(df["App"].unique().tolist())
    print("\n[INFO] Downsample PhantomLimb =", DOWNSAMPLE_PHANTOMLIMB, "| TARGET_N_PL =", TARGET_N_PL)
    print("[INFO] PhantomLimb N =", len(df[df["App"] == "PhantomLimb"]), "| Total N =", len(df))

# -------------------------
# 3) FEATURES (23)
# -------------------------
ID_COLS = [c for c in ["GlobalID", "EntryID"] if c in df.columns]
DROP_COLS = ["App", "Spatial", "Temporal"] + ID_COLS
feature_cols = [c for c in df.columns if c not in DROP_COLS]

for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

F = len(feature_cols)
print("\n================= FEATURES =================")
print("Num features:", F)
print("First 25 feature cols:", feature_cols[:25])

# -------------------------
# 4) Sliding windows: RAW seq dataset (N, T, F)
# -------------------------
def window_is_valid(window, min_non_nan_ratio=0.4):
    total = window.size
    non_nan = np.isfinite(window).sum()
    return (non_nan / total) >= min_non_nan_ratio

def make_raw_sequence_dataset(app_df, label_col="Temporal", seq_len=5, min_non_nan_ratio=0.4):
    X = app_df[feature_cols].values.astype(float)
    y = app_df[label_col].values.astype(int)

    if len(app_df) < seq_len:
        return np.empty((0, seq_len, F)), np.empty((0,), dtype=int)

    X_seq, y_seq = [], []
    center = seq_len // 2

    for i in range(len(app_df) - seq_len + 1):
        window = X[i:i+seq_len]
        if not window_is_valid(window, min_non_nan_ratio=min_non_nan_ratio):
            continue
        X_seq.append(window)
        y_seq.append(y[i + center])

    if len(X_seq) == 0:
        return np.empty((0, seq_len, F)), np.empty((0,), dtype=int)

    return np.stack(X_seq, axis=0), np.array(y_seq, dtype=int)

# -------------------------
# 5) Metrics + threshold tuning
# -------------------------
def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred).tolist(),
    }

def tune_threshold_on_val(y_val, p_val, grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 19)
    best_thr, best_f1 = 0.5, -1.0
    for t in grid:
        yhat = (p_val >= t).astype(int)
        f1 = f1_score(y_val, yhat, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = float(t)
    return best_thr, float(best_f1)

# -------------------------
# 6) Transformer model (RAW seq)
# -------------------------
def transformer_encoder(inputs, num_heads, key_dim, ff_dim, dropout=0.15):
    attn = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(inputs, inputs)
    attn = Dropout(dropout)(attn)
    x = LayerNormalization(epsilon=1e-6)(inputs + attn)

    ffn = Dense(ff_dim, activation="relu")(x)
    ffn = Dense(x.shape[-1])(ffn)
    ffn = Dropout(dropout)(ffn)
    out = LayerNormalization(epsilon=1e-6)(x + ffn)
    return out

def build_rawseq_transformer(seq_len=5, feat_dim=23,
                            d_model=64, num_heads=2, key_dim=32,
                            ff_dim=128, depth=2, dropout=0.15,
                            lr=7e-5, weight_decay=1e-4,
                            label_smoothing=0.0):
    inp = Input(shape=(seq_len, feat_dim), name="raw_seq_in")

    # Project features per timestep
    x = Dense(d_model, activation="relu")(inp)

    for _ in range(depth):
        x = transformer_encoder(x, num_heads=num_heads, key_dim=key_dim, ff_dim=ff_dim, dropout=dropout)

    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(dropout)(x)
    out = Dense(1, activation="sigmoid")(x)

    model = Model(inp, out)

    # AdamW for stability under domain shift
    opt = tf.keras.optimizers.AdamW(learning_rate=lr, weight_decay=weight_decay)

    loss = tf.keras.losses.BinaryCrossentropy(label_smoothing=label_smoothing)
    model.compile(optimizer=opt, loss=loss, metrics=["accuracy"])
    return model

# -------------------------
# 7) tf.data (IMPORTANT: drop_remainder ONLY for train)
# -------------------------
def make_tf_dataset(X, y, batch_size, shuffle=True, drop_remainder=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(y), 5000), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size, drop_remainder=drop_remainder).prefetch(tf.data.AUTOTUNE)
    return ds

# -------------------------
# 8) LOAO runner
# -------------------------
def run_loao_temporal_transformer_rawseq(
    seq_len=5,
    min_non_nan_ratio=0.4,
    epochs=25,
    batch_size=64,
    val_size=0.15,
):
    results = []
    print("\n================= LOAO: TEMPORAL (RAW SEQ) - TRANSFORMER (FIXED) =================")
    print(f"[INFO] SEQ_LEN={seq_len} | raw tokens = (T={seq_len}, F={F})")

    for test_app in apps:
        # Build train sequences across all other apps
        X_train_list, y_train_list = [], []
        for a in apps:
            if a == test_app:
                continue
            Xa, ya = make_raw_sequence_dataset(
                df[df["App"] == a],
                label_col="Temporal",
                seq_len=seq_len,
                min_non_nan_ratio=min_non_nan_ratio
            )
            if len(ya) > 0:
                X_train_list.append(Xa)
                y_train_list.append(ya)

        if len(X_train_list) == 0:
            print(f"[Temporal | T-SeqRaw] HELD-OUT={test_app} -> No train sequences.")
            continue

        X_train_full = np.concatenate(X_train_list, axis=0)  # (N,T,F)
        y_train_full = np.concatenate(y_train_list, axis=0)

        # Test sequences
        X_test, y_test = make_raw_sequence_dataset(
            df[df["App"] == test_app],
            label_col="Temporal",
            seq_len=seq_len,
            min_non_nan_ratio=min_non_nan_ratio
        )
        if len(y_test) == 0:
            print(f"[Temporal | T-SeqRaw] HELD-OUT={test_app} -> No test sequences.")
            continue

        # Train/val split
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_full, y_train_full,
            test_size=val_size,
            random_state=SEED,
            stratify=y_train_full
        )

        # Impute + scale TRAIN ONLY (flatten time)
        imp = SimpleImputer(strategy="median")
        sca = StandardScaler()

        Xtr_flat = X_tr.reshape(-1, F)
        Xval_flat = X_val.reshape(-1, F)
        Xte_flat = X_test.reshape(-1, F)

        Xtr_flat = imp.fit_transform(Xtr_flat)
        Xval_flat = imp.transform(Xval_flat)
        Xte_flat = imp.transform(Xte_flat)

        Xtr_flat = sca.fit_transform(Xtr_flat)
        Xval_flat = sca.transform(Xval_flat)
        Xte_flat = sca.transform(Xte_flat)

        X_tr = Xtr_flat.reshape(-1, seq_len, F).astype(np.float32)
        X_val = Xval_flat.reshape(-1, seq_len, F).astype(np.float32)
        X_te  = Xte_flat.reshape(-1, seq_len, F).astype(np.float32)

        # Class weights
        classes = np.array([0, 1])
        cw = compute_class_weight(class_weight="balanced", classes=classes, y=y_tr)
        class_weight = {0: float(cw[0]), 1: float(cw[1])}

        model = build_rawseq_transformer(
            seq_len=seq_len, feat_dim=F,
            d_model=64, num_heads=2, key_dim=32,
            ff_dim=128, depth=2, dropout=0.15,
            lr=7e-5, weight_decay=1e-4,
            label_smoothing=0.0  # try 0.05 if still collapses
        )

        early = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=0)
        rlrop = ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-6, verbose=0)

        train_ds = make_tf_dataset(X_tr, y_tr, batch_size=batch_size, shuffle=True,  drop_remainder=True)
        val_ds   = make_tf_dataset(X_val, y_val, batch_size=batch_size, shuffle=False, drop_remainder=False)

        model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=epochs,
            verbose=0,
            class_weight=class_weight,
            callbacks=[early, rlrop]
        )

        # IMPORTANT: predict on FULL val/test arrays (no dropped samples)
        p_val = model.predict(X_val, batch_size=batch_size, verbose=0).reshape(-1)
        p_te  = model.predict(X_te,  batch_size=batch_size, verbose=0).reshape(-1)

        thr, val_f1 = tune_threshold_on_val(y_val, p_val)
        yhat = (p_te >= thr).astype(int)

        m = eval_binary(y_test, yhat)

        print(f"[Temporal | T-SeqRaw+thr] HELD-OUT={test_app:12s} (SEQ_LEN={seq_len}) "
              f"thr={thr:.2f} val_f1={val_f1:.3f}  "
              f"acc={m['acc']:.3f} f1={m['f1']:.3f} prec={m['precision']:.3f} rec={m['recall']:.3f}  "
              f"n_test={len(y_test)}")

        results.append({
            "held_out_app": test_app,
            "acc": m["acc"], "f1": m["f1"], "precision": m["precision"], "recall": m["recall"],
            "thr": thr, "val_f1": val_f1, "cm": m["cm"],
            "n_test": int(len(y_test)),
        })

        tf.keras.backend.clear_session()

    return pd.DataFrame(results)

# -------------------------
# 9) RUN
# -------------------------
SEQ_LEN = 5
MIN_NON_NAN_RATIO = 0.4

res = run_loao_temporal_transformer_rawseq(
    seq_len=SEQ_LEN,
    min_non_nan_ratio=MIN_NON_NAN_RATIO,
    epochs=25,
    batch_size=64,
    val_size=0.15
)

print("\n================ SUMMARY (mean over held-out apps) ================")
if len(res) > 0:
    print(res[["acc","f1","precision","recall"]].mean())
else:
    print("No results.")

OUT = "/content/loao_temporal_transformer_seqraw_len5_FIXED.csv"
res.to_csv(OUT, index=False)
print("\nSaved:", OUT)


================= LOADED DATA =================
Loaded: /content/extracted_metrics_all_apps.csv | shape: (1744, 28)
Apps: ['Archery', 'PhantomLimb', 'PianoTiles', 'Puzzle', 'Sea', 'War']

[INFO] Downsample PhantomLimb = True | TARGET_N_PL = 400
[INFO] PhantomLimb N = 325 | Total N = 1744

================= FEATURES =================
Num features: 23
First 25 feature cols: ['missing_joints_count', 'missing_joints_ratio', 'collapsed_joints_count', 'center_of_mass_x', 'center_of_mass_y', 'center_of_mass_z', 'distance_from_origin', 'bbox_width', 'bbox_height', 'bbox_depth', 'bbox_volume', 'max_joint_distance_from_com', 'distance_from_floor', 'below_floor', 'left_forearm_length', 'right_forearm_length', 'left_shin_length', 'right_shin_length', 'arm_length_symmetry', 'leg_length_symmetry', 'body_forward_x', 'body_forward_y', 'body_forward_z']

================= LOAO: TEMPORAL (RAW SEQ) - TRANSFORMER (FIXED) =================
[INFO] SEQ_LEN=5 | raw tokens = (T=5, F=23)
[Temporal | T-SeqRaw+t

In [8]:
# ===============================================================
# VR APP-INDEPENDENT METRICS - LOAO TEMPORAL - TRANSFORMER on AGG TOKENS
#
# Goal:
#   Use your *aggregated* temporal features (the same idea behind the 138-dim RF/LR),
#   but represent them as tokens so a Transformer can attend over (mean/std/delta/range/absvel/velstd).
#
# Input construction:
#   - Start from raw window (SEQ_LEN=5, F=23)
#   - Compute agg blocks: ("mean","std","delta","range","mean_abs_vel","vel_std")
#   - Instead of concatenating to (6*F=138) vector, reshape to tokens:
#         X_tokens shape = (N_seq, 6, F)  => (N_seq, 6, 23)
#
# Train:
#   - LOAO (leave-one-app-out)
#   - Train-only impute+scale (flatten across tokens)
#   - Class weights
#   - Threshold tuned on VAL F1 (like your LR_thr)
#   - tf.data (fixed batch, drop_remainder=True) for stability
# ===============================================================

import os
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.layers import (
    Input, Dense, Dropout, LayerNormalization,
    MultiHeadAttention, GlobalAveragePooling1D
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# -------------------------
# 1) LOAD DATA
# -------------------------
CANDIDATE_PATHS = [
    "/content/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/extracted_metrics_all_apps.csv",
    "/content/drive/MyDrive/VR_Metrics/extracted_metrics_all_apps.csv",
]
DATA_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        DATA_PATH = p
        break
if DATA_PATH is None:
    raise FileNotFoundError("Could not find extracted_metrics_all_apps.csv in common locations.")

df = pd.read_csv(DATA_PATH)
print("\n================= LOADED DATA =================")
print("Loaded:", DATA_PATH, "| shape:", df.shape)

# -------------------------
# 2) CLEAN / TYPES
# -------------------------
df = df.replace("", np.nan)

required = {"App", "Temporal"}
missing_req = required - set(df.columns)
if missing_req:
    raise ValueError(f"Missing required columns: {missing_req}")

df["Temporal"] = pd.to_numeric(df["Temporal"], errors="coerce")
df = df.dropna(subset=["App", "Temporal"]).copy()
df["Temporal"] = df["Temporal"].astype(int)

apps = sorted(df["App"].unique().tolist())
print("Apps:", apps)

# OPTIONAL: Downsample PhantomLimb
DOWNSAMPLE_PHANTOMLIMB = True
TARGET_N_PL = 400
if DOWNSAMPLE_PHANTOMLIMB and ("PhantomLimb" in df["App"].unique()):
    pl = df[df["App"] == "PhantomLimb"]
    others = df[df["App"] != "PhantomLimb"]
    if len(pl) > TARGET_N_PL:
        pl = pl.sample(TARGET_N_PL, random_state=SEED)
    df = pd.concat([others, pl], ignore_index=True)
    apps = sorted(df["App"].unique().tolist())
    print("\n[INFO] Downsample PhantomLimb =", DOWNSAMPLE_PHANTOMLIMB, "| TARGET_N_PL =", TARGET_N_PL)
    print("[INFO] PhantomLimb N =", len(df[df["App"] == "PhantomLimb"]), "| Total N =", len(df))

# -------------------------
# 3) FEATURE COLS (23)
# -------------------------
ID_COLS = [c for c in ["GlobalID", "EntryID"] if c in df.columns]
DROP_COLS = ["App", "Spatial", "Temporal"] + ID_COLS
feature_cols = [c for c in df.columns if c not in DROP_COLS]

for c in feature_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

F = len(feature_cols)
print("\n================= FEATURES =================")
print("Num features:", F)
print("First 25 feature cols:", feature_cols[:25])

# -------------------------
# 4) NaN-safe reducers (Patch B style)
# -------------------------
def nanmean_no_warn(x, axis=0):
    x = np.asarray(x, dtype=float)
    valid = np.isfinite(x)
    denom = valid.sum(axis=axis)
    num = np.where(valid, x, 0.0).sum(axis=axis)
    out = np.divide(num, denom, out=np.full_like(num, np.nan, dtype=float), where=(denom > 0))
    return out

def nanstd_no_warn(x, axis=0):
    x = np.asarray(x, dtype=float)
    mu = nanmean_no_warn(x, axis=axis)
    mu_b = np.expand_dims(mu, axis=axis)
    diff2 = (x - mu_b) ** 2
    valid = np.isfinite(diff2)
    denom = valid.sum(axis=axis)
    num = np.where(valid, diff2, 0.0).sum(axis=axis)
    var = np.divide(num, denom, out=np.full_like(num, np.nan, dtype=float), where=(denom > 0))
    return np.sqrt(var)

def nanrange_no_warn(x, axis=0):
    x = np.asarray(x, dtype=float)
    valid = np.isfinite(x)
    x_for_max = np.where(valid, x, -np.inf)
    x_for_min = np.where(valid, x, +np.inf)
    mx = np.max(x_for_max, axis=axis)
    mn = np.min(x_for_min, axis=axis)
    all_nan = ~valid.any(axis=axis)
    out = mx - mn
    out = np.where(all_nan, np.nan, out)
    return out

def window_is_valid(window, min_non_nan_ratio=0.4):
    total = window.size
    non_nan = np.isfinite(window).sum()
    return (non_nan / total) >= min_non_nan_ratio

# -------------------------
# 5) Build AGG TOKENS dataset: (N, 6, F)
# -------------------------
AGG_USE = ("mean","std","delta","range","mean_abs_vel","vel_std")
N_TOKENS = len(AGG_USE)

def aggregate_window_to_tokens(window, use=AGG_USE):
    """
    window: (T, F)
    returns tokens: (K, F) where K=len(use)
    """
    window = np.asarray(window, dtype=float)
    tokens = []

    if "mean" in use:
        tokens.append(nanmean_no_warn(window, axis=0))
    if "std" in use:
        tokens.append(nanstd_no_warn(window, axis=0))
    if "delta" in use:
        tokens.append(window[-1] - window[0])
    if "range" in use:
        tokens.append(nanrange_no_warn(window, axis=0))

    if "mean_abs_vel" in use or "vel_std" in use:
        diff = np.diff(window, axis=0)  # (T-1,F)
        if "mean_abs_vel" in use:
            tokens.append(nanmean_no_warn(np.abs(diff), axis=0))
        if "vel_std" in use:
            tokens.append(nanstd_no_warn(diff, axis=0))

    toks = np.stack(tokens, axis=0)  # (K,F)
    assert toks.shape[0] == len(use) and toks.shape[1] == window.shape[1]
    return toks

def make_agg_token_sequence_dataset(app_df, label_col="Temporal", seq_len=5,
                                   use=AGG_USE, min_non_nan_ratio=0.4):
    """
    Sliding window -> token matrix (K,F) per window.
    Returns:
      X: (N_seq, K, F)
      y: (N_seq,)
    """
    X = app_df[feature_cols].values.astype(float)
    y = app_df[label_col].values.astype(int)

    if len(app_df) < seq_len:
        return np.empty((0, len(use), len(feature_cols))), np.empty((0,), dtype=int)

    X_seq, y_seq = [], []
    center = seq_len // 2

    for i in range(len(app_df) - seq_len + 1):
        window = X[i:i+seq_len]
        if not window_is_valid(window, min_non_nan_ratio=min_non_nan_ratio):
            continue
        toks = aggregate_window_to_tokens(window, use=use)  # (K,F)
        X_seq.append(toks)
        y_seq.append(y[i + center])

    if len(X_seq) == 0:
        return np.empty((0, len(use), len(feature_cols))), np.empty((0,), dtype=int)

    return np.stack(X_seq, axis=0), np.array(y_seq, dtype=int)

# -------------------------
# 6) Metrics + threshold tuning
# -------------------------
def eval_binary(y_true, y_pred):
    return {
        "acc": float(accuracy_score(y_true, y_pred)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "cm": confusion_matrix(y_true, y_pred).tolist(),
        "pred_dist": dict(zip(*np.unique(y_pred, return_counts=True))),
    }

def tune_threshold_on_val(y_val, p_val, grid=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 19)
    best_thr, best_f1 = 0.5, -1.0
    for t in grid:
        yhat = (p_val >= t).astype(int)
        f1 = f1_score(y_val, yhat, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thr = float(t)
    return best_thr, float(best_f1)

# -------------------------
# 7) Transformer on AGG TOKENS
# -------------------------
def build_transformer_token_model(n_tokens, n_feat,
                                  d_model=64, num_heads=2, key_dim=32,
                                  ff_dim=128, depth=2, dropout=0.15,
                                  lr=1e-4):
    """
    Input: (K, F) tokens.
    Dense projects feature-dim -> d_model per token, then attention across tokens.
    """
    inp = Input(shape=(n_tokens, n_feat), name="agg_tokens_in")
    x = Dense(d_model, activation="relu")(inp)

    for _ in range(depth):
        attn = MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)(x, x)
        attn = Dropout(dropout)(attn)
        x = LayerNormalization(epsilon=1e-6)(x + attn)

        ffn = Dense(ff_dim, activation="relu")(x)
        ffn = Dense(d_model)(ffn)
        ffn = Dropout(dropout)(ffn)
        x = LayerNormalization(epsilon=1e-6)(x + ffn)

    x = GlobalAveragePooling1D()(x)
    x = Dense(64, activation="relu")(x)
    x = Dropout(dropout)(x)
    out = Dense(1, activation="sigmoid")(x)

    model = Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

def make_tf_dataset(X, y, batch_size, shuffle=True):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(len(y), 5000), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.batch(batch_size, drop_remainder=True).prefetch(tf.data.AUTOTUNE)
    return ds

# -------------------------
# 8) LOAO runner
# -------------------------
def run_loao_temporal_transformer_aggtokens(
    label_col="Temporal",
    seq_len=5,
    use=AGG_USE,
    min_non_nan_ratio=0.4,
    d_model=64,
    num_heads=2,
    key_dim=32,
    ff_dim=128,
    depth=2,
    dropout=0.15,
    lr=1e-4,
    epochs=25,
    batch_size=64,
    val_size=0.15,
):
    results = []

    print("\n================= LOAO: TEMPORAL (AGG TOKENS) - TRANSFORMER =================")
    print(f"[INFO] SEQ_LEN={seq_len} | tokens K={len(use)} | token dim F={len(feature_cols)}")
    print(f"[INFO] This corresponds to 6*F = {len(use)*len(feature_cols)} aggregated dims (i.e., ~138).")

    for test_app in apps:
        train_apps = [a for a in apps if a != test_app]

        # Build train token-seqs
        X_train_list, y_train_list = [], []
        for a in train_apps:
            Xa, ya = make_agg_token_sequence_dataset(
                df[df["App"] == a],
                label_col=label_col,
                seq_len=seq_len,
                use=use,
                min_non_nan_ratio=min_non_nan_ratio
            )
            if len(ya) > 0:
                X_train_list.append(Xa)
                y_train_list.append(ya)

        if len(X_train_list) == 0:
            print(f"[{label_col} | T-AggTokens] HELD-OUT={test_app} -> No train sequences.")
            continue

        X_train_full = np.concatenate(X_train_list, axis=0)  # (N, K, F)
        y_train_full = np.concatenate(y_train_list, axis=0)

        # Build test token-seqs
        X_test, y_test = make_agg_token_sequence_dataset(
            df[df["App"] == test_app],
            label_col=label_col,
            seq_len=seq_len,
            use=use,
            min_non_nan_ratio=min_non_nan_ratio
        )
        if len(y_test) == 0:
            print(f"[{label_col} | T-AggTokens] HELD-OUT={test_app} -> No test sequences.")
            continue

        # Split train/val
        X_tr, X_val, y_tr, y_val = train_test_split(
            X_train_full, y_train_full,
            test_size=val_size,
            random_state=SEED,
            stratify=y_train_full
        )

        K = X_tr.shape[1]
        n_feat = X_tr.shape[2]

        # Train-only impute+scale across tokens/features
        # Flatten (N*K, F)
        imp = SimpleImputer(strategy="median")
        sca = StandardScaler()

        X_tr_flat = X_tr.reshape(-1, n_feat)
        X_val_flat = X_val.reshape(-1, n_feat)
        X_te_flat  = X_test.reshape(-1, n_feat)

        X_tr_flat = imp.fit_transform(X_tr_flat)
        X_val_flat = imp.transform(X_val_flat)
        X_te_flat  = imp.transform(X_te_flat)

        X_tr_flat = sca.fit_transform(X_tr_flat)
        X_val_flat = sca.transform(X_val_flat)
        X_te_flat  = sca.transform(X_te_flat)

        X_tr = X_tr_flat.reshape(-1, K, n_feat).astype(np.float32)
        X_val = X_val_flat.reshape(-1, K, n_feat).astype(np.float32)
        X_te  = X_te_flat.reshape(-1, K, n_feat).astype(np.float32)

        # Class weights
        classes = np.array([0, 1])
        cw = compute_class_weight(class_weight="balanced", classes=classes, y=y_tr)
        class_weight = {0: float(cw[0]), 1: float(cw[1])}

        # Build model
        model = build_transformer_token_model(
            n_tokens=K, n_feat=n_feat,
            d_model=d_model, num_heads=num_heads, key_dim=key_dim,
            ff_dim=ff_dim, depth=depth, dropout=dropout, lr=lr
        )

        early = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=0)
        rlrop = ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-6, verbose=0)

        train_ds = make_tf_dataset(X_tr, y_tr, batch_size=batch_size, shuffle=True)
        val_ds   = make_tf_dataset(X_val, y_val, batch_size=batch_size, shuffle=False)

        model.fit(
            train_ds,
            validation_data=val_ds,
            epochs=epochs,
            verbose=0,
            class_weight=class_weight,
            callbacks=[early, rlrop]
        )

        p_val = model.predict(X_val, batch_size=batch_size, verbose=0).reshape(-1)
        p_te  = model.predict(X_te,  batch_size=batch_size, verbose=0).reshape(-1)

        thr, val_f1 = tune_threshold_on_val(y_val, p_val)
        yhat = (p_te >= thr).astype(int)

        m = eval_binary(y_test, yhat)

        row = {
            "label": label_col,
            "task": f"sequence_aggtokens_len_{seq_len}",
            "model": "Transformer_aggtokens",
            "held_out_app": test_app,
            "seq_len": int(seq_len),
            "K_tokens": int(K),
            "F_token": int(n_feat),
            "agg_dims_equiv": int(K * n_feat),
            "n_train": int(len(y_tr)),
            "n_val": int(len(y_val)),
            "n_test": int(len(y_test)),
            "thr": float(thr),
            "val_f1_best": float(val_f1),
            **{k: v for k, v in m.items() if k not in ["cm", "pred_dist"]},
            "cm": m["cm"],
            "pred_dist": m["pred_dist"],
        }

        print(f"[Temporal | T-AggTokens+thr] HELD-OUT={test_app:12s} "
              f"(SEQ_LEN={seq_len}, K={K}, F={n_feat}, equiv={K*n_feat}) "
              f"thr={thr:.2f} val_f1={val_f1:.3f}  "
              f"acc={row['acc']:.3f} f1={row['f1']:.3f} "
              f"prec={row['precision']:.3f} rec={row['recall']:.3f} "
              f"n_test={row['n_test']}")

        results.append(row)
        tf.keras.backend.clear_session()

    return pd.DataFrame(results)

# -------------------------
# 9) RUN
# -------------------------
SEQ_LEN = 5
MIN_NON_NAN_RATIO = 0.4

res = run_loao_temporal_transformer_aggtokens(
    label_col="Temporal",
    seq_len=SEQ_LEN,
    use=AGG_USE,
    min_non_nan_ratio=MIN_NON_NAN_RATIO,
    d_model=64,
    num_heads=2,
    key_dim=32,
    ff_dim=128,
    depth=2,
    dropout=0.15,
    lr=1e-4,
    epochs=25,
    batch_size=64,
    val_size=0.15,
)

print("\n================ SUMMARY (mean over held-out apps) ================")
if len(res) > 0:
    print(res[["acc", "f1", "precision", "recall"]].mean())
else:
    print("No results (no sequences found).")

OUT = "/content/loao_temporal_transformer_aggtokens_len5.csv"
res.to_csv(OUT, index=False)
print("\nSaved:", OUT)


================= LOADED DATA =================
Loaded: /content/extracted_metrics_all_apps.csv | shape: (1744, 28)
Apps: ['Archery', 'PhantomLimb', 'PianoTiles', 'Puzzle', 'Sea', 'War']

[INFO] Downsample PhantomLimb = True | TARGET_N_PL = 400
[INFO] PhantomLimb N = 325 | Total N = 1744

================= FEATURES =================
Num features: 23
First 25 feature cols: ['missing_joints_count', 'missing_joints_ratio', 'collapsed_joints_count', 'center_of_mass_x', 'center_of_mass_y', 'center_of_mass_z', 'distance_from_origin', 'bbox_width', 'bbox_height', 'bbox_depth', 'bbox_volume', 'max_joint_distance_from_com', 'distance_from_floor', 'below_floor', 'left_forearm_length', 'right_forearm_length', 'left_shin_length', 'right_shin_length', 'arm_length_symmetry', 'leg_length_symmetry', 'body_forward_x', 'body_forward_y', 'body_forward_z']

================= LOAO: TEMPORAL (AGG TOKENS) - TRANSFORMER =================
[INFO] SEQ_LEN=5 | tokens K=6 | token dim F=23
[INFO] This corresponds 